# mioh — Retrieval条件付き・トラック固定微細質感 v2

合格済みRetrieval v2が復元画像から予測する材質descriptorだけを条件にします。
検索画像、bank画像、正解画像の位相はgeneratorへ渡しません。ROIトラック開始時に
latentを一度だけ決め、同じ空間latent場で全フレームを生成します。出力はマスク内部の
5px以下輝度残差だけで、フレームごとにゼロ平均です。seed差は細部8倍表示でも確認します。

In [ ]:
#@title 1. 設定
WIDE_V1_ROOT = "/content/drive/MyDrive/mioh-roi-wide-materials-v1" #@param {type:"string"}
WIDE_ROOT = "/content/drive/MyDrive/mioh-roi-wide-materials-v2" #@param {type:"string"}
WIDE_EXTRA_ROOTS = "/content/drive/MyDrive/mioh-roi-wide-materials-v4-deint" #@param {type:"string"}
EXCLUDE_SOURCES = "" #@param {type:"string"}
DISTILL_CHECKPOINT = "/content/drive/MyDrive/mioh-roi-distill-detail-original-aligned-v2/best.pt" #@param {type:"string"}
RETRIEVAL_CHECKPOINT = "/content/drive/MyDrive/mioh-roi-material-retrieval-v2/best.pt" #@param {type:"string"}
SAVE_ROOT = "/content/drive/MyDrive/mioh-roi-retrieval-track-texture-v2" #@param {type:"string"}
WORK_ROOT = "/content/mioh-roi-retrieval-track-texture-v2" #@param {type:"string"}
MAX_STEPS = 6000 #@param {type:"integer"}
EVAL_INTERVAL = 500 #@param {type:"integer"}
BATCH_SIZE = 4 #@param {type:"integer"}

In [ ]:
#@title 2. GPU・Drive・実装・素材
import sys
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムへ変更してください。")
print("GPU:", torch.cuda.get_device_name(0))
# Colab users often start from this setup cell after reconnecting.  Preserve
# values from the parameter cell when present, otherwise restore its defaults.
WIDE_V1_ROOT = globals().get("WIDE_V1_ROOT", "/content/drive/MyDrive/mioh-roi-wide-materials-v1")
WIDE_ROOT = globals().get("WIDE_ROOT", "/content/drive/MyDrive/mioh-roi-wide-materials-v2")
WIDE_EXTRA_ROOTS = globals().get("WIDE_EXTRA_ROOTS", "/content/drive/MyDrive/mioh-roi-wide-materials-v4-deint")
EXCLUDE_SOURCES = globals().get("EXCLUDE_SOURCES", "")
DISTILL_CHECKPOINT = globals().get("DISTILL_CHECKPOINT", "/content/drive/MyDrive/mioh-roi-distill-detail-original-aligned-v2/best.pt")
RETRIEVAL_CHECKPOINT = globals().get("RETRIEVAL_CHECKPOINT", "/content/drive/MyDrive/mioh-roi-material-retrieval-v2/best.pt")
SAVE_ROOT = globals().get("SAVE_ROOT", "/content/drive/MyDrive/mioh-roi-retrieval-track-texture-v2")
WORK_ROOT = globals().get("WORK_ROOT", "/content/mioh-roi-retrieval-track-texture-v2")
MAX_STEPS = globals().get("MAX_STEPS", 6000)
EVAL_INTERVAL = globals().get("EVAL_INTERVAL", 500)
BATCH_SIZE = globals().get("BATCH_SIZE", 4)
from google.colab import drive
if Path("/content/drive/MyDrive").is_dir():
    print("Drive already mounted")
else:
    drive.mount("/content/drive")
work = Path(WORK_ROOT); work.mkdir(parents=True, exist_ok=True)
SOURCES = {'roi_enhancer_distill_runtime.py': '"""Small, bounded SwiftVR-output distillation pilot; not an app replacement.\n\nThe standalone Colab notebook embeds this file. Existing teacher movies are\nread only. No SwiftVR weights/download/inference is required. Public code must\nnot contain private source names, Drive IDs, or media paths.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport subprocess\nimport time\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nVERSION = "roi-distill-pilot-v1"\nARCH = {"channels": 24, "blocks": 6, "window": 5, "scale": 2}\nEXPANDED_ARCH = {**ARCH, "blocks": 12}\n\n\ndef validated_architecture(architecture):\n    if (not isinstance(architecture, dict) or set(architecture) != set(ARCH)\n            or any(type(v) is not int for v in architecture.values())\n            or architecture not in (ARCH, EXPANDED_ARCH)):\n        raise ValueError("対応モデルは24チャンネル・6/12ブロック・5フレーム・2倍のみです。")\n    return dict(architecture)\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".partial")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\\n")\n    os.replace(temporary, path)\n\n\ndef identity(path):\n    size = Path(path).stat().st_size\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        digest.update(handle.read(1024 * 1024))\n        handle.seek(max(0, size - 1024 * 1024))\n        digest.update(handle.read(1024 * 1024))\n    return {"bytes": size, "sample_sha256": digest.hexdigest()}\n\n\ndef probe(path):\n    data = json.loads(subprocess.check_output([\n        "ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", str(path)\n    ], text=True))\n    video = next(s for s in data["streams"] if s["codec_type"] == "video"\n                 and not s.get("disposition", {}).get("attached_pic"))\n    return {"width": int(video["width"]), "height": int(video["height"]),\n            "frames": int(video.get("nb_frames", 0)) if video.get("nb_frames", "0").isdigit() else 0,\n            "duration": float(video.get("duration", data["format"].get("duration", 0)))}\n\n\ndef rate(value):\n    from fractions import Fraction\n    return float(Fraction(str(value)))\n\n\ndef discover(root, source_override=None):\n    """Find complete parts of one full-frame teacher job, never mix jobs."""\n    root = Path(root).resolve()\n    candidates = []\n    for path in sorted((root / "output" / "jobs").glob("*/job.json")):\n        try:\n            job = json.loads(path.read_text())\n            # Tiled and short-run outputs are deliberately not training defaults.\n            if job["mode"] != "full" or job["output_height"] < 1440:\n                continue\n            source = Path(source_override) if source_override else root / Path(job["source"]["name"]).name\n            if not source.is_file() or identity(source) != {\n                k: job["source"][k] for k in ("bytes", "sample_sha256")\n            }:\n                continue\n            info = probe(source)\n            if job["output_width"] * info["height"] != job["output_height"] * info["width"]:\n                continue\n            fps = rate(job["fps"])\n            if not math.isfinite(fps) or fps <= 0:\n                continue\n            stride = max(1, round(float(job["segment_seconds"]) * fps))\n            parts = []\n            for movie in sorted(path.parent.glob("part-*.mp4")):\n                stamp = movie.with_suffix(".complete.json")\n                if not stamp.is_file() or movie.stat().st_size == 0:\n                    continue\n                try:\n                    index = int(movie.stem.split("-")[1])\n                    meta = json.loads(stamp.read_text())\n                    first = int(job["start_frame"]) + stride * index\n                    count = min(stride, int(job["end_frame"]) - first)\n                    if count < 90 or meta["frames"] != count:\n                        continue\n                    if (meta["width"], meta["height"]) != (job["output_width"], job["output_height"]):\n                        continue\n                    if abs(rate(meta["fps"]) - fps) > 1e-4:\n                        continue\n                    parts.append({"path": str(movie), "index": index, "first": first, "frames": count})\n                except (ValueError, KeyError, OSError, json.JSONDecodeError):\n                    continue\n            if len(parts) >= 3:\n                candidates.append({"job": job, "parts": parts, "source": str(source), "source_info": info})\n        except (KeyError, ValueError, OSError, subprocess.CalledProcessError):\n            continue\n    if not candidates:\n        raise RuntimeError("同じ元動画に対応する全画面・完成済み区間が3つ以上必要です。DRIVE_ROOT / SOURCE_OVERRIDEを確認してください。")\n    # Prefer broad temporal coverage, then resolution; not a different teacher per crop.\n    return max(candidates, key=lambda c: (len(c["parts"]), c["job"]["output_width"]))\n\n\ndef split_parts(parts, limit=6):\n    if len(parts) < 3 or limit < 3:\n        raise ValueError("最低3区間が必要です。")\n    chosen = [parts[i] for i in sorted(set(np.linspace(0, len(parts) - 1, min(limit, len(parts))).round().astype(int)))]\n    validation = {len(chosen) - 1}\n    if len(chosen) >= 6:\n        validation.add(len(chosen) // 2)\n    return [(p, "val" if i in validation else "train") for i, p in enumerate(chosen)]\n\n\ndef decode(path, start_seconds, count, fps, width, height):\n    """Mirror teacher preparation\'s CFR conversion; fail on short decode."""\n    raw = subprocess.check_output([\n        "ffmpeg", "-v", "error", "-nostdin", "-ss", f"{start_seconds:.9f}", "-i", str(path),\n        "-map", "0:v:0", "-an", "-vf", f"fps={fps},scale={width}:{height}:flags=bicubic,setsar=1",\n        "-frames:v", str(count), "-f", "rawvideo", "-pix_fmt", "rgb24", "pipe:1"\n    ])\n    expected = count * width * height * 3\n    if len(raw) != expected:\n        raise RuntimeError("フレームの読み出し数が不一致です。学習を開始しません。")\n    return np.frombuffer(raw, np.uint8).reshape(count, height, width, 3).copy()\n\n\ndef sync_check(lq, hq):\n    """Diagnostic, not proof: low-frequency agreement and +/-2 frame lag."""\n    def thumb(frame):\n        return np.asarray(Image.fromarray(frame).resize((80, 45), Image.Resampling.BILINEAR), np.float32) / 255\n    low = np.stack([thumb(x) for x in lq])\n    high = np.stack([thumb(x) for x in hq])\n    scores = {lag: float(np.mean((low[2:-2] - high[2 + lag:len(high) - 2 + lag]) ** 2))\n              for lag in range(-2, 3)}\n    best = min(scores, key=scores.get)\n    # Large colour/structure mismatch fails closed. Ambiguous static clips are\n    # flagged, not claimed as positively frame-synchronised.\n    suspicious = scores[0] > 0.025 or scores[0] > scores[best] * 1.15 + 0.0001\n    spread = max(scores.values()) - min(scores.values())\n    cuts = float(np.abs(np.diff(low, axis=0)).mean(axis=(1, 2, 3)).max()) > 0.18\n    return {"zero_lag_mse": scores[0], "best_lag": best, "ambiguous": spread < 0.0001,\n            "accepted": not suspicious and not cuts, "scene_cut": cuts}\n\n\ndef crop_sequence(lq, hq, x, y, side):\n    if hq.shape[1:3] != (lq.shape[1] * 2, lq.shape[2] * 2):\n        raise ValueError("教師の座標倍率は2倍である必要があります。")\n    return (lq[:, y:y + side, x:x + side].copy(),\n            hq[2:-2, y * 2:(y + side) * 2, x * 2:(x + side) * 2].copy())\n\n\ndef prepare(config):\n    chosen = discover(config["drive_root"], config.get("source_override") or None)\n    work = Path(config["work_root"]).resolve()\n    work.mkdir(parents=True, exist_ok=True)\n    data = work / "dataset"\n    if data.exists():\n        raise FileExistsError("既存データは上書きしません。新しいWORK_ROOTを指定するか保存済みデータを使用してください。")\n    data.mkdir()\n    original = Path(chosen["source"])\n    if shutil.disk_usage(work).free < original.stat().st_size + 3 * 2**30:\n        raise RuntimeError("入力コピーと学習データ用に3 GiB以上の余裕が必要です。")\n    local_source = work / ("source" + original.suffix)\n    shutil.copyfile(original, local_source)\n    if identity(local_source) != identity(original):\n        raise RuntimeError("元動画のコピー検証に失敗しました。")\n    job, info = chosen["job"], chosen["source_info"]\n    fps = rate(job["fps"])\n    side = int(config.get("patch_side", 256))\n    if side < 32 or side > min(info["width"], info["height"]) or side % 8:\n        raise ValueError("patch_sideは入力内に収まる8の倍数です。")\n    rng = np.random.default_rng(20261003)\n    records, checks = [], []\n    for part, split in split_parts(chosen["parts"], int(config.get("max_parts", 6))):\n        local_teacher = work / "teacher-part.mp4"\n        shutil.copyfile(part["path"], local_teacher)\n        actual = probe(local_teacher)\n        if ((actual["width"], actual["height"]) != (job["output_width"], job["output_height"])\n                or actual["frames"] != part["frames"]\n                or abs(actual["duration"] - part["frames"] / fps) > 1 / fps + .002):\n            raise RuntimeError("完成済み区間の動画情報が完了記録と不一致です。")\n        # Far from segment boundaries: no padded context or train/val overlap.\n        for center in np.linspace(45, part["frames"] - 46, int(config.get("windows_per_part", 4))).round().astype(int):\n            first = int(center) - 3\n            lq = decode(local_source, (part["first"] + first) / fps, 7, job["fps"], info["width"], info["height"])\n            hq = decode(local_teacher, first / fps, 7, job["fps"], info["width"] * 2, info["height"] * 2)\n            check = sync_check(lq, hq)\n            checks.append(dict(check, part=part["index"], frame=int(center), split=split))\n            if not check["accepted"]:\n                continue\n            for _ in range(int(config.get("crops_per_window", 4))):\n                x = int(rng.integers(0, info["width"] - side + 1))\n                y = int(rng.integers(0, info["height"] - side + 1))\n                a, b = crop_sequence(lq, hq, x, y, side)\n                if a[3].std() < 5:\n                    continue  # No near-black/flat credits dominating the tiny pilot.\n                name = f"{split}-{len(records):05d}.npz"\n                np.savez(data / name, lq=a, hq=b)\n                records.append({"file": name, "split": split, "part": part["index"],\n                                "source_frame": part["first"] + int(center), "x": x, "y": y})\n        print(f"データ準備: 区間 {part[\'index\']} / {split} / 合計 {len(records)}サンプル", flush=True)\n    if sum(r["split"] == "train" for r in records) < 8 or sum(r["split"] == "val" for r in records) < 4:\n        raise RuntimeError("同期・シーン切替検査を通過したデータが不足しています。学習を開始しません。")\n    manifest = {"version": VERSION, "architecture": ARCH, "source_identity": identity(original),\n                "teacher_model_revision": job.get("model_revision"),\n                "teacher_mode": job["mode"], "teacher_dimensions": [job["output_width"], job["output_height"]],\n                "input_dimensions": [info["width"], info["height"]], "fps": job["fps"],\n                "same_video_holdout_only": True, "records": records, "sync_checks": checks,\n                "notice": "4K教師は2倍へ縮小。ネイティブSwiftVR 2xと同一ではない。通常映像での予備学習。"}\n    atomic_json(data / "manifest.json", manifest)\n    # Neutral names and image previews stay in the private experiment folder.\n    samples = records[:2] + [r for r in records if r["split"] == "val"][:2]\n    preview = Image.new("RGB", (side * 6, side * 2 * len(samples)))\n    for row, record in enumerate(samples):\n        with np.load(data / record["file"]) as sample:\n            low = Image.fromarray(sample["lq"][3])\n            base = low.resize((side * 2, side * 2), Image.Resampling.BICUBIC)\n            teacher = Image.fromarray(sample["hq"][1])\n            # Preserve aspect ratio: input nearest / baseline bicubic / teacher.\n            for col, image in enumerate((low, base, teacher)):\n                resample = Image.Resampling.NEAREST if col == 0 else Image.Resampling.BILINEAR\n                preview.paste(image.resize((side * 2, side * 2), resample), (col * side * 2, row * side * 2))\n    preview.save(work / "alignment-preview.png")\n    store = Path(config["save_root"])\n    store.mkdir(parents=True, exist_ok=True)\n    archive = store / "dataset.zip"\n    if archive.exists():\n        raise FileExistsError("保存済みdataset.zipは上書きしません。SAVE_ROOTを変更してください。")\n    local_zip = Path(shutil.make_archive(str(work / "dataset"), "zip", data))\n    shutil.copyfile(local_zip, archive.with_suffix(".zip.partial"))\n    os.replace(archive.with_suffix(".zip.partial"), archive)\n    shutil.copyfile(work / "alignment-preview.png", store / "alignment-preview.png")\n    atomic_json(store / "data-audit.json", {k: v for k, v in manifest.items() if k != "records"})\n    print(f"保存完了: train={sum(r[\'split\'] == \'train\' for r in records)}, val={sum(r[\'split\'] == \'val\' for r in records)}。先にalignment-preview.pngを確認してください。", flush=True)\n    return manifest\n\n\nclass ResidualBlock(nn.Module):\n    def __init__(self, channels):\n        super().__init__()\n        self.body = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(),\n                                  nn.Conv2d(channels, channels, 3, padding=1))\n\n    def forward(self, x):\n        return x + self.body(x) * 0.1\n\n\nclass TinyROIEnhancer(nn.Module):\n    """Five RGB frames in, centre RGB x2 out; static Conv/PixelShuffle only."""\n    def __init__(self, channels=24, blocks=6, window=5, scale=2):\n        super().__init__()\n        if (window != 5 or scale != 2 or type(channels) is not int or type(blocks) is not int\n                or channels < 1 or blocks < 1):\n            raise ValueError("初期試験は5フレーム・2倍のみです。")\n        self.architecture = {"channels": channels, "blocks": blocks, "window": window, "scale": scale}\n        self.expansion_origin = None\n        self.stem = nn.Conv2d(window * 3, channels, 3, padding=1)\n        self.body = nn.Sequential(*[ResidualBlock(channels) for _ in range(blocks)])\n        self.head = nn.Conv2d(channels, 3 * scale * scale, 3, padding=1)\n        self.shuffle = nn.PixelShuffle(scale)\n        nn.init.zeros_(self.head.weight)\n        nn.init.zeros_(self.head.bias)\n\n    def forward(self, frames):\n        base = F.interpolate(frames[:, 6:9], scale_factor=2., mode="bilinear", align_corners=False)\n        detail = self.shuffle(self.head(self.body(F.relu(self.stem(frames)))))\n        return base + detail  # Clamp only for display; keep gradients during training.\n\n\ndef expand_model(model, target_architecture=None):\n    """Keep all learned tensors; append identity-initialized residual blocks.\n\n    A zero LAST convolution makes each new residual block an exact identity.\n    Its first convolution stays random, allowing the last one to learn on the\n    first update. Zeroing both would strand the block at zero gradients.\n    """\n    target = validated_architecture(target_architecture or EXPANDED_ARCH)\n    if model.architecture != ARCH or target != EXPANDED_ARCH:\n        raise ValueError("拡張は既存6ブロックから12ブロックへの一回だけです。")\n    expanded = TinyROIEnhancer(**target)\n    loaded = expanded.load_state_dict(model.state_dict(), strict=False)\n    expected_missing = {f"body.{index}.body.{layer}.{tensor}"\n                        for index in range(6, 12) for layer in (0, 2) for tensor in ("weight", "bias")}\n    if set(loaded.missing_keys) != expected_missing or loaded.unexpected_keys:\n        raise ValueError("引き継げない重みがあります。拡張しません。")\n    for block in expanded.body[6:]:\n        nn.init.zeros_(block.body[2].weight)\n        nn.init.zeros_(block.body[2].bias)\n    expanded.train(model.training)\n    return expanded\n\n\ndef sequence_prediction(model, frames):\n    # [B,7,3,H,W] -> consecutive centre frames, indices 2/3/4.\n    batch, _, _, height, width = frames.shape\n    windows = torch.stack([frames[:, i:i + 5].reshape(batch, 15, height, width) for i in range(3)], 1)\n    prediction = model(windows.reshape(batch * 3, 15, height, width))\n    return prediction.reshape(batch, 3, 3, height * 2, width * 2)\n\n\ndef loss_terms(prediction, target):\n    delta = prediction.float() - target.float()\n    pixel = torch.sqrt(delta.square() + 1e-6).mean()\n    edge = ((delta[..., 1:, :] - delta[..., :-1, :]).abs().mean()\n            + (delta[..., :, 1:] - delta[..., :, :-1]).abs().mean()) * .5\n    # Match TEACHER temporal change, not zero change: real movement is not\n    # penalised merely for existing. This is not an optical-flow flicker metric.\n    temporal = (delta[:, 1:] - delta[:, :-1]).abs().mean()\n    return pixel + .1 * edge + .1 * temporal, {"pixel": pixel, "edge": edge, "temporal_delta_error": temporal}\n\n\nclass Pairs(torch.utils.data.Dataset):\n    def __init__(self, root, split, crop=128):\n        self.root = Path(root)\n        manifest = json.loads((self.root / "manifest.json").read_text())\n        if manifest["version"] != VERSION:\n            raise ValueError("データの版が不一致です。")\n        self.items = [r for r in manifest["records"] if r["split"] == split]\n        self.crop = crop if split == "train" else None\n        if not self.items:\n            raise ValueError("データがありません。")\n\n    def __len__(self):\n        return len(self.items)\n\n    def __getitem__(self, index):\n        with np.load(self.root / self.items[index]["file"]) as sample:\n            a, b = sample["lq"].copy(), sample["hq"].copy()\n        if self.crop:\n            side = self.crop\n            x = random.randint(0, a.shape[2] - side)\n            y = random.randint(0, a.shape[1] - side)\n            a, b = a[:, y:y + side, x:x + side], b[:, y * 2:(y + side) * 2, x * 2:(x + side) * 2]\n            if random.random() < .5:\n                a, b = a[:, :, ::-1], b[:, :, ::-1]\n        return (torch.from_numpy(np.ascontiguousarray(a.transpose(0, 3, 1, 2))).float() / 255,\n                torch.from_numpy(np.ascontiguousarray(b.transpose(0, 3, 1, 2))).float() / 255)\n\n\ndef restore_dataset(config):\n    import zipfile\n    data = Path(config["work_root"]) / "dataset"\n    if not (data / "manifest.json").exists():\n        data.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(Path(config["save_root"]) / "dataset.zip") as archive:\n            # Never permit archive path traversal or arbitrary file types.\n            if any(Path(n).name != n or not n.endswith((".npz", ".json")) for n in archive.namelist()):\n                raise ValueError("学習データのアーカイブ形式が不正です。")\n            archive.extractall(data)\n    return data\n\n\ndef checkpoint(path, model, optimizer, step, config, manifest):\n    state = {"version": VERSION, "architecture": model.architecture, "model": model.state_dict(),\n             "optimizer": optimizer.state_dict(), "step": step,\n             "dataset_sha256": hashlib.sha256(manifest.read_bytes()).hexdigest(),\n             "seed": int(config.get("seed", 20261003)), "app_integrated": False}\n    path = Path(path)\n    temporary = path.with_suffix(".pt.partial")\n    torch.save(state, temporary)\n    os.replace(temporary, path)\n\n\ndef load_checkpoint(path):\n    state = torch.load(path, map_location="cpu", weights_only=True)\n    if state["version"] != VERSION:\n        raise ValueError("チェックポイントのモデル形式が不一致です。")\n    architecture = validated_architecture(state["architecture"])\n    model = TinyROIEnhancer(**architecture)\n    model.load_state_dict(state["model"], strict=True)\n    if any(not torch.isfinite(value).all() for value in model.state_dict().values()):\n        raise ValueError("チェックポイントの重みが非有限値です。")\n    model.expansion_origin = state.get("expansion_origin")\n    return model, state\n\n\ndef evaluate(model, data, device, out, max_samples=16):\n    model.eval()\n    dataset = Pairs(data, "val")\n    errors, baseline_errors, temporal, baseline_temporal, elapsed = [], [], [], [], []\n    preview = []\n    def sync():\n        if device.type == "cuda":\n            torch.cuda.synchronize()\n        elif device.type == "mps":\n            torch.mps.synchronize()\n    with torch.inference_mode():\n        for i in range(min(len(dataset), max_samples)):\n            a, b = dataset[i]\n            a, b = a.unsqueeze(0).to(device), b.unsqueeze(0).to(device)\n            if i == 0:\n                sequence_prediction(model, a)\n            sync()\n            start = time.perf_counter()\n            prediction = sequence_prediction(model, a)\n            sync()\n            elapsed.append(time.perf_counter() - start)\n            base = F.interpolate(a[:, 2:5].reshape(3, 3, a.shape[-2], a.shape[-1]), scale_factor=2,\n                                 mode="bilinear", align_corners=False).unsqueeze(0)\n            prediction = prediction.clamp(0, 1)\n            errors.append(float((prediction - b).square().mean()))\n            baseline_errors.append(float((base - b).square().mean()))\n            temporal.append(float(((prediction[:, 1:] - prediction[:, :-1]) - (b[:, 1:] - b[:, :-1])).abs().mean()))\n            baseline_temporal.append(float(((base[:, 1:] - base[:, :-1]) - (b[:, 1:] - b[:, :-1])).abs().mean()))\n            if i < 4:\n                images = [base[0, 1], prediction[0, 1], b[0, 1]]\n                preview.append([Image.fromarray((x.cpu().permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)) for x in images])\n    mse, base_mse = float(np.mean(errors)), float(np.mean(baseline_errors))\n    report = {"samples": len(errors), "student_psnr_to_teacher": -10 * math.log10(max(mse, 1e-12)),\n              "baseline_psnr_to_teacher": -10 * math.log10(max(base_mse, 1e-12)),\n              "student_temporal_delta_error": float(np.mean(temporal)),\n              "baseline_temporal_delta_error": float(np.mean(baseline_temporal)),\n              "model_only_fps": len(errors) * 3 / sum(elapsed), "device": device.type,\n              "parameters": sum(p.numel() for p in model.parameters()),\n              "comparison": "bilinear / student / resized SwiftVR teacher; NOT native SwiftVR speed comparison",\n              "same_video_holdout_only": True, "restored_roi_evaluated": False, "app_integrated": False,\n              "provisional_gate": mse < base_mse and float(np.mean(temporal)) <= float(np.mean(baseline_temporal)) * 1.05}\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    atomic_json(out / "evaluation.json", report)\n    if preview:\n        size = preview[0][0].width\n        image = Image.new("RGB", (size * 3, size * len(preview)))\n        for row, images in enumerate(preview):\n            for col, frame in enumerate(images):\n                image.paste(frame, (col * size, row * size))\n        image.save(out / "comparison.png")\n    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)\n    return report\n\n\ndef train(config):\n    device = torch.device(config.get("device", "cuda"))\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("学習セルはGPUランタイムで実行してください。")\n    minutes = float(config.get("max_minutes", 45))\n    steps = int(config.get("max_steps", 1500))\n    if not 0 < minutes <= 120 or not 0 < steps <= 10000:\n        raise ValueError("試験上限は1回120分 / 10000ステップです。")\n    seed = int(config.get("seed", 20261003))\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.set_num_threads(min(4, os.cpu_count() or 1))\n    data = restore_dataset(config)\n    store = Path(config["save_root"])\n    store.mkdir(parents=True, exist_ok=True)\n    latest = store / "latest.pt"\n    model = TinyROIEnhancer(**ARCH).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0)\n    step = 0\n    digest = hashlib.sha256((data / "manifest.json").read_bytes()).hexdigest()\n    if latest.exists():\n        model, state = load_checkpoint(latest)\n        if state["dataset_sha256"] != digest:\n            raise ValueError("別のデータのチェックポイントです。新しいSAVE_ROOTを使用してください。")\n        model.to(device)\n        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0)\n        optimizer.load_state_dict(state["optimizer"])\n        step = int(state["step"])\n        # Optimizer tensors loaded on CPU must follow the CUDA/MPS model.\n        for value in optimizer.state.values():\n            for key, tensor in value.items():\n                if isinstance(tensor, torch.Tensor) and key != "step":\n                    value[key] = tensor.to(device)\n    first_step = step\n    dataset = Pairs(data, "train", crop=int(config.get("train_crop", 128)))\n    loader = torch.utils.data.DataLoader(dataset, batch_size=int(config.get("batch_size", 2)), shuffle=True,\n                                         num_workers=0, pin_memory=device.type == "cuda")\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n    started = time.monotonic()\n    log = store / "train.jsonl"\n    interrupted = False\n    try:\n        model.train()\n        while step < steps and time.monotonic() - started < minutes * 60:\n            for a, b in loader:\n                if step >= steps or time.monotonic() - started >= minutes * 60:\n                    break\n                a, b = a.to(device), b.to(device)\n                optimizer.zero_grad(set_to_none=True)\n                amp = torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else contextlib.nullcontext()\n                with amp:\n                    prediction = sequence_prediction(model, a)\n                    loss, terms = loss_terms(prediction, b)\n                if not torch.isfinite(loss):\n                    raise RuntimeError("損失が非有限値です。学習を中止します。")\n                scaler.scale(loss).backward()\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)\n                scaler.step(optimizer)\n                scaler.update()\n                step += 1\n                if step % 50 == 0:\n                    row = {"step": step, "elapsed_seconds": time.monotonic() - started,\n                           "loss": float(loss.detach()), **{k: float(v.detach()) for k, v in terms.items()}}\n                    with log.open("a") as handle:\n                        handle.write(json.dumps(row) + "\\n")\n                    print(f"step={step} loss={row[\'loss\']:.5f} / {row[\'elapsed_seconds\']/60:.1f}分", flush=True)\n                if step % 200 == 0:\n                    checkpoint(latest, model, optimizer, step, config, data / "manifest.json")\n    except KeyboardInterrupt:\n        interrupted = True\n    finally:\n        checkpoint(latest, model, optimizer, step, config, data / "manifest.json")\n        atomic_json(store / "run-status.json", {"step": step, "steps_this_run": step - first_step,\n                    "train_seconds": time.monotonic() - started, "interrupted": interrupted,\n                    "training_returned": True, "quality_accepted": False,\n                    "compute_units_measured": False, "app_integrated": False})\n    return evaluate(model, data, device, store)\n\n\ndef export_coreml(checkpoint_path, output):\n    import coremltools as ct\n    output = Path(output)\n    if output.exists():\n        raise FileExistsError(output)\n    model, _ = load_checkpoint(checkpoint_path)\n    model.eval()\n    example = torch.rand(1, 15, 256, 256)\n    traced = torch.jit.trace(model, example)\n    converted = ct.convert(traced, convert_to="mlprogram", minimum_deployment_target=ct.target.macOS13,\n                           inputs=[ct.TensorType(name="frames", shape=example.shape, dtype=np.float32)],\n                           outputs=[ct.TensorType(name="enhanced", dtype=np.float32)],\n                           compute_precision=ct.precision.FLOAT16)\n    converted.short_description = "Experimental SwiftVR-output distilled ROI enhancer x2"\n    converted.user_defined_metadata["contract"] = "NCHW [1,15,256,256], RGB 0..1, chronological t-2..t+2; output [1,3,512,512]"\n    output.parent.mkdir(parents=True, exist_ok=True)\n    converted.save(str(output))\n    report = {"runtime_verified": False, "app_integrated": False}\n    if os.uname().sysname == "Darwin":\n        runtime = ct.models.MLModel(str(output), compute_units=ct.ComputeUnit.CPU_AND_GPU)\n        x = example.numpy()\n        with torch.inference_mode():\n            expected = model(example).numpy()\n        actual = runtime.predict({"frames": x})["enhanced"]\n        error = np.abs(expected - actual)\n        if not np.isfinite(actual).all() or error.max() > .05:\n            raise RuntimeError("Core ML数値一致検査に失敗しました。採用しません。")\n        runtime.predict({"frames": x})\n        times = []\n        for _ in range(10):\n            start = time.perf_counter()\n            runtime.predict({"frames": x})\n            times.append(time.perf_counter() - start)\n        report.update(runtime_verified=True, max_abs_error=float(error.max()), mean_abs_error=float(error.mean()),\n                      median_ms=float(np.median(times)) * 1000,\n                      model_only_fps=1 / float(np.median(times)), compute_units="CPU_AND_GPU")\n    atomic_json(output.with_suffix(".validation.json"), report)\n    print(json.dumps(report, indent=2))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("action", choices=("prepare", "train", "evaluate", "export"))\n    parser.add_argument("--config", type=Path)\n    parser.add_argument("--checkpoint", type=Path)\n    parser.add_argument("--output", type=Path)\n    args = parser.parse_args()\n    if args.action == "export":\n        if not args.checkpoint or not args.output:\n            parser.error("exportには--checkpointと--outputが必要です。")\n        export_coreml(args.checkpoint, args.output)\n        return\n    if not args.config:\n        parser.error("--configが必要です。")\n    config = json.loads(args.config.read_text())\n    if args.action == "prepare":\n        prepare(config)\n    elif args.action == "train":\n        train(config)\n    else:\n        model, _ = load_checkpoint(Path(config["save_root"]) / "latest.pt")\n        device = torch.device(config.get("device", "cuda"))\n        evaluate(model.to(device), restore_dataset(config), device, config["save_root"])\n\n\nif __name__ == "__main__":\n    main()\n', 'roi_mask_detail_runtime.py': '"""Private paired ROI detail training. No media/model/app is overwritten.\n\nInput: restored RGB, mosaic RGB, exported mask; target: clear RGB.\nNative sidecars are scanned one frame at a time. CFR windows are read by\npresentation timestamp, WITHOUT an fps filter or silent frame duplication.\n"""\nfrom __future__ import annotations\n\nfrom fractions import Fraction\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport time\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nimport roi_enhancer_distill_runtime as base\n\nVERSION = "roi-mask-detail-v1"\n\n\ndef sha256(path):\n    result = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):\n            result.update(chunk)\n    return result.hexdigest()\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()\n\n\ndef validate_roots(config):\n    dataset = Path(config["dataset_root"]).resolve()\n    prepared = Path(config["prepared_root"]).resolve()\n    save = Path(config["save_root"]).resolve()\n    work = Path(config["work_root"]).resolve()\n    initial = Path(config["init_checkpoint"]).resolve()\n    outputs = [prepared, save, work]\n    for i, a in enumerate(outputs):\n        if a == dataset or a in dataset.parents or dataset in a.parents:\n            raise ValueError("素材フォルダと作業・保存先を分離してください。")\n        if a == initial.parent or a in initial.parents:\n            raise ValueError("元のbest.ptの保存フォルダは出力先にできません。")\n        for b in outputs[i + 1:]:\n            if a == b or a in b.parents or b in a.parents:\n                raise ValueError("準備・学習・作業フォルダは互いに独立させてください。")\n    return dataset, prepared, save, work\n\n\ndef discover(root, overrides=None):\n    root = Path(root)\n    overrides = overrides or {}\n    items = []\n    for source in sorted((root / "mosaic").glob("*.mp4")):\n        # Existing upload layout; no private filenames are embedded in code.\n        target_name = source.name.replace("-mosaic_", "-UC_", 1)\n        restored = root / "restoration" / (source.stem + "-UC.mp4")\n        masks = restored.with_suffix(".masks.json")\n        target = root / "uncensored" / target_name\n        values = {"mosaic": str(source), "restoration": str(restored),\n                  "target": str(target), "masks": str(masks), "target_offset": 0}\n        extra = overrides.get(source.name, {})\n        if set(extra) - set(values):\n            raise ValueError("対応素材の指定項目が不正です。")\n        values.update(extra)\n        if type(values["target_offset"]) is not int or abs(values["target_offset"]) > 5:\n            raise ValueError("target_offsetは検証済みの-5〜5フレームで指定してください。")\n        for key in ("mosaic", "restoration", "target", "masks"):\n            if not Path(values[key]).is_file():\n                raise FileNotFoundError(f"対応する{key}がありません: {Path(values[key]).name}")\n        values["id"] = f"pair-{len(items):02d}"\n        items.append(values)\n    if set(overrides) - {Path(p["mosaic"]).name for p in items}:\n        raise ValueError("対応上書きの素材名が選択素材にありません。")\n    if len(items) < 3:\n        raise ValueError("区間単位でtrain/val/testを分けるため最低3組が必要です。")\n    return items\n\n\ndef video_info(path):\n    import av\n    with av.open(str(path)) as container:\n        stream = container.streams.video[0]\n        if not stream.frames or stream.start_time is None or stream.average_rate is None:\n            raise ValueError("正確なフレーム数・開始PTS・fpsがあるCFR動画が必要です。")\n        if stream.metadata.get("rotate", "0") not in ("0", ""):\n            raise ValueError("回転メタデータのある動画は座標の事前検証が必要です。")\n        rate = stream.average_rate\n        # A longer last frame shifts the average (total length within 1.5 frames): the nominal rate\n        # is the frame grid then (decode_indices still checks every PTS against it).\n        if stream.guessed_rate and stream.frames and abs(stream.frames*(float(stream.guessed_rate/rate)-1)) <= 1.5:\n            rate = stream.guessed_rate\n        return {"width": stream.width, "height": stream.height,\n                "frames": int(stream.frames), "start_pts": int(stream.start_time),\n                "time_base": str(stream.time_base), "fps": str(rate)}\n\n\ndef decode_indices(path, info, indices):\n    import av\n    indices = list(indices)\n    if not indices or min(indices) < 0 or max(indices) >= info["frames"]:\n        raise ValueError("読み出しフレームが動画範囲外です。")\n    wanted = set(indices)\n    rate = Fraction(info["fps"])\n    time_base = Fraction(info["time_base"])\n    frames = {}\n    with av.open(str(path)) as container:\n        stream = container.streams.video[0]\n        stream.thread_type = "AUTO"\n        start = info["start_pts"] + int(Fraction(min(indices), 1) / rate / time_base)\n        container.seek(start, stream=stream, backward=True, any_frame=False)\n        for frame in container.decode(stream):\n            if frame.pts is None:\n                raise ValueError("PTSのないフレームです。")\n            ordinal = (frame.pts * frame.time_base - info["start_pts"] * time_base) * rate\n            index = round(ordinal)\n            if abs(float(ordinal - index)) > .02:\n                raise ValueError("CFRのフレーム番号とPTSが不一致です。変換せず停止します。")\n            if index > max(indices):\n                break\n            if index in wanted:\n                if index in frames:\n                    raise ValueError("重複したフレームPTSです。")\n                frames[index] = frame.to_ndarray(format="rgb24")\n    if set(frames) != wanted:\n        raise RuntimeError("必要な実フレームを読み出せませんでした。")\n    return np.stack([frames[i] for i in indices])\n\n\ndef restore_alignment(info, black_frame):\n    """Verified virtual prepend: logical t in restoration -> source t-1."""\n    restored, mosaic, target = (info[k] for k in ("restoration", "mosaic", "target"))\n    if any((v["width"], v["height"], v["fps"]) !=\n           (restored["width"], restored["height"], restored["fps"])\n           for v in (mosaic, target)):\n        raise ValueError("3本の解像度・fpsが一致しません。自動変換はしません。")\n    if target["frames"] != mosaic["frames"]:\n        raise ValueError("モザイク版と正解のフレーム数が違います。再確認してください。")\n    if restored["frames"] == mosaic["frames"] and black_frame.max() > 3:\n        # Clips that start at PTS 0: the export keeps every frame in place (no prepend).\n        # The per-centre sync audit still checks lag 0 against the mosaic.\n        return 0\n    if restored["frames"] != mosaic["frames"] + 1:\n        raise ValueError("想定した先頭1フレーム差ではありません。再確認してください。")\n    if black_frame.max() > 3:\n        raise ValueError("復元動画の先頭が黒くありません。仮想1フレーム補正は適用できません。")\n    return 1\n\n\ndef stream_mask_index(path, wanted, expected_count, dimensions, expected_fps):\n    """Index selected rows only. Validate the complete native streamed JSON.\n\n    Each native frame is one JSON line. We do not json.load the whole file;\n    arbitrary pretty-printed external JSON is intentionally not supported.\n    """\n    offsets = {}\n    count = 0\n    previous_pts = None\n    with Path(path).open("rb") as handle:\n        line = handle.readline(4 * 1024 * 1024)\n        marker = b\',"frames":[\\n\'\n        if not line.endswith(marker):\n            raise ValueError("miohの逐次書き出しJSON形式ではありません。")\n        header = json.loads(line[:-len(marker)] + b"}")\n        if (header.get("schema") != "mioh-detection-masks-v1" or\n                [header.get("height"), header.get("width")] != list(dimensions) or\n                header.get("frame_index_base") != 0 or\n                header.get("coordinate_system") != "oriented-source-pixels-top-left"):\n            raise ValueError("マスクJSONの形式・座標・映像寸法が不一致です。")\n        for key in ("source_fps", "output_fps"):\n            fps = header[key]\n            if Fraction(fps["numerator"], fps["denominator"]) != Fraction(expected_fps):\n                raise ValueError("マスクJSONにfps変換があります。この補正では扱えません。")\n        while True:\n            position = handle.tell()\n            line = handle.readline(64 * 1024 * 1024)\n            if not line:\n                raise ValueError("マスクJSONが途中で切れています。")\n            if line.startswith(b"]"):\n                footer = json.loads(b"{" + line.strip()[2:])  # ],"frame_count":...\n                if (footer.get("complete") is not True or footer.get("frame_count") != count\n                        or count != expected_count or handle.read(1)):\n                    raise ValueError("マスクJSONが未完了、または動画のフレーム数と不一致です。")\n                break\n            if not line.strip():\n                continue  # final frame\'s newline before footer\n            if len(line) >= 64 * 1024 * 1024:\n                raise ValueError("1フレームのJSONが大きすぎます。")\n            row = json.loads(line.strip().rstrip(b","))\n            pts = row["source_pts_ns"]\n            if row["frame_index"] != count or (previous_pts is not None and pts <= previous_pts):\n                raise ValueError("JSONのフレーム順序・PTSが不正です。")\n            if abs(pts / 1e9 - count / float(Fraction(expected_fps))) > .02 / float(Fraction(expected_fps)):\n                raise ValueError("JSONのPTSと出力番号が想定したCFR関係ではありません。")\n            previous_pts = pts\n            if count in wanted:\n                offsets[count] = position\n            count += 1\n            if count % 10000 == 0:\n                print(f"マスクJSON走査: {count}/{expected_count}フレーム", flush=True)\n        if set(offsets) != set(wanted):\n            raise ValueError("必要なマスクフレームがありません。")\n    return header, offsets\n\n\ndef integer_runs(counts, total):\n    if (any(type(x) is not int or x < 0 for x in counts) or sum(counts) != total):\n        raise ValueError("マスクRLEの長さが不正です。")\n    return counts\n\n\ndef raw_mask(row, height, width):\n    mask = np.zeros((height, width), np.float32)\n    for region in row["detections"]:\n        x, y, w, h = region["bbox"]\n        rle = region["mask"]\n        if (min(x, y) < 0 or min(w, h) <= 0 or x + w > width or y + h > height\n                or rle["size"] != [h, w] or rle.get("order") != "row-major"):\n            raise ValueError("検出マスクの座標が不正です。")\n        runs = integer_runs(rle["counts"], h * w)\n        data = np.repeat(np.arange(len(runs)) % 2, runs).reshape(h, w)\n        mask[y:y+h, x:x+w] = np.maximum(mask[y:y+h, x:x+w], data)\n    return mask\n\n\ndef celf_mask(field, height, width):\n    if field.get("stage") != "after_tracker_and_gap_fill":\n        raise ValueError("補正後のCELF情報ではありません。")\n    gh, gw = field["grid_size"]\n    cell = field["cell_size"]\n    x, y, w, h = field["stored_rect"]\n    if (type(cell) is not int or cell < 1 or [gh, gw] != [math.ceil(height/cell), math.ceil(width/cell)]\n            or min(x, y, w, h) < 0 or x+w > gw or y+h > gh):\n        raise ValueError("CELFのセル座標が不正です。")\n    arrays = []\n    for key in ("owners", "coverage_u8"):\n        rle = field[key]\n        lengths = integer_runs(rle["lengths"], w * h)\n        if len(rle["values"]) != len(lengths) or rle.get("order") != "row-major":\n            raise ValueError("CELFのRLEが不正です。")\n        values = np.asarray(rle["values"], np.int64)\n        if np.any(values < 0) or (key == "coverage_u8" and np.any(values > 255)):\n            raise ValueError("CELFのRLE値が不正です。")\n        array = np.zeros((gh, gw), np.int64)\n        array[y:y+h, x:x+w] = np.repeat(values, lengths).reshape(h, w)\n        arrays.append(array)\n    coverage = np.where(arrays[0] > 0, arrays[1], 0).astype(np.float32) / 255\n    # Cell-level training coverage, NOT exact compositor alpha. No fake\n    # pixel-accurate ownership is inferred from the exported 4px cell grid.\n    return np.repeat(np.repeat(coverage, cell, 0), cell, 1)[:height, :width]\n\n\ndef training_mask(row, height, width, mode="corrected"):\n    if mode not in ("corrected", "raw"):\n        raise ValueError("マスク方式はcorrected/rawです。")\n    if mode == "raw" or "celf" not in row:\n        return raw_mask(row, height, width)\n    mask = celf_mask(row["celf"], height, width)\n    if "celf_crossfade_previous" in row:\n        weight = float(row["crossfade_later_weight"])\n        if not math.isfinite(weight) or not 0 <= weight <= 1:\n            raise ValueError("クロスフェード重みが不正です。")\n        mask = (mask * weight +\n                celf_mask(row["celf_crossfade_previous"], height, width) * (1-weight))\n    return mask\n\n\ndef sync_check(a, b, ignored):\n    """Outside-ROI low-frequency +/-2 lag audit; never relax to get samples."""\n    small_a, small_b, good = [], [], []\n    for aa, bb, mm in zip(a, b, ignored):\n        small_a.append(np.asarray(Image.fromarray(aa).resize((80, 45), Image.Resampling.BILINEAR), np.float32)/255)\n        small_b.append(np.asarray(Image.fromarray(bb).resize((80, 45), Image.Resampling.BILINEAR), np.float32)/255)\n        good.append(np.asarray(Image.fromarray((mm*255).astype(np.uint8)).resize((80,45), Image.Resampling.BOX)) < 3)\n    a, b, good = np.stack(small_a), np.stack(small_b), np.stack(good)\n    scores = {}\n    for lag in range(-2, 3):\n        support = good[2:-2] & good[2+lag:len(good)-2+lag]\n        if support.mean() < .2:\n            return {"accepted": False, "reason": "outside_roi_insufficient"}\n        delta = (a[2:-2]-b[2+lag:len(b)-2+lag])**2\n        scores[lag] = float(delta[support].mean())\n    best = min(scores, key=scores.get)\n    accepted = scores[0] <= .015 and scores[0] <= scores[best]*1.15 + .0001\n    # Abrupt differences are audited outside masked areas; moving ROI pixels\n    # alone must not look like a scene cut.\n    adjacent_support = good[1:] & good[:-1]\n    cut = any(float(np.abs(a[i+1]-a[i])[adjacent_support[i]].mean()) > .18\n              for i in range(len(a)-1) if adjacent_support[i].any())\n    return {"accepted": bool(accepted and not cut), "best_lag": best,\n            "zero_lag_mse": scores[0], "scene_cut": cut,\n            "ambiguous": max(scores.values()) - min(scores.values()) < .0001}\n\n\ndef windows_and_splits(items, interval):\n    if interval < 1:\n        raise ValueError("抽出間隔は1秒以上にしてください。")\n    tasks = []\n    for i, item in enumerate(items):\n        split = "test" if i == len(items)-1 else "val" if i == len(items)-2 else "train"\n        item["split"] = split  # Whole files, never random patches across splits.\n        info = item["info"]\n        fps = float(Fraction(info["restoration"]["fps"]))\n        end = min(info["restoration"]["frames"]-5,\n                  info["target"]["frames"] + 1 - item["target_offset"]-5)\n        first_center = max(5,5-item["target_offset"])\n        centers = list(range(first_center, end, max(1, round(interval*fps))))\n        item["centers"] = centers\n        for first in range(0, len(centers), 16):\n            tasks.append({"pair": i, "centers": centers[first:first+16], "split": split})\n    return tasks\n\n\ndef cached_copy(source, destination):\n    source, destination = Path(source), Path(destination)\n    signature = {**base.identity(source), "mtime_ns": source.stat().st_mtime_ns}\n    stamp = destination.with_suffix(destination.suffix + ".identity.json")\n    if destination.is_file() and stamp.is_file():\n        saved = json.loads(stamp.read_text())\n        if saved["source"] == signature and base.identity(destination) == saved["local"]:\n            return saved\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if shutil.disk_usage(destination.parent).free < source.stat().st_size + 2*2**30:\n        raise RuntimeError("ローカルSSDの空き容量が不足しています。")\n    print(f"素材をローカルへコピー: {destination.name} ({source.stat().st_size/2**30:.2f} GiB)", flush=True)\n    temporary = destination.with_suffix(destination.suffix + ".partial")\n    shutil.copyfile(source, temporary)\n    if base.identity(temporary) != base.identity(source):\n        raise RuntimeError("素材コピーの照合に失敗しました。")\n    os.replace(temporary, destination)\n    saved = {"source": signature, "local": base.identity(destination), "sha256": sha256(destination)}\n    base.atomic_json(stamp, saved)\n    return saved\n\n\ndef prepare(config):\n    dataset, prepared, _, work = validate_roots(config)\n    side = int(config.get("patch_side", 128))\n    if side < 32 or side > 256 or side % 8:\n        raise ValueError("学習パッチは32〜256px、8の倍数です。")\n    items = discover(dataset, config.get("material_overrides"))\n    originals = []\n    for number,item in enumerate(items,1):\n        print(f"素材照合: {number}/{len(items)}組（解像度・fps・先頭フレーム）",flush=True)\n        originals.append({k: {"name": Path(item[k]).name, **base.identity(item[k]),\n                             "mtime_ns": Path(item[k]).stat().st_mtime_ns}\n                          for k in ("mosaic", "restoration", "target", "masks")})\n        item["info"] = {k: video_info(item[k]) for k in ("mosaic", "restoration", "target")}\n        zero = decode_indices(item["restoration"], item["info"]["restoration"], [0])[0]\n        restore_alignment(item["info"], zero)\n        if side > min(item["info"]["mosaic"]["width"], item["info"]["mosaic"]["height"]):\n            raise ValueError("素材がパッチより小さいです。")\n    contract = {"version": VERSION, "originals": originals,\n                "offsets": [p["target_offset"] for p in items],\n                "patch_side": side, "interval_seconds": float(config.get("interval_seconds", 4)),\n                "crops_per_window": int(config.get("crops_per_window", 4)),\n                "mask_mode": config.get("mask_mode", "corrected"),\n                "seed": int(config.get("seed", 20261005))}\n    if not 1 <= contract["crops_per_window"] <= 16:\n        raise ValueError("地点あたりパッチ数は1〜16です。")\n    signature = digest(contract)\n    prepared.mkdir(parents=True, exist_ok=True)\n    state_path = prepared / "preparation.json"\n    if state_path.exists():\n        state = json.loads(state_path.read_text())\n        if state["signature"] != signature:\n            raise ValueError("保存済み素材と設定が異なります。新しいPREPARED_ROOTを指定してください。")\n    else:\n        if any(prepared.iterdir()):\n            raise ValueError("素材準備の保存先が空ではありません。新しいフォルダを指定してください。")\n        state = {"signature": signature, "contract": contract, "groups": [], "checks": []}\n        base.atomic_json(state_path, state)\n    tasks = windows_and_splits(items, contract["interval_seconds"])\n    for group in state["groups"]:\n        if sha256(prepared / group["file"]) != group["sha256"]:\n            raise ValueError("準備済みパッチが破損しています。上書きせず停止します。")\n    done = {g["task"] for g in state["groups"]}\n    started = time.monotonic()\n    minutes = float(config.get("prepare_minutes", 120))\n    if not math.isfinite(minutes) or minutes <= 0:\n        raise ValueError("素材準備の時間上限は正の分数です。")\n    for pair_index, item in enumerate(items):\n        pending = [(n,t) for n,t in enumerate(tasks) if t["pair"] == pair_index and n not in done]\n        if not pending:\n            continue\n        local = work / "media" / item["id"]\n        paths = {}\n        for key in ("mosaic", "restoration", "target", "masks"):\n            suffix = ".json" if key == "masks" else ".mp4"\n            paths[key] = local / (key + suffix)\n            cached_copy(item[key], paths[key])\n        wanted = {j for _, task in pending for c in task["centers"] for j in range(c-4,c+5)}\n        height, width = item["info"]["restoration"]["height"], item["info"]["restoration"]["width"]\n        index_path = local / "mask-index.json"\n        if index_path.is_file():\n            cached = json.loads(index_path.read_text())\n            if cached["signature"] == signature and wanted <= set(map(int,cached["offsets"])):\n                offsets = {int(k):v for k,v in cached["offsets"].items()}\n            else:\n                offsets = None\n        else:\n            offsets = None\n        if offsets is None:\n            _, offsets = stream_mask_index(paths["masks"], wanted,\n                item["info"]["restoration"]["frames"], (height,width), item["info"]["restoration"]["fps"])\n            base.atomic_json(index_path, {"signature": signature, "offsets": offsets})\n        with paths["masks"].open("rb") as mask_file:\n            for task_id, task in pending:\n                arrays = {k: [] for k in ("restoration", "mosaic", "target", "mask")}\n                records, checks = [], []\n                rng = np.random.default_rng(contract["seed"] + task_id)\n                for center in task["centers"]:\n                    logical = np.arange(center-4,center+5)\n                    masks = []\n                    for frame in logical:\n                        mask_file.seek(offsets[int(frame)])\n                        row = json.loads(mask_file.readline().strip().rstrip(b","))\n                        masks.append(training_mask(row, height, width, contract["mask_mode"]))\n                    masks = np.stack(masks)\n                    if not np.any(masks[4] > .05):\n                        checks.append({"pair": item["id"], "center": center, "reason": "no_roi"})\n                        continue\n                    restored = decode_indices(paths["restoration"], item["info"]["restoration"], logical)\n                    mosaic = decode_indices(paths["mosaic"], item["info"]["mosaic"], logical-1)\n                    target = decode_indices(paths["target"], item["info"]["target"], logical-1+item["target_offset"])\n                    a = sync_check(mosaic, restored, masks)\n                    b = sync_check(mosaic, target, masks)\n                    checks.append({"pair": item["id"], "center": center, "restored": a, "target": b})\n                    if not a["accepted"] or not b["accepted"]:\n                        continue\n                    # All 7 temporal frames share one source-coordinate crop.\n                    yy, xx = np.where(masks[4] > .05)\n                    taken = set()\n                    for _ in range(contract["crops_per_window"] * 8):\n                        if len(taken) >= contract["crops_per_window"]:\n                            break\n                        point = int(rng.integers(len(xx)))\n                        x = int(np.clip(xx[point]-side//2+rng.integers(-side//4,side//4+1), 0, width-side))\n                        y = int(np.clip(yy[point]-side//2+rng.integers(-side//4,side//4+1), 0, height-side))\n                        if (x,y) in taken:\n                            continue\n                        roi = masks[1:8, y:y+side, x:x+side]\n                        if min(float(roi[k].mean()) for k in (2,3,4)) < .01:\n                            continue\n                        taken.add((x,y))\n                        arrays["restoration"].append(restored[1:8,y:y+side,x:x+side])\n                        arrays["mosaic"].append(mosaic[1:8,y:y+side,x:x+side])\n                        arrays["target"].append(target[3:6,y:y+side,x:x+side])\n                        arrays["mask"].append(np.rint(roi*255).astype(np.uint8))\n                        records.append({"center": center, "xy": [x,y]})\n                filename = f"shard-{task_id:05d}.npz"\n                local_shard = work / filename\n                work.mkdir(parents=True, exist_ok=True)\n                np.savez_compressed(local_shard, **{k: np.stack(v) if v else np.empty((0,),np.uint8)\n                                                    for k,v in arrays.items()})\n                if shutil.disk_usage(work).free < 2*2**30:\n                    raise RuntimeError("ローカルSSDが残り2 GiB未満です。")\n                temporary = prepared / (filename + ".partial")\n                shutil.copyfile(local_shard, temporary)\n                os.replace(temporary, prepared / filename)\n                group = {"task": task_id, "file": filename, "sha256": sha256(local_shard),\n                         "pair": item["id"], "split": task["split"], "records": records}\n                state["groups"].append(group)\n                state["checks"].extend(checks)\n                base.atomic_json(state_path, state)\n                counts = {s:sum(len(g["records"]) for g in state["groups"] if g["split"]==s)\n                          for s in ("train","val","test")}\n                print(f"素材準備 {len(state[\'groups\'])}/{len(tasks)}区間: {counts}", flush=True)\n                if time.monotonic()-started >= minutes*60:\n                    print("時間上限です。同じセルを再実行すると完成区間を再利用します。", flush=True)\n                    return {"complete": False, "counts": counts}\n    counts = {s:sum(len(g["records"]) for g in state["groups"] if g["split"]==s)\n              for s in ("train","val","test")}\n    checks = state["checks"]\n    manifest = {"version": VERSION, "signature": signature, "contract": contract,\n                "groups": state["groups"], "checks": checks, "counts": counts,\n                "complete": True, "virtual_black_prepend": True, "black_frames_excluded": True,\n                "split_unit": "whole-video-file", "same_production_only": True}\n    base.atomic_json(prepared / "audit.json", manifest)\n    minimum = config.get("minimum_samples", {"train":1000,"val":100,"test":100})\n    if any(counts[s] < minimum[s] for s in counts):\n        raise RuntimeError(f"検査済みROI素材が不足: {counts}。audit.jsonを確認し、同期条件を緩めず原因を確認してください。")\n    base.atomic_json(prepared / "manifest.json", manifest)\n    print(f"素材準備完了: {counts}。まだ学習していません。", flush=True)\n    return manifest\n\n\nclass ConditionalROI(nn.Module):\n    """Old learned RGB branch + zero-init mosaic/mask conditioning branch.\n\n    Prediction is a SAME-SIZE residual correction to restored RGB. We keep the\n    old x2 head internally then map its residual back; this is NOT the old app\n    input contract and requires a separate adapter/export before adoption.\n    """\n    def __init__(self, pretrained):\n        super().__init__()\n        self.backbone = pretrained\n        self.condition = nn.Conv2d(20, pretrained.architecture["channels"], 3, padding=1)\n        nn.init.zeros_(self.condition.weight)\n        nn.init.zeros_(self.condition.bias)\n\n    def forward(self, restored, mosaic, masks):\n        # Each argument [B,5,C,H,W]; masks C=1, frames C=3.\n        batch, _, _, h, w = restored.shape\n        frames = restored.reshape(batch,15,h,w)\n        condition = torch.cat((mosaic.reshape(batch,15,h,w), masks.reshape(batch,5,h,w)),1)\n        features = F.relu(self.backbone.stem(frames) + self.condition(condition))\n        residual = self.backbone.shuffle(self.backbone.head(self.backbone.body(features)))\n        residual = F.interpolate(residual, size=(h,w), mode="area")\n        center = restored[:,2]\n        # Mask=0 is an EXACT bypass, not merely an outside-mask loss.\n        return center + masks[:,2] * residual\n\n\ndef sequence_prediction(model, restored, mosaic, mask):\n    batch, _, _, h, w = restored.shape\n    arguments = []\n    for frames in (restored,mosaic,mask):\n        arguments.append(torch.stack([frames[:,i:i+5] for i in range(3)],1).reshape(batch*3,5,frames.shape[2],h,w))\n    return model(*arguments).reshape(batch,3,3,h,w)\n\n\ndef weighted_mean(value, weight):\n    weight = weight.expand_as(value)\n    return (value*weight).sum()/weight.sum().clamp_min(1e-8)\n\n\ndef loss_terms(prediction, target, mask, lpips_model=None, lpips_weight=.05):\n    prediction, target, mask = prediction.float(), target.float(), mask.float()\n    error = prediction-target\n    pixel = weighted_mean(torch.sqrt(error.square()+1e-6), mask)\n    gx = weighted_mean((error[...,1:]-error[...,:-1]).abs(), torch.minimum(mask[...,1:],mask[...,:-1]))\n    gy = weighted_mean((error[...,1:,:]-error[...,:-1,:]).abs(), torch.minimum(mask[...,1:,:],mask[...,:-1,:]))\n    flat = error.reshape(-1,3,*error.shape[-2:])\n    high = (flat-F.avg_pool2d(flat,5,stride=1,padding=2)).reshape_as(error)\n    inner = -F.max_pool2d(-mask.reshape(-1,1,*mask.shape[-2:]),5,stride=1,padding=2)\n    inner = inner.reshape_as(mask)\n    highpass = weighted_mean(high.abs(), inner)\n    support = torch.minimum(mask[:,1:],mask[:,:-1])\n    temporal = weighted_mean((error[:,1:]-error[:,:-1]).abs(),support)\n    hp_temporal = weighted_mean((high[:,1:]-high[:,:-1]).abs(),torch.minimum(inner[:,1:],inner[:,:-1]))\n    perceptual = error.new_zeros(())\n    if lpips_model is not None and lpips_weight:\n        b,t,c,h,w = prediction.shape\n        # Outside-ROI content identical in prediction and target inputs.\n        composed_target = prediction.detach()*(1-mask)+target*mask\n        distances = lpips_model(prediction.reshape(b*t,c,h,w)*2-1,\n                                 composed_target.reshape(b*t,c,h,w)*2-1)\n        weights = F.interpolate(mask.reshape(b*t,1,h,w),size=distances.shape[-2:],mode="area")\n        perceptual = weighted_mean(distances,weights)\n    metrics = {"pixel":pixel,"gradient":(gx+gy)*.5,"highpass":highpass,\n               "temporal":temporal,"detail_temporal":hp_temporal,"lpips":perceptual,\n               "mse":weighted_mean(error.square(),mask)}\n    total = pixel+.3*metrics["gradient"]+.15*highpass+.2*temporal+.1*hp_temporal+lpips_weight*perceptual\n    return total,metrics\n\n\nclass Patches(torch.utils.data.Dataset):\n    def __init__(self, root, manifest, split):\n        self.root = Path(root)\n        self.items = [(g["file"],i) for g in manifest["groups"] if g["split"]==split\n                      for i in range(len(g["records"]))]\n        self.cached_name = None\n        self.cached = None\n        self.augment = split == "train"\n        if not self.items:\n            raise ValueError("学習・検証素材が空です。")\n\n    def __len__(self):\n        return len(self.items)\n\n    def __getitem__(self, index):\n        name, offset = self.items[index]\n        if name != self.cached_name:\n            with np.load(self.root/name,allow_pickle=False) as archive:\n                self.cached = {key:archive[key] for key in ("restoration","mosaic","target","mask")}\n            self.cached_name = name\n        arrays = [self.cached[k][offset] for k in ("restoration","mosaic","target","mask")]\n        if self.augment and random.random()<.5:\n            arrays = [a[:,:,::-1].copy() for a in arrays]\n        result = []\n        for i,array in enumerate(arrays):\n            if i==3:\n                array = array[:,None]\n            else:\n                array = array.transpose(0,3,1,2)\n            result.append(torch.from_numpy(np.ascontiguousarray(array)).float()/255)\n        # return temporal restoration/mosaic, 3 target frames, 7 mask frames\n        return tuple(result)\n\n\ndef restore_patches(config, manifest):\n    """Copy the prepared shards to the runtime\'s local disk (fast random reads while training).\n    Each shard is read from Drive once: copied, then hashed locally (a mismatch stops before use).\n    Several shards copy at a time; Drive is slow one file at a time."""\n    from concurrent.futures import ThreadPoolExecutor\n    root = Path(config["work_root"])/"patches"\n    root.mkdir(parents=True,exist_ok=True)\n    total = sum(Path(g.get("root", config["prepared_root"]), g["file"]).stat().st_size for g in manifest["groups"]\n                if not (root/g["file"]).is_file())\n    if shutil.disk_usage(root).free < total+2*2**30:\n        raise RuntimeError("学習パッチを配置するSSD容量が不足です。")\n\n    def place(g):\n        dest = root/g["file"]\n        if dest.is_file() and sha256(dest)==g["sha256"]:\n            return\n        source = Path(g.get("root", config["prepared_root"]))/g["file"]   # a merged manifest may keep shards elsewhere\n        temporary = dest.with_suffix(".partial")\n        shutil.copyfile(source,temporary)\n        if sha256(temporary)!=g["sha256"]:\n            temporary.unlink()\n            raise ValueError(f"準備済み素材のハッシュが不一致です: {g[\'file\']}")\n        os.replace(temporary,dest)\n    groups = manifest["groups"]\n    with ThreadPoolExecutor(int(config.get("copy_workers", 8))) as pool:\n        for i,_ in enumerate(pool.map(place, groups)):\n            if (i+1)%50==0 or i+1==len(groups):\n                print(f"学習パッチ配置: {i+1}/{len(groups)}",flush=True)\n    return root\n\n\nclass ShardBatches:\n    """Shuffle shards and examples, avoiding decompression per random sample.\n\n    A batch never crosses a shard boundary. This keeps the one-shard cache\n    useful, while changing both shard and within-shard order each epoch.\n    """\n    def __init__(self, dataset, batch_size):\n        self.batch_size = batch_size\n        self.groups = {}\n        for index,(name,_) in enumerate(dataset.items):\n            self.groups.setdefault(name,[]).append(index)\n\n    def __iter__(self):\n        groups = list(self.groups.values())\n        random.shuffle(groups)\n        for group in groups:\n            indices = list(group)\n            random.shuffle(indices)\n            for first in range(0,len(indices),self.batch_size):\n                yield indices[first:first+self.batch_size]\n\n    def __len__(self):\n        return sum(math.ceil(len(g)/self.batch_size) for g in self.groups.values())\n\n\ndef preview(path, examples):\n    if not examples:\n        return\n    side = examples[0][0].shape[-1]\n    picture = Image.new("RGB",(side*5,side*len(examples)))\n    for row,images in enumerate(examples):\n        for col,item in enumerate(images):\n            item = np.rint(item.detach().cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)\n            picture.paste(Image.fromarray(item),(col*side,row*side))\n    picture.save(path)\n\n\ndef evaluate(model, dataset, device, max_samples=64, preview_path=None, lpips_model=None, lpips_weight=.05):\n    model.eval()\n    sums = None\n    examples = []\n    # Even temporal coverage, fixed deterministic indices across checkpoints.\n    selected = np.linspace(0,len(dataset)-1,min(max_samples,len(dataset))).round().astype(int)\n    with torch.inference_mode():\n        for i,index in enumerate(selected):\n            restored,mosaic,target,mask = [a.unsqueeze(0).to(device) for a in dataset[int(index)]]\n            prediction = sequence_prediction(model,restored,mosaic,mask)\n            _,metrics = loss_terms(prediction,target,mask[:,2:5],lpips_model,lpips_weight)\n            numbers = {k:float(v) for k,v in metrics.items()}\n            if not all(math.isfinite(v) for v in numbers.values()):\n                raise RuntimeError("検証結果が非有限値です。")\n            if sums is None:\n                sums = {k:0. for k in numbers}\n            for key,value in numbers.items():\n                sums[key] += value\n            if i<6:\n                examples.append([mosaic[0,3],restored[0,3],prediction[0,1],target[0,1],mask[0,3].expand(3,-1,-1)])\n    report = {k:v/len(selected) for k,v in sums.items()}\n    report.update(psnr=-10*math.log10(max(report["mse"],1e-12)),samples=len(selected),\n                  lpips_evaluated=lpips_model is not None)\n    if preview_path:\n        preview(preview_path,examples)\n    return report\n\n\ndef score(report, lpips_weight):\n    return (report["mse"]+.03*report["gradient"]+.03*report["highpass"]+\n            .02*report["temporal"]+.02*report["detail_temporal"]+lpips_weight*report["lpips"])\n\n\ndef quality_guard(report, reference):\n    return (report["psnr"]>=reference["psnr"]-.2 and\n            all(report[k]<=reference[k]*1.05+1e-6 for k in ("temporal","detail_temporal")))\n\n\ndef save_checkpoint(path, model, optimizer, scaler, step, contract, reference, best_score, stale):\n    if any(not torch.isfinite(p).all() for p in model.parameters()):\n        raise RuntimeError("非有限の重みは保存しません。直前のチェックポイントは保持します。")\n    temporary = Path(path).with_suffix(".partial")\n    torch.save({"version":VERSION,"base_architecture":model.backbone.architecture,\n                "model":model.state_dict(),"optimizer":optimizer.state_dict(),\n                "scaler":scaler.state_dict(),"step":step,"contract":contract,\n                "reference":reference,"best_score":best_score,"stale":stale,\n                "torch_rng":torch.get_rng_state(),\n                "python_rng":random.getstate(),"app_integrated":False,\n                "input_contract":"restored+separate_mosaic+mask,5frames,same-size-output"},temporary)\n    os.replace(temporary,path)\n\n\ndef train(config):\n    _,prepared,save,_ = validate_roots(config)\n    device = torch.device(config.get("device","cuda"))\n    if device.type=="cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPUへ接続してください。T4/A100などに対応します。")\n    manifest = json.loads((prepared/"manifest.json").read_text())\n    if manifest.get("version")!=VERSION or not manifest.get("complete"):\n        raise ValueError("素材準備が完了していません。")\n    max_steps = int(config.get("max_steps",1000))\n    minutes = float(config.get("max_minutes",120))\n    interval = int(config.get("eval_interval",250))\n    lpips_weight = float(config.get("lpips_weight",.05))\n    batch_size = int(config.get("batch_size",4))\n    lr = float(config.get("lr",2e-5))\n    if (not 0<max_steps<=100000 or not math.isfinite(minutes) or minutes<=0\n            or not 1<=interval<=10000 or not 0<=lpips_weight<=.2\n            or not 1<=batch_size<=64 or not 0<lr<=1e-3):\n        raise ValueError("学習上限・損失・バッチ設定が不正です。")\n    contract = {"dataset":sha256(prepared/"manifest.json"),\n                "initial":sha256(config["init_checkpoint"]),"lr":lr,\n                "batch_size":batch_size,"lpips_weight":lpips_weight,\n                "eval_samples":int(config.get("eval_samples",64)),"seed":int(config.get("seed",20261005))}\n    if contract["eval_samples"]<1:\n        raise ValueError("検証件数は1以上です。")\n    torch.manual_seed(contract["seed"])\n    random.seed(contract["seed"])\n    np.random.seed(contract["seed"])\n    save.mkdir(parents=True,exist_ok=True)\n    latest = save/"latest.pt"\n    donor,_ = base.load_checkpoint(config["init_checkpoint"])\n    model = ConditionalROI(donor).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=0)\n    # BF16 on supported CUDA hardware, FP32 otherwise (T4). Avoid the previous\n    # FP16 nonfinite-gradient failures; no AMP loss scale guesswork on T4.\n    amp_enabled = device.type=="cuda" and torch.cuda.is_bf16_supported()\n    scaler = torch.amp.GradScaler("cuda",enabled=False)\n    step,stale = 0,0\n    state = None\n    if latest.exists():\n        if not (save/"best.pt").is_file():\n            raise ValueError("best.ptがありません。再開を中止します。")\n        state = torch.load(latest,map_location="cpu",weights_only=True)\n        if state["version"]!=VERSION or state["contract"]!=contract:\n            raise ValueError("学習条件・素材・初期モデルが変わりました。新しいSAVE_ROOTにしてください。")\n        model.load_state_dict(state["model"],strict=True)\n        optimizer.load_state_dict(state["optimizer"])\n        for values in optimizer.state.values():\n            for key,tensor in values.items():\n                if isinstance(tensor,torch.Tensor) and key!="step":\n                    values[key]=tensor.to(device)\n        step,stale = int(state["step"]),int(state["stale"])\n        torch.set_rng_state(state["torch_rng"])\n        random.setstate(state["python_rng"])\n    elif any(save.iterdir()):\n        raise ValueError("学習保存先が空ではありません。既存実験には書き込みません。")\n    if any(not torch.isfinite(p).all() for p in model.parameters()):\n        raise ValueError("重みに非有限値があります。")\n    root = restore_patches(config,manifest)\n    training,validation = Patches(root,manifest,"train"),Patches(root,manifest,"val")\n    lpips_model = None\n    if lpips_weight:\n        print("LPIPSの評価ネットワークを読み込みます（初回は公開重みを取得）。",flush=True)\n        import lpips\n        lpips_model = lpips.LPIPS(net="alex",spatial=True).to(device).eval()\n        for p in lpips_model.parameters():\n            p.requires_grad_(False)\n    reference = state["reference"] if state else evaluate(model,validation,device,contract["eval_samples"],save/"comparison-initial.png",lpips_model,lpips_weight)\n    best_score = state["best_score"] if state else score(reference,lpips_weight)\n    if not state:\n        base.atomic_json(save/"reference.json",reference)\n        base.atomic_json(save/"evaluation-best.json",{**reference,"step":0,"selected":False})\n        save_checkpoint(save/"best.pt",model,optimizer,scaler,step,contract,reference,best_score,stale)\n        shutil.copyfile(save/"comparison-initial.png",save/"comparison-best.png")\n        save_checkpoint(latest,model,optimizer,scaler,step,contract,reference,best_score,stale)\n    print(f"学習開始: step={step}/{max_steps}、{device}、{\'BF16\' if amp_enabled else \'FP32\'}、初期PSNR={reference[\'psnr\']:.3f}",flush=True)\n    loader = torch.utils.data.DataLoader(training,batch_sampler=ShardBatches(training,batch_size),num_workers=0)\n    started = time.monotonic()\n    def validate_step():\n        nonlocal stale,best_score\n        report = evaluate(model,validation,device,contract["eval_samples"],save/"comparison-latest.png",lpips_model,lpips_weight)\n        candidate = score(report,lpips_weight)\n        improved = quality_guard(report,reference) and candidate < best_score*(1-1e-4)\n        if improved:\n            best_score,stale = candidate,0\n            save_checkpoint(save/"best.pt",model,optimizer,scaler,step,contract,reference,best_score,stale)\n            shutil.copyfile(save/"comparison-latest.png",save/"comparison-best.png")\n            base.atomic_json(save/"evaluation-best.json",{**report,"step":step})\n        else:\n            stale += 1\n        base.atomic_json(save/"evaluation-latest.json",{**report,"step":step,"stale":stale,"selected":improved})\n        with (save/"training.jsonl").open("a") as log:\n            log.write(json.dumps({**report,"step":step,"stale":stale,"selected":improved})+"\\n")\n        print(f"検証step={step}: マスク内PSNR={report[\'psnr\']:.3f}, highpass={report[\'highpass\']:.6f}, detail-temporal={report[\'detail_temporal\']:.6f}, stale={stale}, best更新={improved}",flush=True)\n        save_checkpoint(latest,model,optimizer,scaler,step,contract,reference,best_score,stale)\n        model.train()\n    last_evaluated = step\n    try:\n        model.train()\n        while step<max_steps and time.monotonic()-started<minutes*60:\n            for restored,mosaic,target,mask in loader:\n                if step>=max_steps or time.monotonic()-started>=minutes*60:\n                    break\n                restored,mosaic,target,mask = [x.to(device) for x in (restored,mosaic,target,mask)]\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=device.type,dtype=torch.bfloat16,enabled=amp_enabled):\n                    prediction = sequence_prediction(model,restored,mosaic,mask)\n                # All detail/perceptual losses calculated FP32.\n                loss,_ = loss_terms(prediction,target,mask[:,2:5],lpips_model,lpips_weight)\n                if not torch.isfinite(loss):\n                    raise RuntimeError("損失が非有限値です。更新せず停止します。")\n                loss.backward()\n                norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=True)\n                if not torch.isfinite(norm):\n                    raise RuntimeError("勾配が非有限値です。更新せず停止します。")\n                optimizer.step()\n                step += 1\n                if step%100==0:\n                    print(f"step {step}/{max_steps}, loss={float(loss.detach()):.5f}, {(time.monotonic()-started)/60:.1f}分",flush=True)\n                if step%interval==0:\n                    validate_step()\n                    last_evaluated = step\n                    if config.get("early_stopping",True) and stale>=int(config.get("patience",8)):\n                        print("検証指標の改善が止まったため終了。最良best.ptは保持します。",flush=True)\n                        return {"step":step,"stale":stale,"best_score":best_score}\n        if step!=last_evaluated:\n            validate_step()\n    finally:\n        save_checkpoint(latest,model,optimizer,scaler,step,contract,reference,best_score,stale)\n    return {"step":step,"stale":stale,"best_score":best_score}\n\n\ndef test_best(config):\n    """Held-out test only on explicit user action, not checkpoint selection."""\n    _,prepared,save,_ = validate_roots(config)\n    state = torch.load(save/"best.pt",map_location="cpu",weights_only=True)\n    manifest = json.loads((prepared/"manifest.json").read_text())\n    if state["version"]!=VERSION or state["contract"]["dataset"]!=sha256(prepared/"manifest.json"):\n        raise ValueError("best.ptと評価素材が不一致です。")\n    model = ConditionalROI(base.TinyROIEnhancer(**base.validated_architecture(state["base_architecture"])))\n    model.load_state_dict(state["model"],strict=True)\n    device = torch.device(config.get("device","cuda"))\n    model.to(device)\n    root = restore_patches(config,manifest)\n    report = evaluate(model,Patches(root,manifest,"test"),device,int(config.get("eval_samples",64)),save/"comparison-test.png")\n    report.update(step=state["step"],same_production_only=True,app_integrated=False)\n    base.atomic_json(save/"evaluation-test.json",report)\n    return report\n', 'roi_mask_gan_runtime.py': '"""Masked ROI detail with a perceptual + adversarial objective (opt-in).\n\nWhy a new stage: roi-mask-detail-v1 learned only pixel-aligned losses (L1,\ngradient, aligned high-pass, LPIPS x0.05) and selected best by MSE, and its\nresidual leaves the x2 head through an area downsample. Detail that the input\ndoes not determine averages out to zero under such losses, so its outputs\ngain colour/shading accuracy but no detail.\n\nHere the v1 best.pt is inherited (trainable, low lr) and a same-resolution\ndetail branch is added with a zero output layer, so step 0 equals v1 exactly.\nLow frequencies are tied to the uncensored target pixel-wise; high\nfrequencies are learned from LPIPS, a conditional PatchGAN whose REAL samples\nare the uncensored target inside the mask, and a local detail-amplitude\nmatch. Flicker is held by the teacher-relative temporal difference of the\nhigh-pass error. Best = perceptual score; PSNR, low-frequency error and\nflicker are guards. Inputs, patches and the v1 checkpoint are read-only.\n\nv2 (after the first run: detail 0.67 -> 0.77 of the target\'s, and pushing\nit to 1.0 also moved colour and shading, PSNR -1.7 dB, so the guard kept the\nweak step): the detail branch can only add high frequencies (its output is\nhigh-passed), so texture can no longer shift colour; LPIPS uses VGG; GAN 0.1;\nPSNR is only a -3 dB safety bound, low-frequency error and flicker decide.\nv1 checkpoints of this stage (no "highpass" in their architecture) still load.\n\nv3 (v2 kept its best at step 2000, detail 0.72: every later step raised the\nlow-frequency error past the guard although the branch was high-passed, and\nthe wide comparison showed no texture inside the mask): the v1 stage is\nfrozen, so the coarse image is exactly v1; the branch is high-passed with\nthe same 9 px box the low-frequency error uses, and its low band is anchored\nto the frozen coarse image instead of the target. LPIPS and the\ndiscriminator compare texture only: the reference is the prediction plus the\n9 px high-pass of (target - prediction) inside the mask, so colour and\nshading are no cue the branch could chase. With colour held by the frozen\nstage the GAN can be stronger (0.3). Every evaluation also keeps its weights\nand comparison images under SAVE_ROOT/snapshots (config "snapshots").\n\nv4 (v3 ran to 20000: detail amount reached the target\'s, but its correlation\nwith the target fell 0.20 -> 0.13; the restoration itself sits at 0.19, and a\nperfect 256 px restoration would keep 0.82, so the missing texture is not in\nthe input at all). The goal changes from matching the target to generating\nplausible, stable texture. Every loss that rewards the target\'s exact texture\npositions is removed (pixel, LPIPS, target-relative flicker). The branch\ninstead learns from: a texture-only discriminator that sees the 9 px high\nband and the mask, so its REAL side needs no pairing (the patch targets plus\nan optional bank of dense uncensored crops); a position-free texture\ndistance (VGG Gram matrices against the target\'s texture on the\nprediction\'s own low band); the local detail amount; and the change of the\nhigh band along the motion (RAFT flow between the frozen coarse frames,\nforward-backward checked), measured on the output itself. Selection uses the\ntexture distance and the amount; colour and flicker relative to the\ntarget\'s own flicker are guards. Correlation is still logged, never used.\n\nv5 (v4 put texture on, but it looked blurred: outlines stayed as soft as v1,\nbecause nothing aligned sharpened them any more, and the added texture was\n4-8 px blobs while the target\'s is 1-2 px lines and grain): the branch, the\ntexture reference, the discriminator and the flicker use the 5 px high band,\nso blobs cannot be produced; one aligned term returns for structure only, an\nL1 on deep VGG features (relu4_3, 1/8 resolution) against the target\'s\ntexture on the prediction\'s own low band. Deep features follow edges and\nshapes but barely the exact place of fine texture, so outlines sharpen\nwithout averaging the texture away. Selection adds this structure distance.\n\nv6 (order swapped inside the model, as the app uses it): restored -> texture\n-> SwiftVR-teacher ROI distill (frozen, x2) -> mask composite at x2. v1 is\nnot used. The texture branch works per frame on the restoration grid (all\nfive frames, so the distill gets a textured window); the distill then\nsharpens edges and texture together, and the mask composite is the last\nstep (outside the mask: the bilinear x2 of the restoration). Training runs\non the app\'s grid: the native 128 px patches are shrunk by a random grid\nscale (0.4-1.0, the app shrinks crops to 256 px), the x2 result is resized\nback to native size and judged against the uncensored target with the v5\nobjective. The texture branch uses a 3 px high band at grid scale.\n\nv7 (v6 in the app looked smoother than v4: the frozen distill, trained on\nsmooth restorations, flattened the texture it received into a faint mesh,\nand edges were no sharper than the distill alone; v4, whose texture comes\nlast, showed visible skin texture): restored -> ROI distill (frozen, x2) ->\ntexture at the x2 resolution -> mask composite. Nothing runs after the\ntexture, so it is not smoothed away. The texture branch sees the distilled\ncentre, the x2 mosaic and the x2 mask and may use the 9 px band (as v4,\nwhose texture size looked right in the app); D, the style reference and\nthe flicker use the 9 px band of the native-size result as in v4. Texture\nmust not overwrite the edges the distill has just sharpened: it is faded out\nwhere the distilled image has a strong edge (gradient of the blurred\nluminance; full texture below none, none above edge_guard per pixel).\n\nv7b (v7 at large size: outlines kept, but the added texture was mostly a\nfine square lattice; feeding v7 the restoration in place of the mosaic\nremoved the lattice together with most of the texture, so the texture came\nfrom the mosaic\'s block edges): as v7, but the texture branch sees only the\ndistilled centre and the x2 mask. The mosaic carries no fine detail of the\ntarget (high-band corr 0.02), only its block edges. The model keeps the\nthree-input call (restored, mosaic, mask) so the app contract is unchanged;\nthe mosaic is ignored.\n\nv8 (v7b at large size: no lattice, outlines kept, but the texture was faint;\nv4, whose texture was added on the 256 grid with a 9 px band, was the one\nwhose texture showed in the app): the v6 order again, restored -> texture\n(each of the five frames, on the 256 grid) -> ROI distill (frozen, x2) ->\nmask composite, but without the mosaic (v6\'s mesh was most likely the\nmosaic lattice flattened by the distill), with the 9 px band of v4 on the\ngrid (v6 used 3 px, fine enough for the distill to average away), and with\nthe edge guard on the restored frames (no texture on their outlines).\nTraining and selection as v7b.\n\nv9 (v8 at large size: edges kept and more visible texture than v7b, but the\nadded texture was a regular fine weave; the texture branch had no source of\nrandomness, so the realism pressure settled on a repeating pattern): v8 plus\na texture seed, four channels of white noise fed to the texture branch. In\ntraining each sample draws its own noise, shared by all its frames and\nwindows (the seed sits still in crop coordinates, as in the app). For\nevaluation and in the app the model uses one fixed 256x256 noise stored in\nthe checkpoint (seed 20261006): anchored to the ROI crop, the same for every\nframe, so it does not flicker by itself; the flicker term still suppresses\ntexture that does not follow the surface. The app contract is unchanged.\n\nv10 / P1 / P2 / v10b (a seed with a logic): the laws of the texture the\ndistill output lacks were measured on the train split (roi_texture_laws.py;\nDrive mioh-roi-texture-laws-v1) and the seed is built from them after the\ndistill at x2 (the finest gap, periods <= 4 px, cannot exist on the 256 grid):\nthe missing power spectrum, orientation steered along contours (calibrated to\nthe measured alignment), local strength = a + b * the distill\'s local detail,\nluminance only. P1 adds that seed directly (laws v1, interior including the\nzone next to outlines); v10 is P1 plus a zero-initialised learned correction.\nP1 put dotted grain along outlines: within ~6 px of an outline the "missing"\ndetail is outline blur and misplacement, not texture (distance profile). P2\nuses laws measured only farther than 6 px from outlines and fades the seed out\nin that zone; v10b is P2 plus the learned correction (step 0 equals P2).\n\nv10c (v10b best in numbers but subtler than v10 by eye): the texture goal is\n1.3 x the truth\'s. Everything that refers to the truth\'s texture amount is\nscaled together, otherwise the rest pulls it back to 1x: the seed strength,\nthe local-amplitude target, the style reference (truth\'s high band x gain),\nthe discriminator\'s real samples (high band x gain), the detail-ratio term of\nthe selection score (distance from gain, not 1) and the flicker allowance.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nimport roi_enhancer_distill_runtime as base\nimport roi_mask_detail_runtime as paired\n\nVERSION = "roi-mask-gan-v19"\nSEED_DIRECT_VERSION = "roi-mask-gan-p1"   # the law-built seed alone, no learned weights\nVERSIONS = ("roi-mask-gan-v1", "roi-mask-gan-v2", "roi-mask-gan-v3", "roi-mask-gan-v4", "roi-mask-gan-v5",\n            "roi-mask-gan-v6", "roi-mask-gan-v7", "roi-mask-gan-v7b", "roi-mask-gan-v8", "roi-mask-gan-v9",\n            SEED_DIRECT_VERSION, "roi-mask-gan-p2", "roi-mask-gan-v10", "roi-mask-gan-v10b", "roi-mask-gan-v10c",\n            "roi-mask-gan-v11", "roi-mask-gan-v12", "roi-mask-gan-v12b", "roi-mask-gan-v13", "roi-mask-gan-v13b", "roi-mask-gan-v13c", "roi-mask-gan-v13d", "roi-mask-gan-v13e", "roi-mask-gan-v13f", "roi-mask-gan-v13g", "roi-mask-gan-v13h", "roi-mask-gan-v13i", "roi-mask-gan-v13j", "roi-mask-gan-v14", "roi-mask-gan-v14b", "roi-mask-gan-v14c", "roi-mask-gan-v14d", "roi-mask-gan-v14e", "roi-mask-gan-v14f", "roi-mask-gan-v16", "roi-mask-gan-v17", "roi-mask-gan-v18", VERSION)   # checkpoints this module can load\nBANK_VERSION = "roi-texture-bank-v1"\nDETAIL_ARCH = {"width": 32, "blocks": 6, "highpass": True, "band": 5}   # v4/v5 GanROI branch\nTEXTURE_DISTILL_ARCH = {"width": 32, "blocks": 6, "band": 3, "order": "texture-then-distill", "scale": 2}  # v6\nDISTILL_TEXTURE_ARCH = {"width": 32, "blocks": 6, "band": 9, "edge_guard": .06,\n                        "order": "distill-then-texture", "scale": 2}  # v7\nDISTILL_TEXTURE_B_ARCH = {**DISTILL_TEXTURE_ARCH, "mosaic_input": False}  # v7b\nTEXTURE_DISTILL_B_ARCH = {"width": 32, "blocks": 6, "band": 9, "edge_guard": .06,\n                          "order": "texture-then-distill", "scale": 2, "mosaic_input": False}  # v8\nTEXTURE_DISTILL_C_ARCH = {**TEXTURE_DISTILL_B_ARCH, "seed_channels": 4}  # v9\nSEED_SIDE, SEED_VALUE = 256, 20261006\nSEED_DIRECT_ARCH = {"width": 32, "blocks": 6, "band": 9, "edge_guard": .06, "order": "distill-then-seed",\n                    "scale": 2, "learned": False}                                            # P1\nTEXTURE_SEED_ARCH = {**SEED_DIRECT_ARCH, "learned": True}                                    # v10\nSEED_DIRECT_P2_ARCH = {**SEED_DIRECT_ARCH, "laws": "roi-texture-laws-v2"}                     # P2\nTEXTURE_SEED_P2_ARCH = {**SEED_DIRECT_P2_ARCH, "learned": True}                              # v10b\nTEXTURE_GAIN = 1.3\nTEXTURE_SEED_P2_GAIN_ARCH = {**TEXTURE_SEED_P2_ARCH, "texture_gain": TEXTURE_GAIN}             # v10c\nSEEDED_TEXTURE_ARCH = {"width": 48, "blocks": 6, "band": 9, "edge_guard": .06, "order": "distill-seeded-texture",\n                       "scale": 2, "types": 24, "type_dim": 32, "seed_channels": 16}               # v11\n# v12: the v11 structure (seed only inside, bank texture types, 16 seed shapes) at\n# the v10 size. The seed is 10 px stamps of 16 shapes scattered at random (seed_stamps),\n# pasted only inside the mask and anchored to it: shifted\n# by the mask centroid (mean of the window\'s five masks, whole output pixels) and\n# multiplied by the mask, so it moves with the mask with no optical flow.\nSEEDED_TEXTURE_FIXED_ARCH = {**SEEDED_TEXTURE_ARCH, "width": 32, "seed_motion": "fixed"}         # (not trained)\nSEEDED_TEXTURE_MASK_ARCH = {**SEEDED_TEXTURE_ARCH, "width": 32, "seed_motion": "mask",\n                            "seed_shapes": "stamps10"}                                           # v12\n# v12b: v12 measured on the truth: the missing detail is mostly 9-33 px (pore groups,\n# fine wrinkles, skin relief shading), which a 9 px band cannot touch. Texture band\n# 33 px at the x2 output (the losses, anchor and discriminator follow it); seed stamps\n# 10-30 px. Shape/shading/colour coarser than 33 px stay the distill\'s.\nSEEDED_TEXTURE_WIDE_ARCH = {**SEEDED_TEXTURE_MASK_ARCH, "band": 33, "seed_shapes": "stamps10-30"}   # v12b\n# v13 (hybrid): v12b\'s texture (33 px band, mask-anchored 10-30 px seed, bank types, outline\n# protection) drawn by a generator that sees the whole ROI. An encoder down to 32x32 (with a\n# masked global summary) knows the part, the folds and the light; the decoder brings that\n# back to 512 px so details can follow the shape where the shape predicts them (inference),\n# while the seed and the bank-realism losses fill what cannot be predicted (generation). The\n# coarse band stays the distill\'s (consistency). Trained on whole-ROI crops, not 128 px patches.\nCONTEXT_TEXTURE_ARCH = {"widths": [24, 32, 48, 64, 96], "mid_blocks": 4, "band": 33, "edge_guard": .06,\n                        "order": "distill-context-texture", "scale": 2, "types": 24, "type_dim": 32,\n                        "seed_channels": 16, "seed_motion": "mask", "seed_shapes": "stamps10-30",\n                        "outline_protect": 6, "outline_fade": 10}                                   # v13\n# v13b: v13 kept the fine-band correlation below the distill\'s (.44-.46 vs .474) and the\n# missing content in the truth is folds/wrinkles/wet highlights of 10-40 px. The texture\n# band is widened to 65 px (shape/shading coarser than that stay the distill\'s), the\n# outline protection narrowed to 2 px + 3 px fade (every outline incl. inner folds keeps\n# exactly; 57% -> 75% of the interior becomes drawable), and the training leans on the\n# paired fine-shape term (fine_weight 1.0, GAN 0.15 in the notebook). Detail goal 1.5x the\n# truth\'s (TEXTURE_GAIN), and best selection rewards moving away from the distill.\nCONTEXT_TEXTURE_B_ARCH = {**CONTEXT_TEXTURE_ARCH, "band": 65, "outline_protect": 2, "outline_fade": 3}   # v13b\n# v13c: v13b\'s 1.5x goal applied to the whole 65 px band also raised the 17-65 px shading\n# relief (a wide unsharp mask: contrast went up). The goal is now 1.5x only for the fine\n# band (<= 17 px: pores, fine wrinkles); 17-65 px keeps the truth\'s amount (1x).\nCONTEXT_TEXTURE_C_ARCH = {**CONTEXT_TEXTURE_B_ARCH, "fine_gain_band": 17}                             # v13c\n# v13d: v13b\'s policy (whole-ROI generator, 65 px band, fine detail 1.5x, mask seed, reward for\n# moving away from the distill) with its two faults fixed. Doubled / receding outlines came\n# from 17-65 px relief drawn right beside outlines (a shifted second step): that relief now\n# keeps 8 px off every outline (16 px fade) while fine texture (<= 17 px) keeps the 2 + 3 px\n# guard. Too much contrast came from 1.5x relief: relief goal 1.25x (fine 1.5x), overshoot\n# of the local amount costs double, and a ghost-edge term penalises new smoothed-luminance\n# gradient within 2-14 px of outlines that the distill did not have.\n# The head output is bounded (0.25*tanh) so it cannot carry a large flat offset: v13b\'s\n# offset made the fp16 box averages of the band lose the detail (Core ML ~3x texture).\nCONTEXT_TEXTURE_D_ARCH = {**CONTEXT_TEXTURE_B_ARCH, "fine_gain_band": 17, "mid_gain": 1.25,\n                          "mid_outline_protect": 8, "mid_outline_fade": 16, "head_bound": .25}      # v13d\n# v13e: v13d drew almost nothing (change <= .9 levels, relief 1.02x): keeping the relief\n# 8 px off outlines left little room, because the visible relief sits on folds/edges. The\n# relief is no longer drawn freely (a shifted second step = doubled outline): the model\n# outputs a smooth gain map (0.5-1.5x, from the 64 px grid) that scales the distill\'s own\n# 17-65 px relief in place, so no new edge can appear; within 2 px (+3 px fade) of every\n# outline the gain is 1. Fine texture (<= 17 px) is generated as before (1.5x goal).\n# The added detail also fades in from the mask border over ~16 px (v13b showed a visible seam\n# where the texture/relief stopped at the mask edge).\nCONTEXT_TEXTURE_E_ARCH = {**CONTEXT_TEXTURE_B_ARCH, "fine_gain_band": 17, "mid_gain": 1.25, "head_bound": .25,\n                          "relief_mode": "gain", "relief_gain_range": .5, "mask_fade": 16}        # v13e\n# v13f: v13e\'s generated fine layer was a random squiggle (orientation coherence .48, along the\n# shape .45, ~1 level) while the truth\'s <= 17 px detail is wrinkle lines that run along the shape\n# (coherence .88, along the distill\'s shape .76). The flow direction of the shape is measured on\n# the distill (structure tensor of its shading, 17 px window) and given to the generator; the\n# fine head (luminance only) is smeared along that direction by steerable line kernels (8\n# orientations, 15 px), so what it draws becomes lines parallel to the shape where the shape has\n# a direction (blended back to the raw head where it has none). Training adds an orientation\n# term: the added detail\'s local direction against the truth\'s fine band (weighted by the truth\'s\n# coherence, outlines excluded); the best checkpoint also rewards it.\nCONTEXT_TEXTURE_F_ARCH = {**CONTEXT_TEXTURE_E_ARCH, "line_mode": "steer", "line_orientations": 8,\n                          "line_length": 15, "line_sharpness": 4., "orient_window": 17}           # v13f\n# v13g: v13f\'s fine layer ran along the shape (alignment .74, truth .755) but was too fine and\n# too weak: 2 px hatching at a 7 px period and 0.5 levels, against the truth\'s wrinkles at a\n# ~17 px period and several levels. The lines are drawn at wrinkle scale: the head is steered on\n# the half-resolution grid (12 orientations, 17 px there = ~34 px long, sigma 1 px across = ~5 px\n# wide strokes) and brought back up. Training adds a fine-band amount term: inside the mask,\n# outlines excluded, the local amount of the <= 17 px band against the goal\'s (1.5x the truth\'s);\n# the best checkpoint is also judged on that ratio.\nCONTEXT_TEXTURE_G_ARCH = {**CONTEXT_TEXTURE_F_ARCH, "line_orientations": 12, "line_length": 17, "line_width": 1.,\n                          "line_grid": 2}                                                         # v13g\n# v13h: v13g\'s wrinkle-scale lines doubled outlines (ghost .27-.32 vs v13f\'s .05-.09; clearly\n# visible in stills): beside an outline a line "along the shape" is a stroke parallel to it, i.e.\n# a second contour. The steered (line) part now keeps 10 px off every outline (incl. inner folds)\n# and fades in over 16 px; there only the direction-free share of the fine texture remains. The\n# relief gain and the direction-free fine texture keep the 2 + 3 px guard. The ghost-edge term\n# (new smoothed-luminance steps 2-14 px beside outlines) is back in the loss, in the best score,\n# and as a guard (a checkpoint above GHOST_LIMIT is never selected).\nCONTEXT_TEXTURE_H_ARCH = {**CONTEXT_TEXTURE_G_ARCH, "line_outline_protect": 10, "line_outline_fade": 16}   # v13h\nGHOST_LIMIT = .10\n# v13i: v13h2 still doubled outlines (ghost .15-.31) and made fingers look swollen; split on the\n# triplet frames, 3.01 of its 3.02 levels of change were the relief gain (v13e step 8000: 1.58).\n# Scaling the 17-65 px relief on a strong outline is an unsharp mask: a dark band inside, a bright\n# rim outside. The gain is now 1 within 12 px of strong outlines (fading back over 12 px), so only\n# soft creases/folds (below the outline threshold) get deepened, and its range is 0.8-1.3x\n# (was 0.5-1.5x). Lines, fine-band amount goal and the ghost guard stay as v13h (ghost loss off).\nCONTEXT_TEXTURE_I_ARCH = {**CONTEXT_TEXTURE_H_ARCH, "relief_gain_low": .2, "relief_gain_high": .3,\n                          "relief_outline_protect": 12, "relief_outline_fade": 12}                  # v13i\n# v13j: the line steering (v13f-v13i) kept adding strokes parallel to every soft edge: doubled\n# outlines (v13g), halos with the relief gain (v13h2) and hatching along floor tiles and leg\n# edges (v13i). Back to v13e\'s architecture and losses, which the user liked, trained on the\n# expanded private real/synthetic materials (12.6x train).\n# v14 (generate, not recover): the paired terms (fine-shape distance, structure, fine-band\n# correlation) punish detail that is not where the truth has it, so every v13 kept its detail\n# faint. v14 draws a 5-33 px band (texture and moderate detail, no pixel-level grain) freely and\n# judges it for realism only: a stronger GAN (0.4) whose discriminator also sees the place (the\n# distill image, or a heavy blur of a bank crop), so the texture it accepts depends on the part.\n# No relief gain (halos) and no line steering (hatching); the distill\'s shading is untouched.\n# 32 bank texture types (bank v2: many titles, key-frame crops). Anchor, Gram style, amplitude\n# goal and flicker stay; best selection: style, amount, change, ghost (guard .25).\nCONTEXT_TEXTURE_V14_ARCH = {**CONTEXT_TEXTURE_B_ARCH, "band": 33, "detail_low_cut": 5, "head_bound": .25,\n                            "mask_fade": 16, "types": 32}                                          # v14\n# v14b: v14 copied the 10-30 px seed stamps straight into its 5-33 px band (crosses and bars in the\n# output) and painted skin-like detail on background inside the mask. With a discriminator that\n# sees the place, the texture can come from the input alone (pix2pix-style): the seed is no longer\n# injected (the input stays for the app\'s contract), so nothing can be copied and the texture moves\n# with the content rather than with the mask centroid. The local amount term weighs 2.0 with\n# overshoot costing double, so smooth background stays smooth.\nCONTEXT_TEXTURE_V14B_ARCH = {**CONTEXT_TEXTURE_V14_ARCH, "seed_inject": False}                         # v14b\n# v14c: v14b showed the mask border (the head\'s flat offset inside the mask turned into a ring by\n# the band-pass) and blotchy 17-33 px noise at 1.3x the truth\'s amount. The band-pass is now taken\n# with mask-weighted (normalised) blurs, so a flat offset inside the mask leaves nothing at its\n# border; the band is 5-17 px (pores, grain, fine wrinkles) and the goal is the truth\'s amount (1x).\nCONTEXT_TEXTURE_V14C_ARCH = {**CONTEXT_TEXTURE_V14B_ARCH, "band": 17, "masked_bandpass": True}           # v14c\n# v14d: v14c left a faint dark band 4-12 px inside the mask edge (-0.1..-0.4 levels) and the\n# texture/no-texture boundary showed. The added detail now ramps in over 32 px and its local mean\n# (17 px, inside the mask) is removed after the ramp, so it cannot form a darker/brighter band.\n# The best score no longer subtracts 2x ghost (it grew with any texture near outlines and kept\n# best from ever updating); the ghost guard (.25) stays.\n# Also: v14c drew a thin bright line 3-8 px inside outlines (finger edges, shaft rim) - a second\n# contour that the ghost metric (5 px blur) misses. Texture now keeps 6 px off strong outlines and\n# fades in over the next 8 px (v14c: 2 + 3).\nCONTEXT_TEXTURE_V14D_ARCH = {**CONTEXT_TEXTURE_V14C_ARCH, "mask_fade": 32, "zero_mean_band": 17,\n                             "outline_protect": 6, "outline_fade": 8}                                    # v14d\n# v14e: v14d added clearly visible detail but its relief (volume on folds) was weaker than v13e\'s,\n# and its per-channel head drew thin rainbow streaks along outlines. The head is grey (one\n# detail for R, G and B), and v13e\'s relief gain comes back: the distill\'s own 17-65 px relief\n# scaled in place by a smooth 0.5-1.5x map (no new edge can appear), kept 1 within 2 px (+3 px\n# fade) of outlines as in v13e. Training judges the whole 5-65 px band (discriminator, style,\n# amount) against a goal of the truth with its 17-65 px relief x1.25 (v13e\'s goal); the\n# generated 5-17 px detail stays as v14d (masked band-pass, 6 + 8 px outline guard, zero mean).\nCONTEXT_TEXTURE_V14E_ARCH = {**CONTEXT_TEXTURE_V14D_ARCH, "luma_detail": True, "judge_band": 65,\n                             "fine_gain_band": 17, "mid_gain": 1.25, "relief_mode": "gain",\n                             "relief_gain_range": .5, "relief_outline_protect": 2,\n                             "relief_outline_fade": 3}                                                  # v14e\n# v14f (user, 2026-10-09: "from 1500 on, leave the contrast alone and focus on putting detail on"):\n# continues from v14e step 1500 with everything that decides the relief gain frozen (encoder, middle\n# blocks, type head, the coarsest decoder level and the gain head), so the 17-65 px contrast stays\n# exactly as at 1500. Only the finer decoder levels and the detail head train, judged on the\n# 5-17 px band they draw (as v14d); the relief keeps its 65 px reach (relief_band).\nCONTEXT_TEXTURE_V14F_ARCH = {**{k: v for k, v in CONTEXT_TEXTURE_V14E_ARCH.items() if k != "mid_gain"},\n                             "judge_band": 17, "relief_band": 65, "masked_relief": True}                # v14f\nFROZEN_PREFIXES_V14F = ("enc.", "mid.", "summary.", "typer.", "type_embed.", "dec.0.", "dec_film.0.",\n                        "relief_gain.", "inject.")\n# v16 (2026-10-10): retrain the detail branch. The generated 5-17 px layer of v14e/v15 is weak\n# (~0.4-0.8 levels) and mostly brightens specular highlights; export-time gains (fine_scale,\n# inner_fine_scale x3) only made highlights punchier and sat at the codec-noise level in video.\n# Starts from v15\'s best (v14e step 2500 + temporal fine-tune) with the relief path frozen (the\n# user\'s chosen contrast stays), mask_fade 64 (v14e\'s border band came from the detail):\n# - goal: 1.3x the truth\'s 5-17 px amount (texture_gain), judged on the 5-17 px band\n# - keep: the added layer\'s local amount may not fall below the frozen start\'s (v14f lost detail)\n# - highlights (bright speculars, from the place and the truth) are out of the discriminator\'s\n#   view and the added layer may not exceed the start\'s there (skin grain, not wet glints)\n# - the texture bank weighs 0.7 of the discriminator\'s real samples (skin grain examples)\n# - temporal: the added layer is compared along the RAFT motion (no boiling texture)\nCONTEXT_TEXTURE_V16_ARCH = {**{k: v for k, v in CONTEXT_TEXTURE_V14E_ARCH.items() if k != "mid_gain"},\n                            "judge_band": 17, "relief_band": 65, "mask_fade": 64}                       # v16\n# v17 (2026-10-10): clean restart after finding that the 0.25*tanh head bound introduced in\n# v13d saturated the detail branch. This is a deterministic plain U-Net: no random seed input,\n# no texture-type classifier/codebook, no FiLM conditioning and no dormant compatibility branch.\n# The bottleneck already sees the complete ROI. Keep only output-domain invariants that prevent\n# independently observed artefacts: masked 5-17 px detail, local-mean removal, luma-only output,\n# outline protection, the 64 px mask fade, and in-place scaling of the distill\'s 17-65 px relief.\n_V17_REMOVED = {"head_bound", "mid_gain", "types", "type_dim", "seed_channels", "seed_motion",\n                "seed_shapes", "seed_inject", "seed_blur", "seed_top_level"}\nCONTEXT_TEXTURE_V17_ARCH = {**{k: v for k, v in CONTEXT_TEXTURE_V14E_ARCH.items()\n                              if k not in _V17_REMOVED}, "plain_unet": True, "mask_fade": 64}          # v17\n# v18: v17 still retained v13e\'s learned 17..65 px relief-gain head.  That head changes the\n# strength of an existing step on either side of an outline and is therefore able to create the\n# second, parallel contour that the texture-only stage must never own.  Remove the whole relief\n# path instead of widening another output mask.  The generator now has one job: a zero-mean,\n# luminance-only 5..17 px residual.  A differentiable edge-echo invariant is used during training\n# and best selection; it does not modify inference output.\n_V18_REMOVED = {"relief_mode", "relief_gain_range", "relief_outline_protect",\n                "relief_outline_fade", "relief_band", "judge_band"}\nCONTEXT_TEXTURE_V18_ARCH = {**{k: v for k, v in CONTEXT_TEXTURE_V17_ARCH.items()\n                              if k not in _V18_REMOVED}, "judge_band": 17}                              # v18\n# v19: SwiftVR as the teacher of the texture residual. ROI Distill learned SwiftVR with a pixel\n# loss, which averaged away SwiftVR\'s finest, partly generated grain (on mosaic ROIs SwiftVR adds\n# 1.38 levels at <= 5 px, Distill 0.85, correlation .47). v18\'s structure stays (no relief branch,\n# no double contour, outline guard 6 + 8 px, 64 px mask fade); the drawn band reaches down to the\n# pixel level (no 5 px low cut) and a paired term asks the added <= 17 px luma layer to match\n# SwiftVR\'s added layer deep inside the mask, away from outlines (the restoration\'s wrong shapes\n# are not sharpened). The discriminator and texture statistics keep the grain from averaging out.\nCONTEXT_TEXTURE_V19_ARCH = {k: v for k, v in CONTEXT_TEXTURE_V18_ARCH.items() if k != "detail_low_cut"}  # v19\nV16 = VERSION == "roi-mask-gan-v16"\nV17 = VERSION == "roi-mask-gan-v17"\nV18 = VERSION == "roi-mask-gan-v18"\nV19 = VERSION == "roi-mask-gan-v19"\nNO_RELIEF = V18 or V19\nCLEAN_UNET = V17 or V18 or V19\nHIGHLIGHT_AWARE = V16\nV14 = VERSION in ("roi-mask-gan-v14", "roi-mask-gan-v14b", "roi-mask-gan-v14c", "roi-mask-gan-v14d",\n                  "roi-mask-gan-v14e", "roi-mask-gan-v14f", "roi-mask-gan-v16", "roi-mask-gan-v17",\n                  "roi-mask-gan-v18", "roi-mask-gan-v19")\nCONTEXT_DISCRIMINATOR = V14\nGHOST_LIMIT_V14 = .25\nGHOST_LIMIT_V18 = .08\nTRAIN_ARCH = (CONTEXT_TEXTURE_V19_ARCH if V19 else\n              CONTEXT_TEXTURE_V18_ARCH if V18 else\n              CONTEXT_TEXTURE_V17_ARCH if V17 else\n              CONTEXT_TEXTURE_V16_ARCH if V16 else\n              CONTEXT_TEXTURE_V14F_ARCH if VERSION == "roi-mask-gan-v14f" else\n              CONTEXT_TEXTURE_V14E_ARCH if VERSION == "roi-mask-gan-v14e" else\n              CONTEXT_TEXTURE_V14D_ARCH if VERSION == "roi-mask-gan-v14d" else\n              CONTEXT_TEXTURE_V14C_ARCH if VERSION == "roi-mask-gan-v14c" else\n              CONTEXT_TEXTURE_V14B_ARCH if VERSION == "roi-mask-gan-v14b" else CONTEXT_TEXTURE_V14_ARCH if V14\n              else CONTEXT_TEXTURE_E_ARCH)\nFINE_GAIN_BAND = TRAIN_ARCH.get("fine_gain_band", 0)\nMID_GAIN = TRAIN_ARCH.get("mid_gain", 1.)\nJUDGE_BAND = TRAIN_ARCH.get("judge_band", TRAIN_ARCH.get("band", 9))   # v14e: judged band wider than the drawn one\nGRID_SCALES, EVAL_GRID_SCALE = (.4, .5, .6, .75, 1.), .5\n\n\ndef blur(x, side=9):\n    shape = x.shape\n    flat = x.reshape(-1, shape[-3], *shape[-2:])\n    if side >= 17:\n        return wide_blur(flat, side).reshape(shape)\n    radius = side // 2\n    return F.avg_pool2d(F.pad(flat, (radius,)*4, mode="reflect"), side, stride=1).reshape(shape)\n\n\ndef wide_blur(flat, side):\n    """Wide low-pass (v12b, side 33) that the Neural Engine runs cheaply: a 33 px\n    stride-1 box costs ~0.5 s per 512 px frame there. Mean over 4x4 cells, a\n    (side//4 | 1) px box on that grid, bilinear back up: about the same width\n    (sigma ~10 px vs 9.5 for the 33 px box). Training and the app use this."""\n    h, w = flat.shape[-2:]\n    ph, pw = (-h) % 4, (-w) % 4\n    if ph or pw:\n        flat = F.pad(flat, (0, pw, 0, ph), mode="replicate")\n    small = F.avg_pool2d(flat, 4, stride=4)\n    k = (side//4) | 1\n    r = k//2\n    small = F.avg_pool2d(F.pad(small, (r,)*4, mode="replicate"), k, stride=1)\n    return F.interpolate(small, scale_factor=4., mode="bilinear", align_corners=False)[..., :h, :w]\n\n\ndef highpass(x, side=5):\n    return x - blur(x, side)\n\n\ndef flow_tensor(y, window=17, pre=5, scale=64., eps=1e-3):\n    """Local direction of y [N,1,H,W] (luminance): the structure tensor J (gradient outer\n    product, `window` px average) as (Jxx-Jyy, 2Jxy)/(Jxx+Jyy+eps) [N,2,H,W], i.e.\n    coherence x (cos 2θ, sin 2θ) of the dominant gradient θ (0 = flat or no direction,\n    1 = one direction). Lines along the shape run at θ+90°. `scale` keeps the squared\n    gradients in fp16 range on the Neural Engine."""\n    y = blur(y*scale, pre) if pre > 1 else y*scale\n    gx = F.pad(y[..., :, 2:]-y[..., :, :-2], (1, 1, 0, 0), mode="replicate")*.5\n    gy = F.pad(y[..., 2:, :]-y[..., :-2, :], (0, 0, 1, 1), mode="replicate")*.5\n    j = blur(torch.cat((gx*gx, gy*gy, gx*gy), 1), window)\n    trace = j[:, 0:1]+j[:, 1:2]+eps\n    return torch.cat(((j[:, 0:1]-j[:, 1:2])/trace, 2*j[:, 2:3]/trace), 1)\n\n\ndef line_kernels(orientations=8, length=15, width=.7):\n    """[K,1,L,L] unit-sum line kernels: a Gaussian elongated along φ_k = kπ/K (sigma\n    length/4 along, `width` px across)."""\n    r = length//2\n    yy, xx = torch.meshgrid(torch.arange(-r, r+1.), torch.arange(-r, r+1.), indexing="ij")\n    kernels = []\n    for k in range(orientations):\n        phi = math.pi*k/orientations\n        u, v = xx*math.cos(phi)+yy*math.sin(phi), -xx*math.sin(phi)+yy*math.cos(phi)\n        g = torch.exp(-u.square()/(2*(length/4)**2)-v.square()/(2*width**2))\n        kernels.append(g/g.sum())\n    return torch.stack(kernels)[:, None]\n\n\ndef orientation_alignment(detail, reference, weight, window=17):\n    """v13f: how well the local direction of `detail` matches `reference` (both [N,1,H,W]\n    luminance bands): weighted mean of cos 2Δθ x the reference\'s coherence weight (the\n    detail\'s own amount does not count, only its direction). 1 = parallel everywhere."""\n    a = flow_tensor(detail, window, pre=1, eps=1e-6)\n    b = flow_tensor(reference, window, pre=1, eps=1e-6)\n    na = (a.square().sum(1, keepdim=True)+1e-8).sqrt()\n    nb = (b.square().sum(1, keepdim=True)+1e-8).sqrt()\n    cos = (a*b).sum(1, keepdim=True)/(na*nb)\n    w = weight*nb\n    return (cos*w).sum()/(w.sum()+1e-6)\n\n\ndef inner(mask, side=5):\n    shape = mask.shape\n    flat = F.pad(mask.reshape(-1, 1, *shape[-2:]), (side//2,)*4, value=0)\n    return (-F.max_pool2d(-flat, side, stride=1)).reshape(shape)\n\n\nclass GanROI(nn.Module):\n    """v1 ConditionalROI + zero-output same-size detail branch. `band` is the\n    box side of the branch\'s high-pass (v2 checkpoints: 5)."""\n    def __init__(self, stage, width=32, blocks=6, highpass=False, band=5):\n        super().__init__()\n        self.stage = stage\n        self.architecture = {"width": width, "blocks": blocks, "highpass": highpass, "band": band}\n        channels = stage.backbone.architecture["channels"]\n        self.detail = nn.Sequential(\n            nn.Conv2d(channels+4, width, 3, padding=1), nn.ReLU(),\n            *[base.ResidualBlock(width) for _ in range(blocks)],\n            nn.Conv2d(width, 3, 3, padding=1))\n        nn.init.zeros_(self.detail[-1].weight)\n        nn.init.zeros_(self.detail[-1].bias)\n\n    def forward(self, restored, mosaic, masks):\n        # [B,5,C,H,W] each; same computation as v1 up to its output.\n        b, _, _, h, w = restored.shape\n        backbone = self.stage.backbone\n        condition = torch.cat((mosaic.reshape(b, 15, h, w), masks.reshape(b, 5, h, w)), 1)\n        features = F.relu(backbone.stem(restored.reshape(b, 15, h, w)) + self.stage.condition(condition))\n        body = backbone.body(features)\n        residual = F.interpolate(backbone.shuffle(backbone.head(body)), size=(h, w), mode="area")\n        mask = masks[:, 2]\n        coarse = restored[:, 2] + mask*residual\n        detail = self.detail(torch.cat((body, coarse, mask), 1))\n        if self.architecture["highpass"]:\n            # Texture only: the branch cannot move colour or shading.\n            detail = highpass(detail[:, None], self.architecture["band"])[:, 0]\n        # Mask = 0 stays an exact bypass.\n        return coarse + mask*detail, coarse\n\n\ndef sequence(model, restored, mosaic, mask):\n    b, _, _, h, w = restored.shape\n    arguments = [torch.stack([frames[:, i:i+5] for i in range(3)], 1).reshape(b*3, 5, frames.shape[2], h, w)\n                 for frames in (restored, mosaic, mask)]\n    prediction, coarse = model(*arguments)\n    return prediction.reshape(b, 3, 3, h, w), coarse.reshape(b, 3, 3, h, w)\n\n\nclass Discriminator(nn.Module):\n    """Training-only conditional PatchGAN on (candidate, restored, mask)."""\n    def __init__(self):\n        super().__init__()\n        sn = nn.utils.spectral_norm\n        self.net = nn.Sequential(\n            sn(nn.Conv2d(7, 64, 3, padding=1)), nn.LeakyReLU(.2),\n            sn(nn.Conv2d(64, 64, 4, stride=2, padding=1)), nn.LeakyReLU(.2),\n            sn(nn.Conv2d(64, 128, 4, stride=2, padding=1)), nn.LeakyReLU(.2),\n            sn(nn.Conv2d(128, 128, 3, padding=1)), nn.LeakyReLU(.2),\n            sn(nn.Conv2d(128, 1, 3, padding=1)))\n\n    def forward(self, candidate, restored, mask):\n        c = candidate.reshape(-1, 3, *candidate.shape[-2:])\n        r = restored.reshape_as(c)\n        m = mask.reshape(-1, 1, *candidate.shape[-2:])\n        logits = self.net(torch.cat((c, r, m), 1))\n        support = F.interpolate(inner(m, 9), size=logits.shape[-2:], mode="area")\n        return logits, support\n\n\ndef texture_goal(target, gain, band, mid_gain=1., wide_band=0):\n    """The texture amount the training aims for: the truth with its band <= `band` px\n    scaled by `gain` (v13c: only the fine band; coarser relief keeps the truth\'s) and,\n    v13d, its relief between `band` and `wide_band` px scaled by `mid_gain`."""\n    goal = target if gain == 1 else target+(gain-1)*highpass(target, band)\n    if mid_gain != 1 and wide_band:\n        goal = goal+(mid_gain-1)*(highpass(target, wide_band)-highpass(target, band))\n    return goal\n\n\ndef ghost_edges(prediction, coarse, mask, inner_px=2, outer_px=14):\n    """v13d: new edges beside outlines. Luminance (5 px box-smoothed, so fine texture does\n    not count) gradient that the prediction has and the coarse (distill) image has not,\n    in a ring 2-14 px around the coarse image\'s outlines, relative to the coarse gradient."""\n    b, t, _, h, w = prediction.shape\n\n    def gradient(y):           # y [N,1,H,W]\n        gx = F.pad(y[..., :, 1:]-y[..., :, :-1], (0, 1, 0, 0), mode="replicate")\n        gy = F.pad(y[..., 1:, :]-y[..., :-1, :], (0, 0, 0, 1), mode="replicate")\n        return torch.sqrt(gx.square()+gy.square()+1e-12)\n\n    def grad(img):\n        img = img.float().reshape(b*t, 3, h, w)\n        y = .299*img[:, 0:1]+.587*img[:, 1:2]+.114*img[:, 2:3]\n        return gradient(blur(y[:, None], 5)[:, 0])\n    g_p, g_c = grad(prediction), grad(coarse)\n    flat = blur(coarse.float().reshape(b*t, 3, h, w).mean(1, keepdim=True)[:, None], 3)[:, 0]\n    outline = (gradient(flat) >= OUTLINE_THRESHOLD).float()\n    outer = F.max_pool2d(outline, 2*outer_px+1, stride=1, padding=outer_px)\n    core = F.max_pool2d(outline, 2*inner_px+1, stride=1, padding=inner_px)\n    ring = (outer-core).clamp(0, 1)*mask.float().reshape(b*t, 1, h, w)\n    wm = paired.weighted_mean\n    return wm(F.relu(g_p-g_c), ring)/(wm(g_c, ring)+1e-6)\n\n\ndef texture_target(prediction, target, mask, side=9, gain=1.):\n    """The prediction plus the 9 px high-pass of (target - prediction) inside\n    the mask: a reference that differs from the prediction only in texture.\n    gain scales the target\'s texture (v10c: a goal of `gain` x the truth\'s)."""\n    p = prediction.detach()\n    return p+mask*(gain*highpass(target, side)-highpass(p, side))\n\n\ndef perceptual(prediction, reference, mask, lpips_model):\n    b, t, c, h, w = prediction.shape\n    distances = lpips_model(prediction.reshape(b*t, c, h, w)*2-1, reference.reshape(b*t, c, h, w)*2-1)\n    weights = F.interpolate(mask.reshape(b*t, 1, h, w), size=distances.shape[-2:], mode="area")\n    return paired.weighted_mean(distances, weights)\n\n\ndef metrics(prediction, target, mask, lpips_model=None, coarse=None, texture_side=None):\n    prediction, target, mask = prediction.float(), target.float(), mask.float()\n    wm = paired.weighted_mean\n    error = prediction-target\n    inside = inner(mask)\n    low = wm((blur(prediction)-blur(target)).abs(), mask)\n    # How far the branch moved the low band away from the (frozen) coarse image.\n    anchor = prediction.new_zeros(()) if coarse is None else wm((blur(prediction)-blur(coarse.float())).abs(), mask)\n    pixel = wm(torch.sqrt(error.square()+1e-6), mask)\n    hp_p, hp_t = highpass(prediction), highpass(target)\n    hp_err = hp_p-hp_t\n    local_p = (blur(hp_p.square(), 9)+1e-8).sqrt()\n    local_t = (blur(hp_t.square(), 9)+1e-8).sqrt()\n    amplitude = wm((local_p-local_t).abs(), inner(mask, 9))\n    support = torch.minimum(inside[:, 1:], inside[:, :-1])\n    detail_temporal = wm((hp_err[:, 1:]-hp_err[:, :-1]).abs(), support)\n    temporal = wm((error[:, 1:]-error[:, :-1]).abs(), torch.minimum(mask[:, 1:], mask[:, :-1]))\n    rms_p = wm(hp_p.square(), inside).sqrt()\n    rms_t = wm(hp_t.square(), inside).sqrt()\n    corr = wm(hp_p*hp_t, inside)/(rms_p*rms_t+1e-8)\n    ratio = rms_p/(rms_t+1e-8)\n    distance = prediction.new_zeros(())\n    if lpips_model is not None:\n        reference = (prediction.detach()*(1-mask)+target*mask if texture_side is None else\n                     texture_target(prediction, target, mask, texture_side))\n        distance = perceptual(prediction, reference, mask, lpips_model)\n    return {"low": low, "anchor": anchor, "pixel": pixel, "amplitude": amplitude,\n            "detail_temporal": detail_temporal,\n            "temporal": temporal, "lpips": distance, "mse": wm(error.square(), mask),\n            "detail_corr": corr, "detail_ratio": ratio,\n            # Correct AND present detail: correlation x amplitude agreement.\n            "detail_match": corr*torch.minimum(ratio, 1/(ratio+1e-8))}\n\n\ndef generator_loss(m, weights):\n    return (weights["low"]*m["low"]+weights.get("anchor", 0.)*m["anchor"]+weights["pixel"]*m["pixel"]+\n            weights["lpips"]*m["lpips"]+weights["amplitude"]*m["amplitude"]+\n            weights["detail_temporal"]*m["detail_temporal"])\n\n\ndef score(report):\n    # Lower is better: perceptual distance plus detail amount away from target.\n    return report["lpips"]+.3*abs(math.log(max(report["detail_ratio"], 1e-3)))\n\n\ndef guard(report, reference):\n    # Added detail that does not sit exactly on the target\'s lowers PSNR by\n    # design; PSNR is only a safety bound. Colour/shading and flicker decide.\n    return (report["psnr"] >= reference["psnr"]-3.0 and\n            report["low"] <= reference["low"]*1.10+1e-6 and\n            report["detail_temporal"] <= reference["detail_temporal"]*1.5+1e-6)\n\n\ndef validate_paths(config):\n    prepared = Path(config["prepared_root"]).resolve()\n    initial = Path(config["init_checkpoint"]).resolve()\n    save = Path(config["save_root"]).resolve()\n    work = Path(config["work_root"]).resolve()\n    for output in (save, work):\n        for protected in (prepared, initial.parent):\n            if output == protected or output in protected.parents or protected in output.parents:\n                raise ValueError("保存先・作業先は、素材準備フォルダと元のbest.ptのフォルダから独立させてください。")\n    if save == work or save in work.parents or work in save.parents:\n        raise ValueError("SAVE_ROOTとWORK_ROOTは別の独立フォルダにしてください。")\n    return prepared, initial, save, work\n\n\ndef read_manifest(prepared):\n    path = prepared/"manifest.json"\n    if not path.is_file():\n        raise FileNotFoundError("PREPARED_ROOTにmanifest.jsonがありません。v1の学習で使った素材準備フォルダを指定してください。")\n    manifest = json.loads(path.read_text())\n    if manifest.get("version") != paired.VERSION or not manifest.get("complete"):\n        raise ValueError("マスク付き学習(v1)の完成済み素材ではありません。")\n    return manifest, paired.sha256(path)\n\n\ndef load_stage_one(path, manifest_hash):\n    state = torch.load(path, map_location="cpu", weights_only=True)\n    if state.get("version") != paired.VERSION:\n        raise ValueError("INIT_CHECKPOINTにはマスク付き学習(v1)のbest.ptを指定してください。")\n    if state["contract"]["dataset"] != manifest_hash:\n        raise ValueError("best.ptとPREPARED_ROOTの素材が対応していません。")\n    stage = paired.ConditionalROI(base.TinyROIEnhancer(**base.validated_architecture(state["base_architecture"])))\n    stage.load_state_dict(state["model"], strict=True)\n    if any(not torch.isfinite(p).all() for p in stage.parameters()):\n        raise ValueError("元モデルの重みに非有限値があります。")\n    return stage\n\n\ndef preview(path, rows, detail=False):\n    from PIL import Image\n    if not rows:\n        return\n    side = rows[0][0].shape[-1]\n    picture = Image.new("RGB", (side*len(rows[0]), side*len(rows)))\n    for r, images in enumerate(rows):\n        for c, image in enumerate(images):\n            if detail:\n                image = highpass(image[None])[0]*6+.5\n            array = np.rint(image.detach().float().cpu().clamp(0, 1).permute(1, 2, 0).numpy()*255).astype(np.uint8)\n            picture.paste(Image.fromarray(array), (c*side, r*side))\n    picture.save(path)\n\n\ndef evaluate(model, dataset, device, samples=64, preview_path=None, lpips_model=None, texture_side=None):\n    model.eval()\n    chosen = np.linspace(0, len(dataset)-1, min(samples, len(dataset))).round().astype(int)\n    sums, rows, detail_rows = {}, [], []\n    with torch.inference_mode():\n        for i, index in enumerate(chosen):\n            restored, mosaic, target, mask = [x[None].to(device) for x in dataset[int(index)]]\n            prediction, coarse = sequence(model, restored, mosaic, mask)\n            roi = mask[:, 2:5]\n            values = {k: float(v) for k, v in metrics(prediction, target, roi, lpips_model, coarse, texture_side).items()}\n            if lpips_model is not None and texture_side is not None:\n                # Whole-target LPIPS as in v1/v2, for comparing across versions.\n                composed = prediction*(1-roi)+target*roi\n                values["lpips_full"] = float(perceptual(prediction.float(), composed.float(), roi.float(), lpips_model))\n            if not all(math.isfinite(v) for v in values.values()):\n                raise RuntimeError("検証指標が非有限値です。")\n            for k, v in values.items():\n                sums[k] = sums.get(k, 0.)+v\n            if i < 6:\n                rows.append([mosaic[0, 3], restored[0, 3], coarse[0, 1], prediction[0, 1], target[0, 1],\n                             mask[0, 3].expand(3, -1, -1)])\n                detail_rows.append([coarse[0, 1], prediction[0, 1], target[0, 1]])\n    report = {k: v/len(chosen) for k, v in sums.items()}\n    report.update(psnr=-10*math.log10(max(report["mse"], 1e-12)), samples=len(chosen),\n                  lpips_evaluated=lpips_model is not None)\n    if preview_path:\n        preview(preview_path, rows)\n        preview(Path(preview_path).with_name("detail-"+Path(preview_path).name), detail_rows, detail=True)\n    return report\n\n\ndef model_metadata(model):\n    if hasattr(model, "distill"):\n        return {"stage_version": base.VERSION, "base_architecture": model.distill.architecture}\n    return {"stage_version": paired.VERSION, "base_architecture": model.stage.backbone.architecture}\n\n\ndef save_checkpoint(path, model, discriminator, optimizers, step, contract, reference, best_score, stale):\n    for net in (model, discriminator):\n        if any(not torch.isfinite(p).all() for p in net.parameters()):\n            raise RuntimeError("非有限の重みは保存しません。直前のチェックポイントは保持します。")\n    temporary = Path(path).with_suffix(".partial")\n    torch.save({"version": VERSION, **model_metadata(model),\n                "detail_architecture": model.architecture, "model": model.state_dict(),\n                "discriminator": discriminator.state_dict(),\n                "optimizers": [o.state_dict() for o in optimizers], "step": step, "contract": contract,\n                "reference": reference, "best_score": best_score, "stale": stale,\n                "torch_rng": torch.get_rng_state(), "python_rng": random.getstate(),\n                "app_integrated": False,\n                "input_contract": "restored+separate_mosaic+mask,5frames,same-size-output"}, temporary)\n    os.replace(temporary, path)\n\n\ndef save_snapshot(path, model, step, contract, report):\n    """Weights only (no discriminator/optimiser): enough for the previews."""\n    if any(not torch.isfinite(p).all() for p in model.parameters()):\n        raise RuntimeError("非有限の重みは保存しません。")\n    temporary = Path(path).with_suffix(".partial")\n    torch.save({"version": VERSION, **model_metadata(model),\n                "detail_architecture": model.architecture, "model": model.state_dict(),\n                "step": step, "contract": contract, "report": report, "app_integrated": False}, temporary)\n    os.replace(temporary, path)\n\n\nclass TextureDiscriminator(nn.Module):\n    """Training-only PatchGAN on the branch\'s high band (`band` px box) and\n    the mask. It never sees colour or the restoration, so any uncensored crop\n    is a REAL sample."""\n    def __init__(self, band=9, gain=4.):\n        super().__init__()\n        self.band, self.gain = band, gain\n        sn = nn.utils.spectral_norm\n        self.net = nn.Sequential(\n            sn(nn.Conv2d(4, 64, 3, padding=1)), nn.LeakyReLU(.2),\n            sn(nn.Conv2d(64, 64, 4, stride=2, padding=1)), nn.LeakyReLU(.2),\n            sn(nn.Conv2d(64, 128, 4, stride=2, padding=1)), nn.LeakyReLU(.2),\n            sn(nn.Conv2d(128, 128, 3, padding=1)), nn.LeakyReLU(.2),\n            sn(nn.Conv2d(128, 1, 3, padding=1)))\n\n    def forward(self, candidate, mask):\n        c = candidate.reshape(-1, 3, *candidate.shape[-2:])\n        m = mask.reshape(-1, 1, *candidate.shape[-2:])\n        logits = self.net(torch.cat((highpass(c, self.band)*self.gain, m), 1))\n        support = F.interpolate(inner(m, 9), size=logits.shape[-2:], mode="area")\n        return logits, support\n\n\nclass ContextTextureDiscriminator(TextureDiscriminator):\n    """v14: PatchGAN on the high band together with the place it belongs to: the low band (a\n    65 px blur) of a context image, plus the mask. Truth and generated output use the distill\n    image as context (same place, different detail); a bank crop uses its own blur. One more\n    stride-2 stage widens the view so the shape of the region is visible."""\n    def __init__(self, band=33, gain=4., low_cut=0):\n        nn.Module.__init__(self)\n        self.band, self.gain, self.low_cut = band, gain, low_cut\n        sn = nn.utils.spectral_norm\n        self.net = nn.Sequential(\n            sn(nn.Conv2d(7, 64, 3, padding=1)), nn.LeakyReLU(.2),\n            sn(nn.Conv2d(64, 64, 4, stride=2, padding=1)), nn.LeakyReLU(.2),\n            sn(nn.Conv2d(64, 128, 4, stride=2, padding=1)), nn.LeakyReLU(.2),\n            sn(nn.Conv2d(128, 128, 4, stride=2, padding=1)), nn.LeakyReLU(.2),\n            sn(nn.Conv2d(128, 128, 3, padding=1)), nn.LeakyReLU(.2),\n            sn(nn.Conv2d(128, 1, 3, padding=1)))\n\n    def forward(self, candidate, mask, context=None):\n        c = candidate.reshape(-1, 3, *candidate.shape[-2:])\n        m = mask.reshape(-1, 1, *candidate.shape[-2:])\n        ctx = c if context is None else context.reshape(-1, 3, *candidate.shape[-2:])\n        low = blur(ctx[:, None], 65)[:, 0]-.5\n        detail = highpass(c, self.band)\n        if self.low_cut:   # the same band the generator draws (no verdict on pixel grain)\n            detail = detail-highpass(c, self.low_cut)\n        logits = self.net(torch.cat((detail*self.gain, low, m), 1))\n        support = F.interpolate(inner(m, 9), size=logits.shape[-2:], mode="area")\n        return logits, support\n\n\ndef load_flow(device):\n    from torchvision.models.optical_flow import Raft_Small_Weights, raft_small\n    return raft_small(weights=Raft_Small_Weights.DEFAULT, progress=False).to(device).eval().requires_grad_(False)\n\n\ndef load_features(device):\n    from torchvision.models import VGG16_Weights, vgg16\n    return vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features[:23].to(device).eval().requires_grad_(False)\n\n\ndef warp(x, flow):\n    """Sample x [N,C,H,W] at p + flow(p) (flow in pixels, [N,2,H,W])."""\n    _, _, h, w = x.shape\n    ys, xs = torch.meshgrid(torch.arange(h, device=x.device, dtype=x.dtype),\n                            torch.arange(w, device=x.device, dtype=x.dtype), indexing="ij")\n    grid = torch.stack(((xs+flow[:, 0])/max(w-1, 1)*2-1, (ys+flow[:, 1])/max(h-1, 1)*2-1), -1)\n    return F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=True)\n\n\ndef motion(flow_model, frames):\n    """Flow t -> t+1 between consecutive frames [B,T,3,H,W] and a\n    forward-backward validity mask; computed without gradients in FP32."""\n    b, t, c, h, w = frames.shape\n    first = frames[:, :-1].reshape(-1, c, h, w).float()*2-1\n    second = frames[:, 1:].reshape(-1, c, h, w).float()*2-1\n    with torch.no_grad(), torch.autocast(device_type=frames.device.type, enabled=False):\n        flows = flow_model(torch.cat((first, second)), torch.cat((second, first)))[-1]\n        forward, backward = flows.chunk(2)\n        back = warp(backward, forward)\n        mismatch = (forward+back).square().sum(1, keepdim=True)\n        scale = forward.square().sum(1, keepdim=True)+back.square().sum(1, keepdim=True)\n        valid = (mismatch <= .01*scale+.5).float()\n    return forward.reshape(b, t-1, 2, h, w), valid.reshape(b, t-1, 1, h, w)\n\n\ndef motion_change(frames, flows, valid, mask, side=9):\n    """Mean change of the high band along the motion: texture that sticks to\n    the surface scores low, texture that boils from frame to frame high."""\n    b, t, c, h, w = frames.shape\n    band = highpass(frames.float(), side)\n    moved = warp(band[:, 1:].reshape(-1, c, h, w), flows.reshape(-1, 2, h, w)).reshape(b, t-1, c, h, w)\n    support = valid*torch.minimum(inner(mask[:, :-1].float()), inner(mask[:, 1:].float()))\n    return paired.weighted_mean((band[:, :-1]-moved).abs(), support)\n\n\nSTYLE_LAYERS = (3, 8, 15)   # VGG16 relu1_2, relu2_2, relu3_3: texture statistics\nSTRUCTURE_LAYER = 22         # VGG16 relu4_3: edges and shapes, aligned\n\n\ndef vgg_pass(features, x, mask):\n    """Masked Gram matrices at STYLE_LAYERS and the STRUCTURE_LAYER map."""\n    mean = x.new_tensor((.485, .456, .406)).view(1, 3, 1, 1)\n    std = x.new_tensor((.229, .224, .225)).view(1, 3, 1, 1)\n    x, out = (x-mean)/std, []\n    for i, layer in enumerate(features):\n        x = layer(x)\n        if i in STYLE_LAYERS:\n            m = F.interpolate(mask, size=x.shape[-2:], mode="area")\n            out.append(torch.einsum("bchw,bdhw->bcd", x*m, x)/m.sum((-2, -1)).clamp_min(1.)[..., None])\n        if i == STRUCTURE_LAYER:\n            return out, x\n    raise ValueError("VGG特徴がrelu4_3まで読み込まれていません。")\n\n\ndef texture_distances(features, prediction, reference, mask):\n    """(style, structure): position-free relative L1 between masked Gram\n    matrices, and relative L1 between the aligned relu4_3 maps in the mask."""\n    h, w = prediction.shape[-2:]\n    p, r = prediction.reshape(-1, 3, h, w).float(), reference.reshape(-1, 3, h, w).float()\n    m = mask.reshape(-1, 1, h, w).float()\n    (gp_all, fp), (gr_all, fr) = vgg_pass(features, p, m), vgg_pass(features, r, m)\n    style = prediction.new_zeros(())\n    for gp, gr in zip(gp_all, gr_all):\n        style = style+((gp-gr).abs().mean((-2, -1))/(gr.abs().mean((-2, -1))+1e-6)).mean()\n    weight = F.interpolate(m, size=fp.shape[-2:], mode="area")\n    structure = paired.weighted_mean((fp-fr).abs(), weight)/(paired.weighted_mean(fr.abs(), weight)+1e-6)\n    return style/len(STYLE_LAYERS), structure\n\n\ndef metrics_v4(prediction, target, mask, coarse, flows, valid, features, side=5, gain=1., fine_gain_band=0):\n    prediction, target, mask, coarse = prediction.float(), target.float(), mask.float(), coarse.float()\n    # v13c: the amount/style goals use the truth with only its fine band x gain; the\n    # ratio, amplitude and style are then judged against that goal at 1x (truth itself\n    # still decides colour, PSNR, flicker reference and the fine-band correlation).\n    goal, goal_gain = ((texture_goal(target, gain, fine_gain_band, MID_GAIN, side), 1.) if fine_gain_band\n                       else (target, gain))\n    wm = paired.weighted_mean\n    inside = inner(mask)\n    # side > 9 (v12b): the texture band is wide, so the detail amount, local\n    # amplitude and the anchor/low terms use the same band (coarser stays fixed).\n    wide = side > 9\n    hp_p, hp_t = (highpass(prediction, side), highpass(goal, side)) if wide else (highpass(prediction), highpass(goal))\n    local_p = (blur(hp_p.square(), side if wide else 9)+1e-8).sqrt()\n    local_t = (blur(hp_t.square(), side if wide else 9)+1e-8).sqrt()\n    coarse_side = side if wide else 9\n    rms_p = wm(hp_p.square(), inside).sqrt()\n    rms_t = wm(hp_t.square(), inside).sqrt()\n    corr = wm(hp_p*hp_t, inside)/(rms_p*rms_t+1e-8)\n    ratio = rms_p/(rms_t+1e-8)\n    # Fine band (9 px box: about a quarter mosaic cell at the x2 output), where the\n    # restoration\'s shapes are wrong (corr ~.14): did the texture land where the truth\'s is?\n    f_p, f_t = highpass(prediction, 9), highpass(target, 9)       # the truth itself\n    fine_corr = wm(f_p*f_t, inside)/(wm(f_p.square(), inside).sqrt()*wm(f_t.square(), inside).sqrt()+1e-8)\n    style, structure = texture_distances(features, prediction, texture_target(prediction, goal, mask, side, goal_gain), mask)\n    orient = fine_amp = prediction.new_zeros(())\n    fine_ratio = prediction.new_ones(())\n    if wide:\n        # v13f: direction of the added fine detail (<= 17 px) against the truth\'s fine band,\n        # weighted by the truth\'s coherence, inside the mask, outlines excluded.\n        h, w = prediction.shape[-2:]\n\n        def lum(x):\n            return (.299*x[:, :, 0]+.587*x[:, :, 1]+.114*x[:, :, 2]).reshape(-1, 1, h, w)\n        flat = coarse.reshape(-1, 3, h, w)\n        weight = inside.reshape(-1, 1, h, w)*(1-outline_protection(flat, 4, 4))\n        orient = orientation_alignment(highpass(lum(prediction-coarse), 17), highpass(lum(target), 17), weight)\n        # v13g: amount of the <= 17 px band inside (outlines excluded) against the goal\'s.\n        f_p = highpass(lum(prediction), 17)\n        f_g = highpass(lum(goal), 17)\n        lp = (blur(f_p.square(), 17)+1e-8).sqrt()\n        lg = (blur(f_g.square(), 17)+1e-8).sqrt()\n        distance = (lp-lg).abs() if CLEAN_UNET else (lp-lg).abs()+F.relu(lp-lg)\n        fine_amp = (weight*distance).sum()/(weight.sum()+1e-6)/((weight*lg).sum()/(weight.sum()+1e-6)+1e-4)\n        fine_ratio = ((weight*f_p.square()).sum()/((weight*f_g.square()).sum()+1e-8)).sqrt()\n    return {"low": wm((blur(prediction, coarse_side)-blur(target, coarse_side)).abs(), mask),\n            "anchor": wm((blur(prediction, coarse_side)-blur(coarse, coarse_side)).abs(), mask),\n            # Overshoot of the local amount costs double for v13d (contrast stays in check).\n            "amplitude": wm((local_p-goal_gain*local_t).abs()+(F.relu(local_p-goal_gain*local_t) if (MID_GAIN != 1 or VERSION in ("roi-mask-gan-v14b", "roi-mask-gan-v14c", "roi-mask-gan-v14d", "roi-mask-gan-v16")) else 0.),\n                            inner(mask, coarse_side)),\n            "style": style, "structure": structure,\n            "flicker": motion_change(prediction, flows, valid, mask, side),\n            "flicker_truth": motion_change(target, flows, valid, mask, side),\n            "mse": wm((prediction-target).square(), mask), "detail_ratio": ratio,\n            # Logged for reference only: invented texture cannot correlate.\n            "detail_corr": corr, "detail_match": corr*torch.minimum(ratio, 1/(ratio+1e-8)), "fine_corr": fine_corr,\n            # How far the output moved away from the distill alone (mean |change| inside the mask).\n            "change": wm((prediction-coarse).abs(), inside),\n            # Goal of detail_ratio (1 when the goal already carries the gain, v13c).\n            "goal_gain": prediction.new_tensor(goal_gain),\n            # Relief (fine band .. side) amount vs the truth\'s, for the contrast check.\n            "mid_ratio": (wm((hp_p-highpass(prediction, 17)).square(), inside).sqrt()\n                          /(wm((highpass(target, side)-highpass(target, 17)).square(), inside).sqrt()+1e-8)) if wide\n                         else prediction.new_zeros(()),\n            "ghost": ghost_edges(prediction, coarse, mask) if wide else prediction.new_zeros(()),\n            # v13f: 1 = the added fine detail runs the same way as the truth\'s (wrinkles along the shape).\n            "orient": orient,\n            # v13g: <= 17 px amount inside (outlines excluded): loss (relative, overshoot x2) and ratio to the goal.\n            "fine_amp": fine_amp, "fine_ratio": fine_ratio}\n\n\ndef score_v4(report, gain=1.):\n    if V14:\n        # Realism (Gram style), amount near the goal, visible change, few new edges by outlines.\n        return (report["style"]+.3*abs(math.log(max(report["detail_ratio"], 1e-3)/report.get("goal_gain", gain)))\n                +(.3*abs(math.log(max(report.get("fine_ratio", 1.), 1e-3))) if CLEAN_UNET else 0.)\n                -(0. if CLEAN_UNET else 10*report.get("change", 0.))\n                -(.5*report.get("teacher_corr", 0.) if V19 else 0.)\n                +(2*report.get("ghost", 0.) if NO_RELIEF else\n                  0. if VERSION in ("roi-mask-gan-v14d", "roi-mask-gan-v14e", "roi-mask-gan-v14f", "roi-mask-gan-v16", "roi-mask-gan-v17")\n                  else 2*report.get("ghost", 0.)))\n    return _score_v13(report, gain)\n\n\ndef _score_v13(report, gain=1.):\n    # Lower is better: texture statistics and structure away from the\n    # target\'s, plus detail amount away from the goal (gain x the target\'s).\n    # v13 adds the paired fine-shape distance (what the whole ROI predicts).\n    # v13 also rewards the fine-band correlation with the truth (shapes that landed right).\n    # v13b: picking up detail means moving away from the distill, so the change from the\n    # distill is rewarded too (20 per unit = .08 per 8-bit level), within the guards.\n    context = "fine" in report\n    return (report["style"]+report.get("structure", 0.)+report.get("fine", 0.)\n            -(report.get("fine_corr", 0.) if context else 0.)\n            -(20*report.get("change", 0.) if context and VERSION in ("roi-mask-gan-v13b", "roi-mask-gan-v13c", "roi-mask-gan-v13d", "roi-mask-gan-v13e", "roi-mask-gan-v13f", "roi-mask-gan-v13g", "roi-mask-gan-v13h", "roi-mask-gan-v13i", "roi-mask-gan-v13j") else 0.)\n            -(report.get("orient", 0.) if VERSION in ("roi-mask-gan-v13f", "roi-mask-gan-v13g", "roi-mask-gan-v13h", "roi-mask-gan-v13i") else 0.)\n            +(.5*abs(math.log(max(report.get("fine_ratio", 1.), 1e-3))) if VERSION in ("roi-mask-gan-v13g", "roi-mask-gan-v13h", "roi-mask-gan-v13i") else 0.)\n            +(report.get("ghost", 0.) if VERSION == "roi-mask-gan-v13d" else 2*report.get("ghost", 0.) if VERSION in ("roi-mask-gan-v13h", "roi-mask-gan-v13i") else 0.)\n            +.3*abs(math.log(max(report["detail_ratio"], 1e-3)/report.get("goal_gain", gain))))\n\n\ndef up_masks(masks, side):\n    """[B,T,1,s,s] masks -> [B,T,1,side,side] (bilinear, as the model\'s x2 mask)."""\n    b, t = masks.shape[:2]\n    return F.interpolate(masks.reshape(b*t, 1, *masks.shape[-2:]).float(), size=(side, side), mode="bilinear",\n                         align_corners=False).reshape(b, t, 1, side, side)\n\n\ndef guard_v4(report, reference, gain=1.):\n    # Flicker scales with the texture amount, so its allowance scales with the goal.\n    # At the 512 output the distill alone can already flicker more than the truth (v13):\n    # then the allowance is not worse than the start (+5%). v13d ignores the colour guard\n    # (user decision for the v13b line).\n    return (report["psnr"] >= reference["psnr"]-3.0 and\n            (VERSION in ("roi-mask-gan-v13d", "roi-mask-gan-v13e", "roi-mask-gan-v13f", "roi-mask-gan-v13g", "roi-mask-gan-v13h", "roi-mask-gan-v13i", "roi-mask-gan-v13j", "roi-mask-gan-v14", "roi-mask-gan-v14b", "roi-mask-gan-v14c", "roi-mask-gan-v14d", "roi-mask-gan-v14e", "roi-mask-gan-v14f", "roi-mask-gan-v16", "roi-mask-gan-v17", "roi-mask-gan-v18") or report["low"] <= reference["low"]*1.10+1e-6) and\n            report["flicker"] <= max(report["flicker_truth"]*1.2*gain, reference["flicker"]*1.05)+1e-6 and\n            (VERSION not in ("roi-mask-gan-v13h", "roi-mask-gan-v13i") or report.get("ghost", 0.) <= GHOST_LIMIT) and\n            (not V14 or report.get("ghost", 0.) <= GHOST_LIMIT_V14) and\n            (not NO_RELIEF or report.get("ghost", 0.) <= GHOST_LIMIT_V18))\n\n\ndef evaluate_v4(model, dataset, device, flow_model, features, samples=64, preview_path=None, side=5):\n    model.eval()\n    chosen = np.linspace(0, len(dataset)-1, min(samples, len(dataset))).round().astype(int)\n    sums, rows, detail_rows = {}, [], []\n    with torch.inference_mode():\n        for i, index in enumerate(chosen):\n            restored, mosaic, target, mask = [x[None].to(device) for x in dataset[int(index)]]\n            prediction, coarse = sequence(model, restored, mosaic, mask)\n            flows, valid = motion(flow_model, coarse)\n            values = {k: float(v) for k, v in\n                      metrics_v4(prediction, target, mask[:, 2:5], coarse, flows, valid, features, side).items()}\n            if not all(math.isfinite(v) for v in values.values()):\n                raise RuntimeError("検証指標が非有限値です。")\n            for k, v in values.items():\n                sums[k] = sums.get(k, 0.)+v\n            if i < 6:\n                rows.append([mosaic[0, 3], restored[0, 3], coarse[0, 1], prediction[0, 1], target[0, 1],\n                             mask[0, 3].expand(3, -1, -1)])\n                detail_rows.append([coarse[0, 1], prediction[0, 1], target[0, 1]])\n    report = {k: v/len(chosen) for k, v in sums.items()}\n    report.update(psnr=-10*math.log10(max(report["mse"], 1e-12)), samples=len(chosen))\n    if preview_path:\n        preview(preview_path, rows)\n        preview(Path(preview_path).with_name("detail-"+Path(preview_path).name), detail_rows, detail=True)\n    return report\n\n\nclass TextureDistillROI(nn.Module):\n    """restored -> texture (per frame, grid scale) -> ROI distill (frozen, x2)\n    -> mask composite at x2. Zero texture output layer: step 0 is exactly the\n    distill alone. v8: mosaic_input=False, edge_guard on the restored frames.\n    v9: seed_channels of noise (training: per sample; otherwise the stored seed)."""\n    def __init__(self, distill, width=32, blocks=6, band=3, edge_guard=0., mosaic_input=True, seed_channels=0, **_):\n        super().__init__()\n        self.distill = distill\n        self.architecture = {"width": width, "blocks": blocks, "band": band,\n                             "order": "texture-then-distill", "scale": 2}\n        if edge_guard:\n            self.architecture["edge_guard"] = edge_guard\n        if not mosaic_input:\n            self.architecture["mosaic_input"] = False\n        if seed_channels:\n            self.architecture["seed_channels"] = seed_channels\n            generator = torch.Generator().manual_seed(SEED_VALUE)\n            self.register_buffer("seed", torch.randn(1, seed_channels, SEED_SIDE, SEED_SIDE, generator=generator))\n        self.texture = nn.Sequential(\n            nn.Conv2d((7 if mosaic_input else 4)+seed_channels, width, 3, padding=1), nn.ReLU(),\n            *[base.ResidualBlock(width) for _ in range(blocks)],\n            nn.Conv2d(width, 3, 3, padding=1))\n        nn.init.zeros_(self.texture[-1].weight)\n        nn.init.zeros_(self.texture[-1].bias)\n\n    def fixed_seed(self, b, h, w):\n        """The stored seed, anchored to the crop\'s top-left (tiled beyond 256 px)."""\n        seed = self.seed\n        if h > seed.shape[-2] or w > seed.shape[-1]:\n            seed = seed.repeat(1, 1, -(-h//seed.shape[-2]), -(-w//seed.shape[-1]))\n        return seed[..., :h, :w].expand(b, -1, -1, -1)\n\n    def textured(self, restored, mosaic, masks, noise=None):\n        """[B,5,C,h,w] each -> the five restored frames with texture inside their masks."""\n        b, t, _, h, w = restored.shape\n        if self.architecture.get("mosaic_input", True):\n            inputs = torch.cat((restored, mosaic, masks), 2)\n        else:\n            inputs = torch.cat((restored, masks), 2)\n        if self.architecture.get("seed_channels"):\n            noise = self.fixed_seed(b, h, w) if noise is None else noise\n            inputs = torch.cat((inputs, noise.to(inputs.dtype)[:, None].expand(-1, t, -1, -1, -1)), 2)\n        detail = self.texture(inputs.reshape(b*t, inputs.shape[2], h, w)).reshape(b, t, 3, h, w)\n        detail = highpass(detail, self.architecture["band"])\n        if self.architecture.get("edge_guard"):\n            detail = detail*(1-DistillTextureROI.edges(self, restored.reshape(b*t, 3, h, w)).reshape(b, t, 1, h, w))\n        return restored+masks*detail\n\n    def composite(self, frames, restored, masks):\n        """x2 distill of a five-frame window, pasted through the x2 centre mask."""\n        b, _, _, h, w = frames.shape\n        enhanced = self.distill(frames.reshape(b, 15, h, w))\n        under = F.interpolate(restored[:, 2], scale_factor=2., mode="bilinear", align_corners=False)\n        mask = F.interpolate(masks[:, 2], scale_factor=2., mode="bilinear", align_corners=False)\n        return under+mask*(enhanced-under)\n\n    def enhance(self, restored, mosaic, masks, noise=None):\n        return self.composite(self.textured(restored, mosaic, masks, noise), restored, masks)\n\n    def forward(self, restored, mosaic, masks, noise=None):\n        # Second output: the same path without texture (the distill alone).\n        return self.enhance(restored, mosaic, masks, noise), self.composite(restored, restored, masks)\n\n\nclass DistillTextureROI(nn.Module):\n    """restored -> ROI distill (frozen, x2) -> texture at x2 -> mask composite\n    at x2. Zero texture output layer: step 0 is exactly the distill alone.\n    mosaic_input=False (v7b): the texture branch does not see the mosaic."""\n    def __init__(self, distill, width=32, blocks=6, band=9, edge_guard=.06, mosaic_input=True, **_):\n        super().__init__()\n        self.distill = distill\n        self.architecture = {"width": width, "blocks": blocks, "band": band, "edge_guard": edge_guard,\n                             "order": "distill-then-texture", "scale": 2}\n        if not mosaic_input:\n            self.architecture["mosaic_input"] = False\n        self.texture = nn.Sequential(\n            nn.Conv2d(7 if mosaic_input else 4, width, 3, padding=1), nn.ReLU(),\n            *[base.ResidualBlock(width) for _ in range(blocks)],\n            nn.Conv2d(width, 3, 3, padding=1))\n        nn.init.zeros_(self.texture[-1].weight)\n        nn.init.zeros_(self.texture[-1].bias)\n\n    def forward(self, restored, mosaic, masks):\n        b, _, _, h, w = restored.shape\n        enhanced = self.distill(restored.reshape(b, 15, h, w))\n\n        def up(x):\n            return F.interpolate(x, scale_factor=2., mode="bilinear", align_corners=False)\n        under, mask = up(restored[:, 2]), up(masks[:, 2])\n        if self.architecture.get("mosaic_input", True):\n            detail = self.texture(torch.cat((enhanced, up(mosaic[:, 2]), mask), 1))\n        else:\n            detail = self.texture(torch.cat((enhanced, mask), 1))\n        detail = highpass(detail[:, None], self.architecture["band"])[:, 0]\n        if self.architecture["edge_guard"]:\n            detail = detail*(1-self.edges(enhanced))\n        # Second output: the distill alone, pasted the same way.\n        coarse = under+mask*(enhanced-under)\n        return coarse+mask*detail, coarse\n\n    def edges(self, image):\n        """0 on flat areas .. 1 on edges at least edge_guard per pixel (blurred luminance)."""\n        luminance = blur(image.mean(1, keepdim=True)[:, None], 3)[:, 0]\n        gx = F.pad(luminance[..., :, 1:]-luminance[..., :, :-1], (0, 1, 0, 0), mode="replicate")\n        gy = F.pad(luminance[..., 1:, :]-luminance[..., :-1, :], (0, 0, 0, 1), mode="replicate")\n        strength = torch.sqrt(gx.square()+gy.square()+1e-12)\n        strength = F.max_pool2d(strength, 3, stride=1, padding=1)   # cover both sides of an edge\n        return (strength/self.architecture["edge_guard"]).clamp(0, 1)\n\n\n# Laws of the texture the distill output lacks, measured 2026-10-06 with\n# roi_texture_laws.py: train split, 1200 patches, interior only (mask > .9,\n# edge guard < .5), grid scale .5 so one x2 output pixel is one native pixel,\n# luminance in 8-bit levels. Power per radial frequency bin (cycles/pixel).\nSEED_LAWS = {\n    "frequency": [.0208, .0625, .1042, .1458, .1875, .2292, .2708, .3125, .3542, .3958, .4375, .4792],\n    "power_target": [17773.79, 1360.678, 110.977, 28.3377, 8.6347, 2.9387, 1.09, .5498, .3531, .2664, .2168, .1951],\n    "power_distill": [16801.13, 1082.066, 57.7428, 10.758, 2.7932, .8619, .2534, .069, .0161, .0049, .0031, .0041],\n    # Below 1/8 cycle per pixel the gap is misplaced shading, not texture.\n    "lowest_frequency": .125,\n    # Weighted mean cos 2(missing fine structure - distill contour gradient); 0.044 for mismatched patches.\n    "alignment": .5001,\n    # Local RMS (5 px high band, 9 px window) of the missing texture = a + b * the same of the\n    # distill output (600 patches, interior, Spearman 0.66; luminance -0.05, highlight 0.02).\n    "amplitude": (.5379, .8706),\n    # Steering strength calibrated on 199 real train patches so that the seed\'s alignment under\n    # the same measurement equals the measured 0.50 (0.07 unsteered, 0.53 at 0.5).\n    "steering_kappa": .4637,\n    "chroma_over_luma_power": .147,   # small: the seed is luminance only\n}\n# P2: the same laws measured only farther than 6 px from outlines (edge guard >= .5). Within\n# 6 px the "missing" detail tracks the distill output\'s own detail (3.7 vs 3.5 at 1 px, 1.75\n# vs 1.07 at 4 px): outline blur and misplacement, not texture; beyond 6 px the distill\'s\n# detail is within 2x of its far level (profile, 400 train patches). The seed is faded out\n# inside that zone instead of extrapolating a law that was not measured there.\nSEED_LAWS_P2 = {\n    "frequency": [.0208, .0625, .1042, .1458, .1875, .2292, .2708, .3125, .3542, .3958, .4375, .4792],\n    "power_target": [10727.08, 747.2622, 62.0494, 17.4094, 5.7633, 2.0333, .785, .4127, .2805, .2229, .1879, .1716],\n    "power_distill": [9654.46, 434.1896, 7.2293, .6218, .0918, .0177, .0044, .0019, .0013, .001, .001, .0022],\n    "lowest_frequency": .125,\n    "alignment": .2772,            # null .0479; half of v1\'s .50 came from outline errors\n    "amplitude": (.488, 1.1267),   # far-zone local hp5 RMS law (Spearman .58)\n    "chroma_over_luma_power": .185,\n    "outline_distance": 6,\n    # Calibrated on 198 real train patches with the same far-zone measurement: steering to the\n    # measured alignment, gain so the seed\'s mean local amplitude equals the measured .697 (.585\n    # before: the fade near the zone border and the RMS estimator).\n    "steering_kappa": .2771,\n    "amplitude_gain": .697/.585,\n}\nSEED_ORIENTATIONS, SEED_WINDOW_KAPPA = 8, 4.\nSEED_WRAP = 2\n\n\ndef seed_gap_spectrum(frequency, laws=SEED_LAWS):\n    """Missing power (target - distill, levels^2 per frequency sample) at |f|: the\n    measured value of the radial bin |f| falls in (bins of 1/24 cycle per pixel, as\n    measured), zero below the texture band."""\n    gap = (torch.tensor(laws["power_target"], dtype=frequency.dtype, device=frequency.device)\n           - torch.tensor(laws["power_distill"], dtype=frequency.dtype, device=frequency.device)).clamp(min=0)\n    bins = len(laws["frequency"])\n    index = (frequency*2*bins).floor().long().clamp(0, bins-1)\n    return gap[index]*(frequency >= laws["lowest_frequency"])\n\n\ndef seed_angles(device=None, dtype=torch.float32):\n    return torch.arange(SEED_ORIENTATIONS, device=device, dtype=dtype)*math.pi/SEED_ORIENTATIONS\n\n\ndef seed_fields(white, laws=SEED_LAWS):\n    """K independent white fields [B,K,H,W] -> [B,K,H,W]: field k carries the\n    missing-texture spectrum in orientation band k (bands are a partition of\n    unity in angle), in luminance levels. Independent fields keep the sum at\n    exactly the measured spectrum and let per-pixel weights steer it."""\n    b, k, h, w = white.shape\n    if k != SEED_ORIENTATIONS:\n        raise ValueError("種の白色ノイズは向きの数だけ独立に必要です。")\n    fy = torch.fft.fftfreq(h, device=white.device, dtype=white.dtype)[:, None]\n    fx = torch.fft.rfftfreq(w, device=white.device, dtype=white.dtype)[None, :]\n    amplitude = torch.sqrt(seed_gap_spectrum(torch.sqrt(fx**2+fy**2), laws))\n    phi = torch.atan2(fy.expand(h, fx.shape[1]), fx.expand(h, fx.shape[1]))      # stripe gradient direction\n    logits = SEED_WINDOW_KAPPA*torch.cos(2*(phi[None]-seed_angles(white.device, white.dtype)[:, None, None]))\n    windows = torch.softmax(logits, 0)                                            # [K,h,w/2+1], sums to 1\n    spectrum = torch.fft.rfft2(white.float())\n    fields = torch.fft.irfft2(spectrum*(amplitude*torch.sqrt(windows))[None], s=(h, w))\n    return fields.to(white.dtype)\n\n\n# v11 seed primitives: periodic (512 torus) fields of different shapes. The seed\n# never reaches the output directly; it is injected inside the texture generator\n# and only decides where details fall. Shapes are varied so the generator is not\n# limited to grain: oriented bands (measured spectrum), dots (pores/specks),\n# short and long strokes (hairs, fine wrinkles), ridges (veins, creases),\n# cells (skin micro-relief) and slow blobs (mottling). Each channel: zero mean,\n# unit variance.\nSEED_PRIMITIVES = ("band0", "band1", "band2", "band3", "band4", "band5", "band6", "band7",\n                   "dots_fine", "dots_coarse", "strokes_short", "strokes_long",\n                   "ridges_fine", "ridges_coarse", "cells", "blobs")\n\n\ndef _torus_noise(generator, side, period_lo, period_hi):\n    """Band-limited periodic noise with periods between period_lo and period_hi px."""\n    white = torch.randn(side, side, generator=generator)\n    f = torch.fft.fftfreq(side)\n    radius = torch.sqrt(f[:, None]**2+f[None, :]**2)\n    band = ((radius >= 1/period_hi) & (radius <= 1/period_lo)).float()\n    return torch.fft.ifft2(torch.fft.fft2(white)*band).real\n\n\ndef _splat(side, xs, ys, values, sigma):\n    """Sum of periodic Gaussians at (xs, ys)."""\n    out = torch.zeros(side, side)\n    r = max(1, int(math.ceil(3*sigma)))\n    offsets = torch.arange(-r, r+1)\n    for x, y, v in zip(xs.tolist(), ys.tolist(), values.tolist()):\n        ix, iy = int(math.floor(x)), int(math.floor(y))\n        gx = torch.exp(-((ix+offsets)-x)**2/(2*sigma**2))\n        gy = torch.exp(-((iy+offsets)-y)**2/(2*sigma**2))\n        rows, cols = (iy+offsets) % side, (ix+offsets) % side\n        out[rows[:, None], cols[None, :]] += v*gy[:, None]*gx[None, :]\n    return out\n\n\ndef _strokes(generator, side, count, length_lo, length_hi, width):\n    """Antialiased straight strokes with random orientation and length, periodic."""\n    out = torch.zeros(side, side)\n    for _ in range(count):\n        cx, cy = (torch.rand(2, generator=generator)*side).tolist()\n        angle = float(torch.rand(1, generator=generator))*math.pi\n        length = length_lo+float(torch.rand(1, generator=generator))*(length_hi-length_lo)\n        sign = 1. if float(torch.rand(1, generator=generator)) < .6 else -1.\n        dx, dy = math.cos(angle), math.sin(angle)\n        half = length/2+2*width\n        r = int(math.ceil(half))\n        ys = torch.arange(int(cy)-r, int(cy)+r+1)\n        xs = torch.arange(int(cx)-r, int(cx)+r+1)\n        X, Y = (xs[None, :].float()-cx), (ys[:, None].float()-cy)\n        along = (X*dx+Y*dy).clamp(-length/2, length/2)\n        dist2 = (X-along*dx)**2+(Y-along*dy)**2\n        out[(ys % side)[:, None], (xs % side)[None, :]] += sign*torch.exp(-dist2/(2*width**2))\n    return out\n\n\ndef _cells(generator, side, cell):\n    """Worley F2-F1 on a jittered periodic grid: a network of thin cell borders."""\n    n = side//cell\n    step = side/n                       # exact tiling of the torus\n    grid = torch.stack(torch.meshgrid(torch.arange(n).float(), torch.arange(n).float(), indexing="ij"), -1)\n    pts = ((grid+torch.rand(n, n, 2, generator=generator))*step).reshape(-1, 2)\n    yy, xx = torch.meshgrid(torch.arange(side).float(), torch.arange(side).float(), indexing="ij")\n    best = torch.full((side, side, 2), 1e9)\n    for py, px in pts.tolist():\n        dy = torch.remainder(yy-py+side/2, side)-side/2\n        dx = torch.remainder(xx-px+side/2, side)-side/2\n        d = dx*dx+dy*dy\n        first = torch.minimum(best[..., 0], d)\n        second = torch.where(d < best[..., 0], best[..., 0], torch.minimum(best[..., 1], d))\n        best = torch.stack((first, second), -1)\n    edge = torch.sqrt(best[..., 1])-torch.sqrt(best[..., 0])\n    return -torch.exp(-edge/1.2)\n\n\ndef seed_primitives(seed_value=SEED_VALUE, side=2*SEED_SIDE, laws=None):\n    """[16, side, side] periodic seed fields (SEED_PRIMITIVES order), unit variance."""\n    generator = torch.Generator().manual_seed(seed_value)\n    bands = seed_fields(torch.randn(1, SEED_ORIENTATIONS, side, side, generator=generator),\n                        laws or SEED_LAWS_P2)[0]\n    area = side*side\n    channels = list(bands)\n    for density, sigma in ((1/45, .7), (1/400, 1.6)):\n        count = int(area*density)\n        xs, ys = torch.rand(count, generator=generator)*side, torch.rand(count, generator=generator)*side\n        values = torch.where(torch.rand(count, generator=generator) < .7, -1., 1.)*(.5+torch.rand(count, generator=generator))\n        channels.append(_splat(side, xs, ys, values, sigma))\n    channels.append(_strokes(generator, side, int(area/220), 4, 14, .7))\n    channels.append(_strokes(generator, side, int(area/1400), 14, 44, 1.))\n    for lo, hi in ((5, 12), (14, 36)):\n        n = _torus_noise(generator, side, lo, hi)\n        n = n/n.abs().amax()\n        channels.append((1-n.abs())**6)\n    channels.append(_cells(generator, side, 12))\n    channels.append(_torus_noise(generator, side, 18, 60))\n    out = torch.stack(channels)\n    out = out-out.mean((1, 2), keepdim=True)\n    return out/(out.std((1, 2), keepdim=True)+1e-6)\n\n\ndef seed_steering_kappa(alignment):\n    """Per-pixel concentration so that, on a fully oriented contour, the local\n    texture power over angle has mean cos 2(dtheta) = the measured alignment."""\n    phi = torch.linspace(-math.pi/2, math.pi/2, 721, dtype=torch.float64)[:-1]\n    angles = seed_angles(dtype=torch.float64)\n    windows = torch.softmax(SEED_WINDOW_KAPPA*torch.cos(2*(phi[None]-angles[:, None])), 0)\n    def achieved(kappa):\n        weights = SEED_ORIENTATIONS*torch.softmax(kappa*torch.cos(2*angles), 0)   # contour gradient at 0\n        power = (weights[:, None]*windows).sum(0)\n        return float((power*torch.cos(2*phi)).sum()/power.sum())\n    low, high = 0., 50.\n    for _ in range(60):\n        middle = (low+high)/2\n        low, high = (middle, high) if achieved(middle) < alignment else (low, middle)\n    return (low+high)/2\n\n\nclass SeedTexture(nn.Module):\n    """The law-built seed at x2: oriented colored noise (fixed, anchored to the\n    crop) steered by the distill output\'s contours, scaled by its local detail.\n    No learned weights. Returns luminance levels/255 to add inside the mask."""\n    def __init__(self, laws=SEED_LAWS, side=2*SEED_SIDE, seed_value=SEED_VALUE):\n        super().__init__()\n        self.laws = laws\n        self.outline_distance = int(laws.get("outline_distance", 0))\n        generator = torch.Generator().manual_seed(seed_value)\n        fields = seed_fields(torch.randn(1, SEED_ORIENTATIONS, side, side, generator=generator), laws)\n        self.register_buffer("fields", fields)\n        # The fields are periodic (FFT-built); a 2 px wrap pad lets bilinear sampling\n        # at any wrapped coordinate stay exact (moving seed, contract roi-texture-v3).\n        self.register_buffer("fields_padded", F.pad(fields, (SEED_WRAP,)*4, mode="circular"), persistent=False)\n        total = fields.sum(1, keepdim=True)\n        self.reference_rms = float(torch.sqrt((highpass(total[:, None], 5)[:, 0]**2).mean()))\n        # Steering strength: calibrated on real patches so that the seed reproduces the\n        # measured alignment under the same measurement ("steering_kappa"); the\n        # power-distribution estimate is only a fallback.\n        self.kappa = laws.get("steering_kappa") or seed_steering_kappa(laws["alignment"])\n        angles = seed_angles()\n        self.register_buffer("cos2", torch.cos(2*angles).view(1, -1, 1, 1))\n        self.register_buffer("sin2", torch.sin(2*angles).view(1, -1, 1, 1))\n\n    def crop_fields(self, b, h, w, noise=None):\n        if noise is not None:\n            return seed_fields(noise, self.laws)\n        fields = self.fields\n        if h > fields.shape[-2] or w > fields.shape[-1]:\n            fields = fields.repeat(1, 1, -(-h//fields.shape[-2]), -(-w//fields.shape[-1]))\n        return fields[..., :h, :w].expand(b, -1, -1, -1)\n\n    def sample_fields(self, coords, weights):\n        """Moving seed: coords [B,4,H,W] = two layers of (x, y) positions in the\n        periodic fields (pixel units, any range), advected with the skin by the\n        app; weights [B,2,H,W] cross-fade the layers (power preserving)."""\n        side = self.fields.shape[-1]\n        padded = self.fields_padded.expand(coords.shape[0], -1, -1, -1)\n        total = float(side+2*SEED_WRAP)\n        layers = []\n        for k in range(2):\n            xy = torch.remainder(coords[:, 2*k:2*k+2], float(side))\n            grid = ((xy+SEED_WRAP+.5)/total*2-1).permute(0, 2, 3, 1)\n            layers.append(F.grid_sample(padded, grid, mode="bilinear", padding_mode="border", align_corners=False))\n        w0, w1 = weights[:, 0:1], weights[:, 1:2]\n        return (w0*layers[0]+w1*layers[1])/torch.sqrt(w0*w0+w1*w1+1e-6)\n\n    def forward(self, enhanced, noise=None, coords=None, seed_weights=None, fields=None):\n        b, _, h, w = enhanced.shape\n        y = (.299*enhanced[:, 0:1]+.587*enhanced[:, 1:2]+.114*enhanced[:, 2:3])*255\n        # contour orientation of the distill output (Scharr gradients of a 2 px blur, sigma 3 tensor)\n        smooth = blur(blur(y[:, None], 3), 3)[:, 0]\n        gx = F.conv2d(F.pad(smooth, (1, 1, 1, 1), mode="replicate"), SCHARR_X.to(y))\n        gy = F.conv2d(F.pad(smooth, (1, 1, 1, 1), mode="replicate"), SCHARR_X.to(y).transpose(-1, -2))\n        jxx, jyy, jxy = [blur(blur(v[:, None], 7), 5)[:, 0] for v in (gx*gx, gy*gy, gx*gy)]\n        trace = jxx+jyy+1e-6\n        c2, s2 = (jxx-jyy)/trace, 2*jxy/trace                    # coherence * (cos, sin) 2 theta\n        weights = SEED_ORIENTATIONS*torch.softmax(self.kappa*(c2*self.cos2+s2*self.sin2), 1)\n        # fields given: already sampled by the app at advected coordinates (fp32, exact).\n        if fields is None:\n            fields = self.sample_fields(coords, seed_weights) if coords is not None else self.crop_fields(b, h, w, noise)\n        texture = (torch.sqrt(weights)*fields).sum(1, keepdim=True)\n        # local strength law: missing hp5 RMS = a + b * the distill output\'s hp5 RMS (9 px window)\n        detail = torch.sqrt(blur((highpass(y[:, None], 5)[:, 0])**2, 9).clamp(min=0)+1e-6)\n        a, slope = self.laws["amplitude"]\n        texture = texture*(a+slope*detail)*self.laws.get("amplitude_gain", 1.)/self.reference_rms\n        if self.outline_distance:\n            # fade out within outline_distance px of an outline (edge guard of the distill output)\n            guard = self.outline_guard(enhanced)\n            texture = texture*(1-guard)\n        return texture/255\n\n    def outline_guard(self, enhanced):\n        """1 within outline_distance px of an outline (edge guard .06 on blurred luminance), 0 far away."""\n        luminance = blur(enhanced.mean(1, keepdim=True)[:, None], 3)[:, 0]\n        gx = F.pad(luminance[..., :, 1:]-luminance[..., :, :-1], (0, 1, 0, 0), mode="replicate")\n        gy = F.pad(luminance[..., 1:, :]-luminance[..., :-1, :], (0, 0, 0, 1), mode="replicate")\n        strength = torch.sqrt(gx.square()+gy.square()+1e-12)\n        strength = F.max_pool2d(strength, 3, stride=1, padding=1)\n        outline = (strength/.06).clamp(0, 1)\n        d = self.outline_distance\n        return F.max_pool2d(outline, 2*d+1, stride=1, padding=d)\n\n\nSCHARR_X = torch.tensor([[-3., 0., 3.], [-10., 0., 10.], [-3., 0., 3.]]).view(1, 1, 3, 3)/32\n\n\nclass DistillSeedROI(nn.Module):\n    """restored -> ROI distill (frozen, x2) -> + law-built seed (P1) [+ learned\n    correction (v10)] at x2 -> mask composite. No texture on outlines (edge\n    guard). With learned=False the model has no trainable weights (P1); with\n    learned=True a zero-initialised branch sees the distilled centre, the mask\n    and the seed and adds a correction in the same band, so step 0 equals P1."""\n    def __init__(self, distill, width=32, blocks=6, band=9, edge_guard=.06, learned=False,\n                 laws="roi-texture-laws-v1", amplitude=None, steering_kappa=None, texture_gain=1., **_):\n        super().__init__()\n        self.distill = distill\n        table = {"roi-texture-laws-v1": SEED_LAWS, "roi-texture-laws-v2": SEED_LAWS_P2}[laws]\n        amplitude = table["amplitude"] if amplitude is None else amplitude\n        steering_kappa = table["steering_kappa"] if steering_kappa is None else steering_kappa\n        self.architecture = {"width": width, "blocks": blocks, "band": band, "edge_guard": edge_guard,\n                             "order": "distill-then-seed", "scale": 2, "learned": bool(learned),\n                             "laws": laws, "amplitude": [float(x) for x in amplitude],\n                             "steering_kappa": float(steering_kappa)}\n        if texture_gain != 1:\n            self.architecture["texture_gain"] = float(texture_gain)\n        self.seed = SeedTexture({**table, "amplitude": tuple(amplitude), "steering_kappa": steering_kappa,\n                                 "amplitude_gain": table.get("amplitude_gain", 1.)*texture_gain})\n        if learned:\n            self.texture = nn.Sequential(\n                nn.Conv2d(5, width, 3, padding=1), nn.ReLU(),\n                *[base.ResidualBlock(width) for _ in range(blocks)],\n                nn.Conv2d(width, 3, 3, padding=1))\n            nn.init.zeros_(self.texture[-1].weight)\n            nn.init.zeros_(self.texture[-1].bias)\n\n    def training_noise(self, b, size, device):\n        return torch.randn(b, SEED_ORIENTATIONS, 2*size, 2*size, device=device)\n\n    def forward(self, restored, mosaic, masks, noise=None, seed_coords=None, seed_weights=None, seed_fields=None):\n        b, _, _, h, w = restored.shape\n        enhanced = self.distill(restored.reshape(b, 15, h, w))\n\n        def up(x):\n            return F.interpolate(x, scale_factor=2., mode="bilinear", align_corners=False)\n        under, mask = up(restored[:, 2]), up(masks[:, 2])\n        seed = self.seed(enhanced, noise, seed_coords, seed_weights, seed_fields)\n        detail = seed.expand(-1, 3, -1, -1)\n        if self.architecture["learned"]:\n            correction = self.texture(torch.cat((enhanced, mask, seed*255/8), 1))\n            detail = detail+highpass(correction[:, None], self.architecture["band"])[:, 0]\n        if self.architecture["edge_guard"]:\n            detail = detail*(1-DistillTextureROI.edges(self, enhanced))\n        coarse = under+mask*(enhanced-under)\n        return coarse+mask*detail, coarse\n\n\n# v12 seed: small stamps (10 px across at the x2 output) of 16 shapes scattered at\n# random positions, rotations, polarities and strengths, one shape per channel.\n# Periodic (512 torus), zero mean, unit variance per channel. Anchored to the mask\n# by the caller, so each stamp keeps its place relative to the mask.\nSEED_STAMPS = ("dot", "dot_soft", "pore", "ring", "stroke", "stroke_thick", "arc", "crease",\n               "cross", "chevron", "cluster", "blob_long", "box", "crescent", "comma", "speck")\nSTAMP_RADIUS, STAMP_SPACING = 5., 22.\n\n\ndef _stamp(kind, u, v, generator):\n    """One stamp in its own rotated frame (u along, v across), radius 5 px."""\n    def g(d2, s):\n        return torch.exp(-d2/(2*s*s))\n\n    def segment(u0, v0, u1, v1, w):\n        du, dv = u1-u0, v1-v0\n        t = (((u-u0)*du+(v-v0)*dv)/(du*du+dv*dv+1e-9)).clamp(0, 1)\n        return g((u-u0-t*du)**2+(v-v0-t*dv)**2, w)\n    r2 = u*u+v*v\n    r = torch.sqrt(r2)\n    if kind == "dot":\n        return g(r2, 1.2)\n    if kind == "dot_soft":\n        return g(r2, 2.2)\n    if kind == "pore":\n        return g(r2, 1.3)-.45*g(r2, 3.)\n    if kind == "ring":\n        return g((r-3.2)**2, .7)\n    if kind == "stroke":\n        return segment(-4, 0, 4, 0, .7)\n    if kind == "stroke_thick":\n        return segment(-4.5, 0, 4.5, 0, 1.3)\n    if kind == "arc":\n        return g((torch.sqrt(u*u+(v-6)**2)-6)**2, .7)*(v < 4).float()\n    if kind == "crease":\n        return segment(-4, -1.5, 4, -1.5, .6)+segment(-4, 1.5, 4, 1.5, .6)\n    if kind == "cross":\n        return torch.maximum(segment(-4, 0, 4, 0, .7), segment(0, -4, 0, 4, .7))\n    if kind == "chevron":\n        return torch.maximum(segment(-3.5, -3, 0, 2, .7), segment(0, 2, 3.5, -3, .7))\n    if kind == "cluster":\n        pts = (torch.rand(3, 2, generator=generator)-.5)*6\n        return sum(g((u-a)**2+(v-b)**2, .8) for a, b in pts.tolist())\n    if kind == "blob_long":\n        return g(u*u/(3.5**2)+v*v/(1.2**2), 1.)\n    if kind == "box":\n        d = torch.maximum(u.abs(), v.abs())\n        return g((d-3.5)**2, .6)\n    if kind == "crescent":\n        return (g(r2, 2.8)-g((u-1.6)**2+v*v, 2.4)).clamp(min=0)*2\n    if kind == "comma":\n        return g(r2, 1.4)+segment(0, 0, 3.5, 3, .6)\n    if kind == "speck":\n        return torch.randn(u.shape, generator=generator)*g(r2, 2.2)\n    raise ValueError(kind)\n\n\ndef seed_stamps(seed_value=SEED_VALUE, side=2*SEED_SIDE, radius=STAMP_RADIUS, spacing=STAMP_SPACING,\n                max_scale=1.):\n    """[16, side, side] periodic fields (SEED_STAMPS order): each channel holds\n    stamps of one shape, 10 px across (x a log-uniform scale up to max_scale, v12b:\n    10-30 px), random place/rotation/sign; spacing grows with the mean size."""\n    generator = torch.Generator().manual_seed(seed_value)\n    mean_scale = 1. if max_scale == 1 else (max_scale-1)/math.log(max_scale)\n    count = int(side*side/(spacing*mean_scale)**2)\n    channels = []\n    for kind in SEED_STAMPS:\n        out = torch.zeros(side, side)\n        for _ in range(count):\n            cx, cy = (torch.rand(2, generator=generator)*side).tolist()\n            angle = float(torch.rand(1, generator=generator))*2*math.pi\n            value = (.5+float(torch.rand(1, generator=generator)))*(1. if float(torch.rand(1, generator=generator)) < .5 else -1.)\n            scale = math.exp(float(torch.rand(1, generator=generator))*math.log(max_scale)) if max_scale > 1 else 1.\n            r = radius*scale\n            reach = int(math.ceil(r))+1\n            offsets = torch.arange(-reach, reach+1).float()\n            ix, iy = int(math.floor(cx)), int(math.floor(cy))\n            X = (ix+offsets)[None, :]-cx\n            Y = (iy+offsets)[:, None]-cy\n            X, Y = X.expand(len(offsets), -1), Y.expand(-1, len(offsets))\n            ca, sa = math.cos(angle), math.sin(angle)\n            u, v = (X*ca+Y*sa)/scale, (-X*sa+Y*ca)/scale\n            window = ((r+.5-torch.sqrt(X*X+Y*Y)).clamp(0, 1))           # disc, antialiased\n            stamp = _stamp(kind, u, v, generator)*window*value\n            rows, cols = (iy+offsets).long() % side, (ix+offsets).long() % side\n            out[rows[:, None], cols[None, :]] += stamp\n        channels.append(out)\n    out = torch.stack(channels)\n    out = out-out.mean((1, 2), keepdim=True)\n    return out/out.std((1, 2), keepdim=True).clamp_min(1e-6)\n\n\nSEED_SOURCES = {"fields": (seed_primitives, SEED_PRIMITIVES), "stamps10": (seed_stamps, SEED_STAMPS),\n                "stamps10-30": (lambda **k: seed_stamps(max_scale=3., **k), SEED_STAMPS)}\n\n\n# ---------------------------------------------------------------------------\n# v11: texture type from the bank, seed only inside the generator.\nCODEBOOK_VERSION = "roi-texture-codebook-v1"\nCODEBOOK_LAYERS = (8, 15)            # VGG16 relu2_2, relu3_3 of the texture high band\nCODEBOOK_TYPES, CODEBOOK_DIM = 24, 32\n\n\ndef texture_descriptor(features, image, mask, band=9):\n    """Texture-only description of masked crops: masked mean and log-std of VGG\n    relu2_2 / relu3_3 on the high band (x4 around grey), so colour and shading\n    do not decide the type. image [N,3,H,W] 0..1, mask [N,1,H,W] -> [N,768]."""\n    x = (highpass(image.float()[:, None], band)[:, 0]*4+.5).clamp(0, 1)\n    mean = x.new_tensor((.485, .456, .406)).view(1, 3, 1, 1)\n    std = x.new_tensor((.229, .224, .225)).view(1, 3, 1, 1)\n    x, out = (x-mean)/std, []\n    for i, layer in enumerate(features):\n        x = layer(x)\n        if i in CODEBOOK_LAYERS:\n            m = F.interpolate(mask.float(), size=x.shape[-2:], mode="area")\n            total = m.sum((-2, -1)).clamp_min(1.)\n            mu = (x*m).sum((-2, -1))/total\n            var = ((x-mu[..., None, None])**2*m).sum((-2, -1))/total\n            out += [mu, .5*torch.log(var+1e-4)]\n        if i == max(CODEBOOK_LAYERS):\n            return torch.cat(out, 1)\n    raise ValueError("VGG特徴が足りません。")\n\n\ndef build_texture_codebook(bank, features, device, path, types=CODEBOOK_TYPES, dim=CODEBOOK_DIM, seed=20261007):\n    """Types of real texture: bank descriptors -> standardised -> PCA(dim) ->\n    k-means(types). Saved once next to the bank and reused."""\n    path = Path(path)\n    if path.is_file():\n        book = torch.load(path, map_location="cpu", weights_only=True)\n        if book.get("version") == CODEBOOK_VERSION and book.get("bank") == bank.digest:\n            return book\n        raise ValueError("質感の種類の辞書が別の質感バンクのものです。")\n    rows = []\n    with torch.inference_mode():\n        for start in range(0, len(bank), 256):\n            tex = torch.from_numpy(bank.texture[start:start+256]).to(device).permute(0, 3, 1, 2).float()/255\n            msk = torch.from_numpy(bank.mask[start:start+256]).to(device)[:, None].float()/255\n            rows.append(texture_descriptor(features, tex, msk).cpu())\n    d = torch.cat(rows)\n    mean, scale = d.mean(0), d.std(0)+1e-6\n    z = (d-mean)/scale\n    _, _, v = torch.linalg.svd(z-z.mean(0), full_matrices=False)\n    basis = v[:dim].T\n    y = z@basis\n    generator = torch.Generator().manual_seed(seed)\n    centres = y[torch.randint(len(y), (1,), generator=generator)]\n    for _ in range(types-1):                     # k-means++\n        d2 = torch.cdist(y, centres).min(1).values**2\n        centres = torch.cat((centres, y[torch.multinomial(d2/d2.sum(), 1, generator=generator)]))\n    for _ in range(60):\n        assign = torch.cdist(y, centres).argmin(1)\n        centres = torch.stack([y[assign == k].mean(0) if (assign == k).any() else centres[k] for k in range(types)])\n    dist2 = torch.cdist(y, centres)**2\n    nearest = dist2.min(1).values\n    book = {"version": CODEBOOK_VERSION, "bank": bank.digest, "mean": mean, "scale": scale, "basis": basis,\n            "centres": centres, "temperature": float(nearest.median())+1e-6,\n            "sizes": torch.bincount(dist2.argmin(1), minlength=types)}\n    path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(book, path)\n    return book\n\n\ndef texture_type_targets(book, descriptors):\n    """Soft type assignment of real texture descriptors [N,768] -> [N,types]."""\n    y = ((descriptors-book["mean"].to(descriptors))/book["scale"].to(descriptors))@book["basis"].to(descriptors)\n    d2 = torch.cdist(y, book["centres"].to(descriptors))**2\n    return torch.softmax(-d2/book["temperature"], 1)\n\n\nclass SeededBlock(nn.Module):\n    """Residual block: the seed is injected into the features (placement only),\n    the texture type modulates the activations (FiLM)."""\n    def __init__(self, width, seed_channels, type_dim):\n        super().__init__()\n        self.inject = nn.Conv2d(seed_channels, width, 1, bias=False)\n        nn.init.normal_(self.inject.weight, std=.05)\n        self.film = nn.Conv2d(type_dim, 2*width, 1)\n        nn.init.zeros_(self.film.weight)\n        nn.init.zeros_(self.film.bias)\n        self.conv1 = nn.Conv2d(width, width, 3, padding=1)\n        self.conv2 = nn.Conv2d(width, width, 3, padding=1)\n\n    def forward(self, features, seed, types):\n        gamma, beta = self.film(types).chunk(2, 1)\n        h = F.relu(self.conv1(features+self.inject(seed)))\n        return features+.1*self.conv2(h*(1+gamma)+beta)\n\n\nclass DistillSeededTexture(nn.Module):\n    """v11: restored -> ROI distill (frozen, x2) -> texture generator at x2 ->\n    mask composite. The generator sees the distilled centre and the mask. A\n    small head predicts the texture type per pixel (types = clusters of real\n    texture in the bank); the type embedding modulates every block. The moving\n    seed (16 primitive shapes) is injected inside every block and never reaches\n    the output directly: the bank-defined type decides what the texture is, the\n    seed only where its details fall. Zero output layer: step 0 = distill alone."""\n    def __init__(self, distill, width=48, blocks=6, band=9, edge_guard=.06, types=CODEBOOK_TYPES,\n                 type_dim=CODEBOOK_DIM, seed_channels=len(SEED_PRIMITIVES), seed_motion="moving",\n                 seed_shapes="fields", outline_protect=0, outline_fade=0, **_):\n        super().__init__()\n        self.distill = distill\n        self.architecture = {"width": width, "blocks": blocks, "band": band, "edge_guard": edge_guard,\n                             "order": "distill-seeded-texture", "scale": 2, "types": types,\n                             "type_dim": type_dim, "seed_channels": seed_channels,\n                             "seed_primitives": list(SEED_SOURCES[seed_shapes][1])}\n        if seed_shapes != "fields":     # v12: scattered 10 px stamps\n            self.architecture["seed_shapes"] = seed_shapes\n        if outline_protect:             # outlines are absolute: no texture within this many px\n            self.architecture["outline_protect"] = int(outline_protect)\n            self.architecture["outline_fade"] = int(outline_fade)\n        if seed_motion != "moving":     # "fixed": crop coordinates; "mask": anchored to the mask (v12)\n            self.architecture["seed_motion"] = seed_motion\n        self.typer = nn.Sequential(\n            nn.Conv2d(4, 32, 3, stride=2, padding=1), nn.ReLU(),\n            nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.ReLU(),\n            nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.ReLU(),\n            nn.Conv2d(32, types, 1))\n        self.type_embed = nn.Conv2d(types, type_dim, 1, bias=False)\n        self.stem = nn.Conv2d(4, width, 3, padding=1)\n        self.blocks = nn.ModuleList([SeededBlock(width, seed_channels, type_dim) for _ in range(blocks)])\n        self.head = nn.Conv2d(width, 3, 3, padding=1)\n        nn.init.zeros_(self.head.weight)\n        nn.init.zeros_(self.head.bias)\n        # The app\'s seed source (also used for evaluation): periodic 512 px fields.\n        self.register_buffer("seed_base", SEED_SOURCES[seed_shapes][0]()[None].to(torch.float16))\n        self.last_type_logits = None\n\n    def fixed_seed(self, b, h, w):\n        base = self.seed_base\n        if h > base.shape[-2] or w > base.shape[-1]:\n            base = base.repeat(1, 1, -(-h//base.shape[-2]), -(-w//base.shape[-1]))\n        return base[..., :h, :w].expand(b, -1, -1, -1).float()\n\n    def forward(self, restored, mosaic, masks, noise=None, seed_fields=None, type_probs=None):\n        b, _, _, h, w = restored.shape\n        enhanced = self.distill(restored.reshape(b, 15, h, w))\n\n        def up(x):\n            return F.interpolate(x, scale_factor=2., mode="bilinear", align_corners=False)\n        under, mask = up(restored[:, 2]), up(masks[:, 2])\n        x = torch.cat((enhanced, mask), 1)\n        logits = self.typer(x)\n        self.last_type_logits = logits\n        if type_probs is None:\n            probs = F.interpolate(torch.softmax(logits, 1), size=enhanced.shape[-2:], mode="bilinear",\n                                  align_corners=False)\n        else:   # teacher types [B,types] (training) broadcast over the crop\n            probs = type_probs[..., None, None].expand(-1, -1, *enhanced.shape[-2:])\n        types = self.type_embed(probs)\n        seed = seed_fields if seed_fields is not None else self.fixed_seed(b, *enhanced.shape[-2:])\n        if self.architecture.get("seed_motion") == "mask":\n            seed = seed*mask             # v12: the seed exists only inside the mask\n        features = F.relu(self.stem(x))\n        for block in self.blocks:\n            features = block(features, seed.to(features.dtype), types)\n        detail = highpass(self.head(features)[:, None], self.architecture["band"])[:, 0]\n        if self.architecture["edge_guard"]:\n            detail = detail*(1-DistillTextureROI.edges(self, enhanced))\n        if self.architecture.get("outline_protect"):\n            detail = detail*(1-outline_protection(enhanced, self.architecture["outline_protect"],\n                                                  self.architecture["outline_fade"]))\n        coarse = under+mask*(enhanced-under)\n        return coarse+mask*detail, coarse\n\n\ndef mask_fade(mask, width):\n    """0 at the mask border rising to 1 about `width` px inside (erode by width/2, then a\n    width-px box), on a half-resolution grid; keeps added detail from ending in a seam."""\n    cell = 2 if width < 32 else 4      # wide ramps on a coarser grid (small kernels on the Neural Engine)\n    small = F.avg_pool2d(mask, cell, stride=cell)\n    r = max(1, width//(2*cell))\n    eroded = -F.max_pool2d(F.pad(-small, (r,)*4, value=-1.), 2*r+1, stride=1)\n    k = max(3, (width//cell) | 1)\n    soft = F.avg_pool2d(F.pad(eroded, (k//2,)*4, mode="replicate"), k, stride=1)\n    return F.interpolate(soft, scale_factor=float(cell), mode="bilinear", align_corners=False).clamp(0, 1)\n\n\nOUTLINE_THRESHOLD = .04   # luminance step per px (after a 3 px blur) that counts fully as an outline\n\n\ndef outline_protection(enhanced, protect, fade, cell=2):\n    """1 within `protect` px of an outline of the distill output, falling to 0 over\n    the next `fade` px (outlines are never textured; no step at the guard\'s edge).\n    Computed on a half-resolution grid (max pools stay small for the Neural\n    Engine) and brought back with bilinear upsampling."""\n    luminance = blur(enhanced.mean(1, keepdim=True)[:, None], 3)[:, 0]\n    gx = F.pad(luminance[..., :, 1:]-luminance[..., :, :-1], (0, 1, 0, 0), mode="replicate")\n    gy = F.pad(luminance[..., 1:, :]-luminance[..., :-1, :], (0, 0, 0, 1), mode="replicate")\n    # Only real outlines: 0 below half the threshold (soft shading), 1 from the threshold up.\n    outline = (torch.sqrt(gx.square()+gy.square()+1e-12)/OUTLINE_THRESHOLD*2-1).clamp(0, 1)\n    # cell 2 (fine texture guard) or 4 (v13d relief guard: wide radii stay small kernels)\n    half = F.max_pool2d(outline, cell, stride=cell)\n    core_r = (protect+cell-1)//cell+1               # coarse-grid radius covering `protect` px plus the pooling cell\n    core = F.max_pool2d(F.pad(half, (core_r,)*4, mode="replicate"), 2*core_r+1, stride=1)\n    fade_r = max(1, (fade+cell-1)//cell)\n    # linear fall-off: mean of the dilated core over a (2*fade_r+1) box, kept at 1 inside the core\n    soft = F.avg_pool2d(F.pad(core, (fade_r,)*4, mode="replicate"), 2*fade_r+1, stride=1)\n    soft = (2*soft).clamp(max=1.)                   # the box mean is .5 at the core boundary\n    guard = torch.maximum(core, soft)\n    return F.interpolate(guard, scale_factor=float(cell), mode="bilinear", align_corners=False).clamp(0, 1)\n\n\nclass ContextBlock(nn.Module):\n    """Residual block on the coarse grid, modulated by the texture type and the\n    masked global summary of the whole ROI."""\n    def __init__(self, width, type_dim):\n        super().__init__()\n        self.film = nn.Conv2d(type_dim, 2*width, 1)\n        nn.init.zeros_(self.film.weight)\n        nn.init.zeros_(self.film.bias)\n        self.conv1 = nn.Conv2d(width, width, 3, padding=1)\n        self.conv2 = nn.Conv2d(width, width, 3, padding=1)\n\n    def forward(self, x, types):\n        gamma, beta = self.film(types).chunk(2, 1)\n        return x+.1*self.conv2(F.relu(self.conv1(x))*(1+gamma)+beta)\n\n\nclass PlainBlock(nn.Module):\n    """Ordinary residual block used by the clean deterministic v17 U-Net."""\n    def __init__(self, width):\n        super().__init__()\n        self.conv1 = nn.Conv2d(width, width, 3, padding=1)\n        self.conv2 = nn.Conv2d(width, width, 3, padding=1)\n\n    def forward(self, x):\n        return x+.1*self.conv2(F.relu(self.conv1(x)))\n\n\nclass DistillContextTexture(nn.Module):\n    """v13: restored -> ROI distill (frozen, x2) -> U-Net texture generator that sees\n    the whole 512 px ROI (encoder to 32x32 + masked global summary) -> 33 px band ->\n    edge guard + outline protection -> mask composite. The mask-anchored seed is\n    injected inside the two finest decoder levels only; the bank texture type (head\n    on the 32x32 grid) modulates the decoder. Zero head: step 0 = distill alone."""\n    def __init__(self, distill, widths=(24, 32, 48, 64, 96), mid_blocks=4, band=33, edge_guard=.06,\n                 types=CODEBOOK_TYPES, type_dim=CODEBOOK_DIM, seed_channels=16, seed_motion="mask",\n                 seed_shapes="stamps10-30", outline_protect=6, outline_fade=10, fine_gain_band=0,\n                 mid_gain=1., mid_outline_protect=0, mid_outline_fade=0, head_bound=0., relief_mode="",\n                 relief_gain_range=.5, mask_fade=0, line_mode="", line_orientations=8, line_length=15,\n                 line_sharpness=4., orient_window=17, line_width=.7, line_grid=1, line_outline_protect=0,\n                 line_outline_fade=0, relief_gain_low=0., relief_gain_high=0., relief_outline_protect=0,\n                 relief_outline_fade=0, detail_low_cut=0, seed_blur=0, seed_top_level=True, seed_inject=True,\n                 masked_bandpass=False, zero_mean_band=0, luma_detail=False, judge_band=0, relief_band=0,\n                 masked_relief=False, plain_unet=False, **_):\n        super().__init__()\n        self.distill = distill\n        widths = [int(w) for w in widths]\n        self.architecture = {"widths": widths, "mid_blocks": mid_blocks, "band": band, "edge_guard": edge_guard,\n                             "order": "distill-context-texture", "scale": 2,\n                             "outline_protect": int(outline_protect), "outline_fade": int(outline_fade)}\n        if plain_unet:\n            self.architecture["plain_unet"] = True\n        else:\n            self.architecture.update(types=types, type_dim=type_dim, seed_channels=seed_channels,\n                                     seed_motion=seed_motion, seed_shapes=seed_shapes,\n                                     seed_primitives=list(SEED_SOURCES[seed_shapes][1]))\n        if fine_gain_band:      # v13c: the texture goal gain applies only below this many px\n            self.architecture["fine_gain_band"] = int(fine_gain_band)\n        if mid_gain != 1:       # v13d: goal gain of the relief between fine_gain_band and band\n            self.architecture["mid_gain"] = float(mid_gain)\n        if head_bound:          # v13d: bounded head output (fp16-safe band averages)\n            self.architecture["head_bound"] = float(head_bound)\n        if seed_blur:           # seed only as smooth placement variation (not used by v14b)\n            self.architecture.update(seed_blur=int(seed_blur), seed_top_level=bool(seed_top_level))\n        if not seed_inject:     # v14b: texture from the input alone; the seed input is accepted, unused\n            self.architecture["seed_inject"] = False\n        if masked_bandpass:     # v14c: band-pass with mask-weighted blurs (no ring at the mask border)\n            self.architecture["masked_bandpass"] = True\n        if zero_mean_band:      # v14d: the added detail has no local mean (no dark/bright band)\n            self.architecture["zero_mean_band"] = int(zero_mean_band)\n        if luma_detail:         # v14e: one grey detail for R, G and B (no colour fringes)\n            self.architecture["luma_detail"] = True\n        if judge_band:          # v14e: training judges (and the relief reaches) up to this many px\n            self.architecture["judge_band"] = int(judge_band)\n        if relief_band:         # v14f: the relief\'s reach when training judges a narrower band\n            self.architecture["relief_band"] = int(relief_band)\n        if masked_relief:       # v14f: relief from mask-weighted blurs (no seam at the mask border)\n            self.architecture["masked_relief"] = True\n        if detail_low_cut:      # v14: band-pass detail (no pixel-level grain below this many px)\n            self.architecture["detail_low_cut"] = int(detail_low_cut)\n        if mask_fade:           # v13e: added detail ramps in from the mask border over this many px\n            self.architecture["mask_fade"] = int(mask_fade)\n        if relief_mode:         # v13e: the 17-65 px relief is the distill\'s own, scaled by a gain map\n            self.architecture["relief_mode"] = relief_mode\n            self.architecture["relief_gain_range"] = float(relief_gain_range)\n            if relief_gain_low or relief_gain_high:   # v13i: asymmetric gain range (1-low .. 1+high)\n                self.architecture.update(relief_gain_low=float(relief_gain_low), relief_gain_high=float(relief_gain_high))\n            if relief_outline_protect:                # v13i: gain 1 near strong outlines (no halo)\n                self.architecture.update(relief_outline_protect=int(relief_outline_protect),\n                                         relief_outline_fade=int(relief_outline_fade))\n        if mid_outline_protect:  # v13d: the relief keeps further away from outlines than fine texture\n            self.architecture["mid_outline_protect"] = int(mid_outline_protect)\n            self.architecture["mid_outline_fade"] = int(mid_outline_fade)\n        if line_mode:           # v13f: fine texture as lines along the distill\'s shape direction\n            self.architecture.update(line_mode=line_mode, line_orientations=int(line_orientations),\n                                     line_length=int(line_length), line_sharpness=float(line_sharpness),\n                                     orient_window=int(orient_window))\n            if line_grid != 1:  # v13g: steering on a coarser grid (wrinkle-scale strokes, cheaper)\n                self.architecture.update(line_width=float(line_width), line_grid=int(line_grid))\n            if line_outline_protect:   # v13h: lines keep off outlines (a parallel stroke = double contour)\n                self.architecture.update(line_outline_protect=int(line_outline_protect),\n                                         line_outline_fade=int(line_outline_fade))\n            self.register_buffer("line_bank", line_kernels(int(line_orientations), int(line_length), float(line_width)))\n            phi = torch.arange(int(line_orientations))*math.pi/int(line_orientations)\n            # logit_k = κ cos 2(φ_k - (θ+90°)) = -κ (c2 cos 2φ_k + s2 sin 2φ_k), (c2, s2) from flow_tensor\n            self.register_buffer("line_steer", -float(line_sharpness)*torch.stack(\n                (torch.cos(2*phi), torch.sin(2*phi)), 1)[..., None, None])\n        self.enc = nn.ModuleList([nn.Sequential(nn.Conv2d(6 if line_mode else 4, widths[0], 3, padding=1), nn.ReLU(),\n                                                nn.Conv2d(widths[0], widths[0], 3, padding=1), nn.ReLU())])\n        for a, b in zip(widths[:-1], widths[1:]):\n            self.enc.append(nn.Sequential(nn.Conv2d(a, b, 3, stride=2, padding=1), nn.ReLU(),\n                                          nn.Conv2d(b, b, 3, padding=1), nn.ReLU()))\n        if plain_unet:\n            self.mid = nn.ModuleList([PlainBlock(widths[-1]) for _ in range(mid_blocks)])\n        else:\n            self.summary = nn.Linear(widths[-1], type_dim)\n            self.mid = nn.ModuleList([ContextBlock(widths[-1], type_dim) for _ in range(mid_blocks)])\n            self.typer = nn.Conv2d(widths[-1], types, 1)\n            self.type_embed = nn.Conv2d(types, type_dim, 1, bias=False)\n        self.dec = nn.ModuleList()\n        if not plain_unet:\n            self.dec_film = nn.ModuleList()\n        for a, b in zip(widths[::-1][:-1], widths[::-1][1:]):      # 96->64, 64->48, 48->32, 32->24\n            self.dec.append(nn.Sequential(nn.Conv2d(a+b, b, 3, padding=1), nn.ReLU(),\n                                          nn.Conv2d(b, b, 3, padding=1)))\n            if not plain_unet:\n                film = nn.Conv2d(type_dim, 2*b, 1)\n                nn.init.zeros_(film.weight)\n                nn.init.zeros_(film.bias)\n                self.dec_film.append(film)\n        if not plain_unet:\n            self.inject = nn.ModuleList([nn.Conv2d(seed_channels, widths[1], 1, bias=False),\n                                         nn.Conv2d(seed_channels, widths[0], 1, bias=False)])\n            for conv in self.inject:\n                nn.init.normal_(conv.weight, std=.05)\n        self.head = nn.Conv2d(widths[0], 1 if line_mode else 3, 3, padding=1)   # v13f: luminance only\n        nn.init.zeros_(self.head.weight)\n        nn.init.zeros_(self.head.bias)\n        if relief_mode == "gain":   # gain logits on the 64 px decoder grid (smooth once upsampled)\n            self.relief_gain = nn.Conv2d(widths[3], 1, 1)\n            nn.init.zeros_(self.relief_gain.weight)\n            nn.init.zeros_(self.relief_gain.bias)\n        if not plain_unet:\n            self.register_buffer("seed_base", SEED_SOURCES[seed_shapes][0]()[None].to(torch.float16))\n        self.last_type_logits = None\n\n    def forward(self, restored, mosaic, masks, noise=None, seed_fields=None, type_probs=None):\n        b, _, _, h, w = restored.shape\n        enhanced = self.distill(restored.reshape(b, 15, h, w))\n\n        def up(x):\n            return F.interpolate(x, scale_factor=2., mode="bilinear", align_corners=False)\n        under, mask = up(restored[:, 2]), up(masks[:, 2])\n        line_mode = self.architecture.get("line_mode")\n        if line_mode:   # v13f: where the shape runs (from the distill\'s shading), given to the generator\n            orient = flow_tensor(.299*enhanced[:, 0:1]+.587*enhanced[:, 1:2]+.114*enhanced[:, 2:3],\n                                 self.architecture["orient_window"])\n            x = torch.cat((enhanced, mask, orient), 1)\n        else:\n            x = torch.cat((enhanced, mask), 1)\n        skips = []\n        for i, layer in enumerate(self.enc):\n            x = layer(x)\n            skips.append(x)\n        plain_unet = self.architecture.get("plain_unet", False)\n        if plain_unet:\n            for block in self.mid:\n                x = block(x)\n        else:\n            coarse_mask = F.adaptive_avg_pool2d(mask, x.shape[-2:])\n            summary = (x*coarse_mask).sum((-2, -1))/coarse_mask.sum((-2, -1)).clamp_min(1.)\n            context = self.summary(summary)[..., None, None]\n            logits = self.typer(x)\n            self.last_type_logits = logits\n            if type_probs is None:\n                probs = torch.softmax(logits, 1)\n            else:   # teacher types [B,types] (training) broadcast over the grid\n                probs = type_probs[..., None, None].expand(-1, -1, *x.shape[-2:])\n            types = self.type_embed(probs)+context\n            for block in self.mid:\n                x = block(x, types)\n        if not plain_unet and self.architecture.get("seed_inject", True):\n            seed = seed_fields if seed_fields is not None else self.seed_base[..., :2*h, :2*w].expand(b, -1, -1, -1).float()\n            if self.architecture.get("seed_blur"):\n                k = self.architecture["seed_blur"]\n                seed = F.interpolate(F.avg_pool2d(seed.to(x.dtype), k), scale_factor=float(k), mode="bilinear", align_corners=False)\n            seed = (seed*mask).to(x.dtype)                  # the seed exists only inside the mask\n        levels = len(self.dec)\n        for i, layer in enumerate(self.dec):\n            skip = skips[-2-i]\n            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)\n            x = layer(torch.cat((x, skip), 1))\n            if not plain_unet:\n                t = F.interpolate(types, size=x.shape[-2:], mode="bilinear", align_corners=False)\n                gamma, beta = self.dec_film[i](t).chunk(2, 1)\n                x = x*(1+gamma)+beta\n            if (not plain_unet and self.architecture.get("seed_inject", True) and i >= levels-2\n                    and (i == levels-2 or self.architecture.get("seed_top_level", True))):   # placement only\n                s = seed if i == levels-1 else F.avg_pool2d(seed, 2)\n                x = x+self.inject[i-(levels-2)](s)\n            x = F.relu(x)\n            if i == 0:\n                relief_features = x                     # 64 px grid\n        head = self.head(x)\n        if self.architecture.get("head_bound"):\n            head = self.architecture["head_bound"]*torch.tanh(head/self.architecture["head_bound"])\n        if self.architecture.get("luma_detail"):\n            # Grey detail: per-channel heads drew thin rainbow streaks along outlines (v14d 13000).\n            head = head.mean(1, keepdim=True).expand(-1, 3, -1, -1)\n        if line_mode:\n            # Smear the head along the shape direction (soft choice among K line kernels), so the\n            # fine texture becomes lines parallel to the shape; where the shape has no direction\n            # (coherence ~0) the raw head stays.\n            r = self.architecture["line_length"]//2\n            grid = self.architecture.get("line_grid", 1)\n            source, steer = (head, orient) if grid == 1 else (F.avg_pool2d(head, grid), F.avg_pool2d(orient, grid))\n            lines = F.conv2d(F.pad(source, (r,)*4, mode="reflect"), self.line_bank.to(head.dtype))\n            weights = torch.softmax(F.conv2d(steer, self.line_steer.to(orient.dtype)), 1)\n            steered = (weights*lines).sum(1, keepdim=True)\n            if grid != 1:\n                steered = F.interpolate(steered, scale_factor=float(grid), mode="bilinear", align_corners=False)\n            coherence = (orient.square().sum(1, keepdim=True)+1e-6).sqrt().clamp(max=1.)\n            if self.architecture.get("line_outline_protect"):\n                # v13h: near outlines only the direction-free share stays (no strokes parallel to them).\n                far = 1-outline_protection(enhanced, self.architecture["line_outline_protect"],\n                                           self.architecture["line_outline_fade"], cell=4)\n                head = coherence*far*steered+(1-coherence)*head\n            else:\n                head = coherence*steered+(1-coherence)*head\n            head = head.expand(-1, 3, -1, -1)\n        head = head[:, None]\n        fade = mask_fade(mask, self.architecture["mask_fade"]) if self.architecture.get("mask_fade") else None\n        if self.architecture.get("masked_bandpass"):\n            w = (fade if fade is not None else mask_fade(mask, 16))[:, None]\n\n            def inside_blur(x, side):     # blur of x over the mask only\n                return blur(x*w, side)/(blur(w, side)+1e-3)\n            low_cut = self.architecture.get("detail_low_cut", 1)\n            detail = ((inside_blur(head, low_cut) if low_cut > 1 else head)\n                      - inside_blur(head, self.architecture["band"]))[:, 0]\n        else:\n            detail = highpass(head, self.architecture["band"])[:, 0]\n            if self.architecture.get("detail_low_cut"):\n                detail = detail-highpass(head, self.architecture["detail_low_cut"])[:, 0]\n        if self.architecture["edge_guard"]:\n            detail = detail*(1-DistillTextureROI.edges(self, enhanced))\n        near = 1-outline_protection(enhanced, self.architecture["outline_protect"], self.architecture["outline_fade"])\n        relief_part, detail_fade = None, None\n        if self.architecture.get("relief_mode") == "gain" and self.architecture.get("masked_bandpass"):\n            # v14e: v14d\'s generated 5-17 px detail (6 + 8 px off outlines) plus v13e\'s relief: the\n            # distill\'s own 17-65 px relief scaled in place by a smooth gain map (no new edge can\n            # appear), gain 1 only within the relief guard of outlines. The relief is added after\n            # the zero-mean step (that step would remove it).\n            # Grey relief (as the detail): one channel through the wide blurs (ANE cost /3).\n            shade = enhanced.mean(1, keepdim=True)[:, None]\n            reach = self.architecture.get("relief_band", self.architecture["judge_band"])\n            if self.architecture.get("masked_relief"):\n                # v14f: v14e\'s relief, taken over the whole crop, held the restoration\'s seam at the\n                # mask border and the gain amplified it (+1.5 levels 2-6 px inside, -0.9 at 14-18 px).\n                # Mask-weighted blurs as for the detail: nothing outside the mask enters the relief.\n                relief = (inside_blur(shade, self.architecture["band"]) - inside_blur(shade, reach))[:, 0]\n            else:\n                relief = (highpass(shade, reach) - highpass(shade, self.architecture["band"]))[:, 0]\n            t = torch.tanh(self.relief_gain(relief_features))\n            gain = F.interpolate(self.architecture["relief_gain_range"]*t, size=enhanced.shape[-2:],\n                                 mode="bilinear", align_corners=False)\n            relief_guard = 1-outline_protection(\n                enhanced, self.architecture["relief_outline_protect"],\n                self.architecture["relief_outline_fade"])\n            gain = gain*relief_guard\n            # Export-time, fixed mid-band contrast lift.  `relief` contains only\n            # the 17..reach px component of the already restored image, so flat\n            # areas stay flat and the change grows naturally with visible\n            # surface relief.  Reuse the outline guard to avoid double contours.\n            fixed_sharpen = float(self.architecture.get("relief_sharpen", 0.))\n            relief_part = (gain+fixed_sharpen*relief_guard)*relief\n            # v14e/v15 keep the generated 5..17 px detail in this masked\n            # band-pass branch.  Apply the export-time fine-detail gain here;\n            # the older non-masked relief branch applies the same option below.\n            # Relief and protected outlines are deliberately unaffected.\n            scale = float(self.architecture.get("fine_scale", 1.))\n            if self.architecture.get("inner_fine_scale") and fade is not None:\n                # Export-time: the extra gain follows the detail\'s own border ramp, so it is ~1x where\n                # v14e\'s border band lives (fade ~0) and full from mask_fade px inside. A ramp starting\n                # only at mask_fade px reached almost no pixels (inradius is ~130-180 px of 512).\n                # Applied below as detail += k*(detail*fade) after the zero-mean step: as a gain map\n                # multiplied in before it, the Neural Engine build silently dropped the term (CPU and\n                # GPU builds were right).\n                detail_fade = float(self.architecture["inner_fine_scale"])/scale-1\n            detail = scale*detail*near\n        elif self.architecture.get("relief_mode") == "gain":\n            # v13e: generated fine texture + the distill\'s relief scaled in place (no new edges).\n            fine = highpass(head, self.architecture["fine_gain_band"])[:, 0]\n            if self.architecture["edge_guard"]:\n                fine = fine*(1-DistillTextureROI.edges(self, enhanced))\n            relief = (highpass(enhanced[:, None], self.architecture["band"])\n                      - highpass(enhanced[:, None], self.architecture["fine_gain_band"]))[:, 0]\n            t = torch.tanh(self.relief_gain(relief_features))\n            if self.architecture.get("relief_gain_high"):\n                # v13i: -low at t=-1, 0 at t=0, +high at t=1 (monotone, smooth)\n                lo, hi = self.architecture["relief_gain_low"], self.architecture["relief_gain_high"]\n                gain = (hi+lo)/2*t+(hi-lo)/2*t*t\n            else:\n                gain = self.architecture["relief_gain_range"]*t\n            gain = F.interpolate(gain, size=enhanced.shape[-2:], mode="bilinear", align_corners=False)\n            if self.architecture.get("relief_outline_protect"):\n                gain = gain*(1-outline_protection(enhanced, self.architecture["relief_outline_protect"],\n                                                  self.architecture["relief_outline_fade"], cell=4))\n            # fine_scale: export-time strength of the generated fine texture (1 = as trained).\n            detail = (self.architecture.get("fine_scale", 1.)*fine+gain*relief)*near\n        elif self.architecture.get("mid_outline_protect"):\n            # v13d: fine texture (<= split px) may come close to outlines, the relief may not.\n            fine = highpass(head, self.architecture["fine_gain_band"])[:, 0]\n            if self.architecture["edge_guard"]:\n                fine = fine*(1-DistillTextureROI.edges(self, enhanced))\n            far = 1-outline_protection(enhanced, self.architecture["mid_outline_protect"],\n                                       self.architecture["mid_outline_fade"], cell=4)\n            detail = fine*near+(detail-fine)*far\n        else:\n            detail = detail*near\n        if fade is not None:\n            detail = detail*fade\n            if self.architecture.get("zero_mean_band"):\n                # v14d: remove the local mean inside the mask (no darker/brighter band at the edge)\n                k = self.architecture["zero_mean_band"]\n                detail = detail-fade*blur(detail*mask, k)/(blur(mask, k)+1e-3)\n            if detail_fade is not None:\n                detail = detail+detail_fade*(detail*fade)\n        if relief_part is not None:\n            detail = detail+relief_part*(fade if fade is not None else 1.)\n        coarse = under+mask*(enhanced-under)\n        return coarse+mask*detail, coarse\n\n\ndef fine_structure(features, prediction, reference, mask):\n    """Paired shape term for v13 (what the whole ROI can predict): relative L1 between\n    the aligned relu2_2 and relu3_3 maps of the prediction and of the truth\'s texture\n    on the prediction\'s base, inside the mask. Deep enough to ignore exact pixel phase,\n    fine enough to reward wrinkles/creases where the truth has them."""\n    h, w = prediction.shape[-2:]\n    p, r = prediction.reshape(-1, 3, h, w).float(), reference.reshape(-1, 3, h, w).float()\n    m = mask.reshape(-1, 1, h, w).float()\n    mean = p.new_tensor((.485, .456, .406)).view(1, 3, 1, 1)\n    std = p.new_tensor((.229, .224, .225)).view(1, 3, 1, 1)\n    x = torch.cat(((p-mean)/std, (r-mean)/std))\n    total = p.new_zeros(())\n    for i, layer in enumerate(features):\n        x = layer(x)\n        if i in (8, 15):\n            fp, fr = x.chunk(2)\n            weight = F.interpolate(m, size=fp.shape[-2:], mode="area")\n            total = total+paired.weighted_mean((fp-fr).abs(), weight)/(paired.weighted_mean(fr.abs(), weight)+1e-6)\n        if i == 15:\n            return total/2\n    raise ValueError("VGG特徴が足りません。")\n\n\ndef mask_anchor_shifts(mask_windows, scale=2):\n    """v12 seed anchor: [N,5,1,s,s] masks of N model windows -> [N,2] integer\n    (x, y) shifts in output pixels = floor(mean centroid of the non-empty masks\n    * scale + .5). Centroid in pixel-edge coordinates ((x+.5) per pixel). The app\n    computes the same from the same five masks (ROITextureMaskSeed)."""\n    m = mask_windows.float()[:, :, 0]\n    s = m.shape[-1]\n    centre = torch.arange(s, device=m.device, dtype=torch.float32)+.5\n    total = m.sum((-2, -1))\n    cx = (m.sum(-2)*centre).sum(-1)/total.clamp_min(1e-6)\n    cy = (m.sum(-1)*centre).sum(-1)/total.clamp_min(1e-6)\n    valid = (total > 1e-6).float()\n    count = valid.sum(1)\n    mean_x = (cx*valid).sum(1)/count.clamp_min(1)*scale\n    mean_y = (cy*valid).sum(1)/count.clamp_min(1)*scale\n    shifts = torch.stack((torch.floor(mean_x+.5), torch.floor(mean_y+.5)), 1)\n    return (shifts*(count > 0).float()[:, None]).long()\n\n\ndef anchored_seeds(bases, shifts, out_size):\n    """bases [N,C,S,S] periodic seeds, shifts [N,2] -> seeds [N,C,out,out] with\n    out[y, x] = base[(y - sy) mod S, (x - sx) mod S]: the pattern\'s origin sits on\n    the mask centroid and moves with it."""\n    return torch.stack([torch.roll(b, shifts=(int(sh[1]), int(sh[0])), dims=(-2, -1))[:, :out_size, :out_size]\n                        for b, sh in zip(bases, shifts)])\n\n\ndef seed_pool(count, device, start=SEED_VALUE+1000, shapes="fields"):\n    """Training seeds: `count` independent seed sets (not the app\'s set)."""\n    make = SEED_SOURCES[shapes][0]\n    return torch.stack([make(seed_value=start+i) for i in range(count)]).to(device, torch.float16)\n\n\ndef moving_training_seeds(pool, restored_grids, flow_model, out_size, fixed=False):\n    """Seeds of the three window centres (frames 2, 3, 4 of seven) that move\n    with the content: a random crop of a random pool set at frame 3, carried to\n    frames 2 and 4 along RAFT flow of the restored frames at the output size."""\n    b = restored_grids.shape[0]\n    side = pool.shape[-1]\n    if fixed:\n        base = pool[:1, :, :out_size, :out_size].expand(b, -1, -1, -1).float()\n    else:\n        index = torch.randint(len(pool), (b,), device=pool.device)\n        shifts = torch.randint(side, (b, 2))\n        base = []\n        for i in range(b):\n            s = torch.roll(pool[index[i]], shifts=(int(shifts[i, 0]), int(shifts[i, 1])), dims=(-2, -1))\n            k = int(torch.randint(4, (1,)))\n            s = torch.rot90(s, k, (-2, -1))\n            base.append(s[:, :out_size, :out_size])\n        base = torch.stack(base).float()\n    flow_size = max(128, -(-out_size//8)*8)       # RAFT needs >= 128 px, multiples of 8\n    frames = F.interpolate(restored_grids[:, 2:5].reshape(-1, 3, *restored_grids.shape[-2:]).float(),\n                           size=(flow_size, flow_size), mode="bilinear", align_corners=False).reshape(b, 3, 3, flow_size, flow_size)\n    with torch.no_grad(), torch.autocast(device_type=frames.device.type, enabled=False):\n        source = torch.cat((frames[:, 0], frames[:, 2]))*2-1           # frames 2 and 4\n        target = torch.cat((frames[:, 1], frames[:, 1]))*2-1           # frame 3\n        flows = flow_model(source, target)[-1]                         # p in 2/4 -> where it is in 3\n        if flow_size != out_size:\n            flows = F.interpolate(flows, size=(out_size, out_size), mode="bilinear", align_corners=False)*(out_size/flow_size)\n    before, after = flows.chunk(2)\n    seeds = torch.stack((warp(base, before), base, warp(base, after)), 1)\n    return seeds.reshape(b*3, *base.shape[1:])\n\n\ndef texture_distill_model(distill, architecture):\n    """v6/v8/v9 (texture first) or v7/v7b (distill first) from a saved architecture."""\n    if architecture.get("order") == "distill-then-texture":\n        return DistillTextureROI(distill, **architecture)\n    if architecture.get("order") == "texture-then-distill":\n        return TextureDistillROI(distill, **architecture)\n    if architecture.get("order") == "distill-then-seed":\n        return DistillSeedROI(distill, **architecture)\n    if architecture.get("order") == "distill-seeded-texture":\n        return DistillSeededTexture(distill, **architecture)\n    if architecture.get("order") == "distill-context-texture":\n        return DistillContextTexture(distill, **architecture)\n    raise ValueError("未知のdistill付き質感モデルです。")\n\n\ndef save_seed_direct(path, distill_checkpoint, p2=False):\n    """P1 / P2 as a checkpoint (frozen distill + the law-built seed, no learned\n    weights), so it is evaluated, compared and exported like the trained versions."""\n    model = texture_distill_model(load_distill(distill_checkpoint), SEED_DIRECT_P2_ARCH if p2 else SEED_DIRECT_ARCH)\n    torch.save({"version": "roi-mask-gan-p2" if p2 else SEED_DIRECT_VERSION, **model_metadata(model),\n                "detail_architecture": model.architecture, "model": model.state_dict(), "step": 0,\n                "laws": {k: v for k, v in (SEED_LAWS_P2 if p2 else SEED_LAWS).items()}, "app_integrated": False}, path)\n    return model\n\n\ndef load_distill(path):\n    distill, _ = base.load_checkpoint(path)\n    return distill.eval().requires_grad_(False)\n\n\ndef to_grid(frames, size):\n    """[B,T,C,H,W] -> [B,T,C,size,size]: antialiased RGB, area-averaged masks."""\n    b, t, c, h, w = frames.shape\n    if size == h and size == w:\n        return frames\n    flat = frames.reshape(b*t, c, h, w)\n    if c == 1:\n        small = F.interpolate(flat, size=(size, size), mode="area")\n    else:\n        small = F.interpolate(flat, size=(size, size), mode="bilinear", align_corners=False, antialias=True)\n    return small.reshape(b, t, c, size, size)\n\n\ndef sequence_v6(model, restored, mosaic, mask, scale, flow_model=None, pool=None, type_probs=None):\n    """Seven native frames -> three centres at native size: shrink to the grid,\n    texture + distill x2 there, resize the x2 result back (as the app pastes)."""\n    b, _, _, h, w = restored.shape\n    # v13: whole-ROI samples are already the app\'s 256 grid; the x2 result is the output.\n    wide = getattr(model, "architecture", {}).get("order") == "distill-context-texture"\n    size = h if (scale >= 1 or wide) else max(32, int(round(h*scale/4))*4)\n    grids = [to_grid(x, size) for x in (restored, mosaic, mask)]\n    arguments = [torch.stack([x[:, i:i+5] for i in range(3)], 1).reshape(b*3, 5, x.shape[2], size, size)\n                 for x in grids]\n    channels = getattr(model, "architecture", {}).get("seed_channels")\n    motion_kind = getattr(model, "architecture", {}).get("seed_motion")\n    fixed_seed = motion_kind == "fixed"\n    if motion_kind == "mask":\n        # v12: one seed per sample (training: a random pool set, rolled and rotated;\n        # evaluation: the app\'s built-in set), anchored to each window\'s masks.\n        side = (pool if pool is not None else model.seed_base).shape[-1]\n        bases = []\n        for i in range(b):\n            if model.training and pool is not None:\n                k = int(torch.randint(len(pool), (1,)))\n                shifts = torch.randint(side, (2,))\n                base_seed = torch.roll(pool[k], shifts=(int(shifts[0]), int(shifts[1])), dims=(-2, -1))\n                base_seed = torch.rot90(base_seed, int(torch.randint(4, (1,))), (-2, -1))\n            else:\n                base_seed = model.seed_base[0]\n            bases.append(base_seed)\n        bases = torch.stack(bases).repeat_interleave(3, 0)\n        windows = torch.stack([grids[2][:, i:i+5] for i in range(3)], 1).reshape(b*3, 5, 1, size, size)\n        seeds = anchored_seeds(bases, mask_anchor_shifts(windows), 2*size).float()\n        prediction, coarse = model(*arguments, seed_fields=seeds, type_probs=type_probs)\n    elif fixed_seed and model.training and pool is not None:\n        # v12: the app keeps the built-in seed in crop coordinates, so training uses one\n        # random pool set (rolled, rotated) per sample, shared by its three windows.\n        side = pool.shape[-1]\n        seeds = []\n        for i in range(b):\n            k = int(torch.randint(len(pool), (1,)))\n            shifts = torch.randint(side, (2,))\n            s = torch.roll(pool[k], shifts=(int(shifts[0]), int(shifts[1])), dims=(-2, -1))\n            s = torch.rot90(s, int(torch.randint(4, (1,))), (-2, -1))\n            seeds.append(s[:, :2*size, :2*size])\n        seeds = torch.stack(seeds).float().repeat_interleave(3, 0)\n        prediction, coarse = model(*arguments, seed_fields=seeds, type_probs=type_probs)\n    elif fixed_seed:\n        prediction, coarse = model(*arguments, type_probs=type_probs)   # the built-in seed at the origin\n    elif getattr(model, "architecture", {}).get("order") == "distill-seeded-texture":\n        # v11: the seed moves with the content across the three window centres\n        # (training: random pool sets; evaluation: the app\'s fixed set at the origin).\n        if flow_model is None:\n            raise ValueError("v11は種を動かすために動き推定が必要です。")\n        source = pool if (model.training and pool is not None) else model.seed_base\n        if model.training and pool is not None:\n            seeds = moving_training_seeds(source, grids[0], flow_model, 2*size)\n        else:\n            seeds = moving_training_seeds(source, grids[0], flow_model, 2*size, fixed=True)\n        prediction, coarse = model(*arguments, seed_fields=seeds, type_probs=type_probs)\n    elif channels and model.training:\n        # One fresh seed per sample, shared by its three windows (still in crop coordinates).\n        noise = torch.randn(b, channels, size, size, device=restored.device).repeat_interleave(3, 0)\n        prediction, coarse = model(*arguments, noise=noise)\n    elif model.training and getattr(model, "architecture", {}).get("learned"):\n        # v10: a fresh white field per sample for the law-built seed, shared by its windows.\n        noise = model.training_noise(b, size, restored.device).repeat_interleave(3, 0)\n        prediction, coarse = model(*arguments, noise=noise)\n    else:\n        prediction, coarse = model(*arguments)\n\n    def native(x):\n        if wide:\n            return x.reshape(b, 3, 3, *x.shape[-2:])\n        return F.interpolate(x, size=(h, w), mode="bilinear", align_corners=False,\n                             antialias=x.shape[-1] > h).reshape(b, 3, 3, h, w)\n    return native(prediction), native(coarse)\n\n\ndef evaluate_v6(model, dataset, device, flow_model, features, samples=64, preview_path=None, side=5,\n                scale=EVAL_GRID_SCALE, gain=1., reference=None):\n    """`reference` (v16: the frozen start model) replaces the restoration in the previews and sits\n    right next to the current model (mosaic / distill / start / current / truth / mask), so each\n    step is seen against where it started."""\n    model.eval()\n    chosen = np.linspace(0, len(dataset)-1, min(samples, len(dataset))).round().astype(int)\n    sums, rows, detail_rows = {}, [], []\n    with torch.inference_mode():\n        for i, index in enumerate(chosen):\n            item = [x[None].to(device) for x in dataset[int(index)]]\n            restored, mosaic, target, mask = item[:4]\n            swift = item[4] if len(item) > 4 else None     # v19: SwiftVR teacher frames\n            prediction, coarse = sequence_v6(model, restored, mosaic, mask, scale, flow_model=flow_model)\n            flows, valid = motion(flow_model, coarse)\n            wide = prediction.shape[-1] != mask.shape[-1]\n            roi = up_masks(mask[:, 2:5], prediction.shape[-1]) if wide else mask[:, 2:5]\n            values = {k: float(v) for k, v in\n                      metrics_v4(prediction, target, roi, coarse, flows, valid, features, side, gain,\n                                 FINE_GAIN_BAND).items()}\n            if wide:\n                goal, goal_gain = ((texture_goal(target, gain, FINE_GAIN_BAND, MID_GAIN, JUDGE_BAND), 1.) if FINE_GAIN_BAND else (target, gain))\n                values["fine"] = float(fine_structure(features, prediction,\n                                                      texture_target(prediction, goal, roi, side, goal_gain), roi))\n            if swift is not None:\n                values["teacher_corr"] = float(teacher_correlation(prediction, swift, coarse, roi))\n            if not all(math.isfinite(v) for v in values.values()):\n                raise RuntimeError("検証指標が非有限値です。")\n            for k, v in values.items():\n                sums[k] = sums.get(k, 0.)+v\n            if i < 6:\n                big = (lambda x: F.interpolate(x[None], size=prediction.shape[-2:], mode="bilinear",\n                                               align_corners=False)[0]) if wide else (lambda x: x)\n                if swift is not None:   # v19: mosaic / distill / SwiftVR teacher / current / truth / mask\n                    rows.append([big(mosaic[0, 3]), coarse[0, 1].clamp(0, 1), swift[0, 1].clamp(0, 1),\n                                 prediction[0, 1].clamp(0, 1), target[0, 1], big(mask[0, 3]).expand(3, -1, -1)])\n                    detail_rows.append([coarse[0, 1], swift[0, 1], prediction[0, 1], target[0, 1]])\n                elif reference is not None:\n                    before, _ = sequence_v6(reference, restored, mosaic, mask, scale, flow_model=flow_model)\n                    rows.append([big(mosaic[0, 3]), coarse[0, 1].clamp(0, 1), before[0, 1].clamp(0, 1),\n                                 prediction[0, 1].clamp(0, 1), target[0, 1], big(mask[0, 3]).expand(3, -1, -1)])\n                    detail_rows.append([coarse[0, 1], before[0, 1], prediction[0, 1], target[0, 1]])\n                else:\n                    rows.append([big(mosaic[0, 3]), big(restored[0, 3]), coarse[0, 1].clamp(0, 1),\n                                 prediction[0, 1].clamp(0, 1), target[0, 1], big(mask[0, 3]).expand(3, -1, -1)])\n                    detail_rows.append([coarse[0, 1], prediction[0, 1], target[0, 1]])\n    report = {k: v/len(chosen) for k, v in sums.items()}\n    report.update(psnr=-10*math.log10(max(report["mse"], 1e-12)), samples=len(chosen), grid_scale=scale)\n    if preview_path:\n        preview(preview_path, rows)\n        preview(Path(preview_path).with_name("detail-"+Path(preview_path).name), detail_rows, detail=True)\n    return report\n\n\nWIDE_KIND = "wide-roi-v1"\n\n\ndef wide_box(mask, width, height):\n    """The app\'s ROI crop (cropToBox, simplified): bounding box of the largest\n    detection, a max(20 px, 6%) border, grown to a square inside the frame."""\n    from scipy import ndimage\n    labels, count = ndimage.label(mask > .05)\n    if count == 0:\n        return None\n    sizes = ndimage.sum(mask > .05, labels, range(1, count+1))\n    ys, xs = np.where(labels == int(np.argmax(sizes))+1)\n    left, right, top, bottom = xs.min(), xs.max(), ys.min(), ys.max()\n    border = max(20, int(max(right-left+1, bottom-top+1)*.06))\n    left, top = max(0, left-border), max(0, top-border)\n    right, bottom = min(width-1, right+border), min(height-1, bottom+border)\n    side = min(max(right-left+1, bottom-top+1), width, height)\n    cx, cy = (left+right+1)/2, (top+bottom+1)/2\n    x = int(np.clip(round(cx-side/2), 0, width-side))\n    y = int(np.clip(round(cy-side/2), 0, height-side))\n    return x, y, int(side)\n\n\ndef wide_resize(frames, side, channels_last=True):\n    """uint8 [T,H,W(,C)] -> uint8 [T,side,side(,C)]: area when shrinking, bicubic when enlarging."""\n    t = torch.from_numpy(np.ascontiguousarray(frames)).float()\n    t = t.permute(0, 3, 1, 2) if channels_last else t[:, None]\n    mode = "area" if t.shape[-1] > side else "bicubic"\n    out = F.interpolate(t, size=(side, side), mode=mode, **({} if mode == "area" else {"align_corners": False}))\n    out = out.clamp(0, 255).round().to(torch.uint8)\n    return (out.permute(0, 2, 3, 1) if channels_last else out[:, 0]).numpy()\n\n\ndef prepare_wide(config):\n    """v13 materials: one whole-ROI sample per checked centre (same videos, same\n    whole-file splits and sync audit as the v1 patches): restored/mosaic 7 frames and\n    masks at the app\'s 256 grid, the uncensored centre 3 frames at the 512 output.\n    Shards keep the v1 keys, so the v1 loader and batcher are reused."""\n    dataset, prepared, _, work = paired.validate_roots(config)\n    items = paired.discover(dataset, config.get("material_overrides"))\n    for number, item in enumerate(items, 1):\n        print(f"素材照合: {number}/{len(items)}組", flush=True)\n        item["info"] = {k: paired.video_info(item[k]) for k in ("mosaic", "restoration", "target")}\n        zero = paired.decode_indices(item["restoration"], item["info"]["restoration"], [0])[0]\n        item["lead"] = paired.restore_alignment(item["info"], zero)   # restoration frame t = source t-lead\n    contract = {"version": paired.VERSION, "kind": WIDE_KIND, "grid": 256, "target_side": 512,\n                "offsets": [p["target_offset"] for p in items], "names": [Path(p["restoration"]).name for p in items],\n                "interval_seconds": float(config.get("interval_seconds", 4)),\n                "mask_mode": config.get("mask_mode", "corrected"), "minimum_side": int(config.get("minimum_side", 96))}\n    signature = paired.digest(contract)\n    prepared.mkdir(parents=True, exist_ok=True)\n    state_path = prepared/"preparation.json"\n    if state_path.exists():\n        state = json.loads(state_path.read_text())\n        if state["signature"] != signature:\n            raise ValueError("保存済みの全体素材と設定が異なります。新しいWIDE_ROOTを指定してください。")\n    else:\n        if any(prepared.iterdir()):\n            raise ValueError("全体素材の保存先が空ではありません。新しいフォルダを指定してください。")\n        state = {"signature": signature, "contract": contract, "groups": [], "checks": []}\n        base.atomic_json(state_path, state)\n    tasks = paired.windows_and_splits(items, contract["interval_seconds"])\n    done = {g["task"] for g in state["groups"]}\n    started, minutes = time.monotonic(), float(config.get("prepare_minutes", 150))\n    for pair_index, item in enumerate(items):\n        pending = [(n, t) for n, t in enumerate(tasks) if t["pair"] == pair_index and n not in done]\n        if not pending:\n            continue\n        local = work/"media"/item["id"]\n        paths = {}\n        for key in ("mosaic", "restoration", "target", "masks"):\n            paths[key] = local/(key+(".json" if key == "masks" else ".mp4"))\n            paired.cached_copy(item[key], paths[key])\n        height, width = item["info"]["restoration"]["height"], item["info"]["restoration"]["width"]\n        wanted = {j for _, task in pending for c in task["centers"] for j in range(c-4, c+5)}\n        _, offsets = paired.stream_mask_index(paths["masks"], wanted, item["info"]["restoration"]["frames"],\n                                              (height, width), item["info"]["restoration"]["fps"])\n        with paths["masks"].open("rb") as mask_file:\n            for task_id, task in pending:\n                arrays = {k: [] for k in ("restoration", "mosaic", "target", "mask")}\n                records, checks = [], []\n                for center in task["centers"]:\n                    logical = np.arange(center-4, center+5)\n                    masks = []\n                    for frame in logical:\n                        mask_file.seek(offsets[int(frame)])\n                        row = json.loads(mask_file.readline().strip().rstrip(b","))\n                        masks.append(paired.training_mask(row, height, width, contract["mask_mode"]))\n                    masks = np.stack(masks)\n                    box = wide_box(masks[4], width, height)\n                    if box is None or box[2] < contract["minimum_side"]:\n                        checks.append({"pair": item["id"], "center": center, "reason": "no_roi"})\n                        continue\n                    restored = paired.decode_indices(paths["restoration"], item["info"]["restoration"], logical)\n                    mosaic = paired.decode_indices(paths["mosaic"], item["info"]["mosaic"], logical-item["lead"])\n                    target = paired.decode_indices(paths["target"], item["info"]["target"],\n                                                   logical-item["lead"]+item["target_offset"])\n                    a, b = paired.sync_check(mosaic, restored, masks), paired.sync_check(mosaic, target, masks)\n                    checks.append({"pair": item["id"], "center": center, "restored": a, "target": b})\n                    if not a["accepted"] or not b["accepted"]:\n                        continue\n                    x, y, side = box\n                    cut = (slice(None), slice(y, y+side), slice(x, x+side))\n                    arrays["restoration"].append(wide_resize(restored[1:8][cut], 256))\n                    arrays["mosaic"].append(wide_resize(mosaic[1:8][cut], 256))\n                    arrays["target"].append(wide_resize(target[3:6][cut], 512))\n                    arrays["mask"].append(wide_resize(np.rint(masks[1:8][cut]*255).astype(np.uint8), 256, False))\n                    records.append({"center": int(center), "box": [x, y, side]})\n                filename = f"wide-{task_id:05d}.npz"\n                local_shard = work/filename\n                work.mkdir(parents=True, exist_ok=True)\n                np.savez_compressed(local_shard, **{k: np.stack(v) if v else np.empty((0,), np.uint8)\n                                                    for k, v in arrays.items()})\n                temporary = prepared/(filename+".partial")\n                shutil.copyfile(local_shard, temporary)\n                os.replace(temporary, prepared/filename)\n                state["groups"].append({"task": task_id, "file": filename, "sha256": paired.sha256(local_shard),\n                                        "pair": item["id"], "split": task["split"], "records": records})\n                state["checks"].extend(checks)\n                base.atomic_json(state_path, state)\n                local_shard.unlink()\n                counts = {s: sum(len(g["records"]) for g in state["groups"] if g["split"] == s)\n                          for s in ("train", "val", "test")}\n                print(f"全体素材 {len(state[\'groups\'])}/{len(tasks)}区間: {counts}", flush=True)\n                if time.monotonic()-started >= minutes*60:\n                    print("時間上限です。同じセルを再実行すると完成区間を再利用します。", flush=True)\n                    return {"complete": False, "counts": counts}\n    groups = [g for g in state["groups"] if g["records"]]\n    counts = {s: sum(len(g["records"]) for g in groups if g["split"] == s) for s in ("train", "val", "test")}\n    manifest = {"version": paired.VERSION, "kind": WIDE_KIND, "signature": signature, "contract": contract,\n                "groups": groups, "counts": counts, "complete": True, "split_unit": "whole-video-file"}\n    if counts["train"] < 200 or counts["val"] < 20:\n        raise RuntimeError(f"全体素材が不足: {counts}")\n    base.atomic_json(prepared/"manifest.json", manifest)\n    print(f"全体素材の準備完了: {counts}。まだ学習していません。", flush=True)\n    return manifest\n\n\ndef merge_wide_manifests(base_root, new_root, exclude=(), extra_roots=()):\n    """v13g: whole-ROI materials = the v1 set where it is (each group keeps its folder as\n    "root") + the shards prepared elsewhere and uploaded to new_root with manifest-new.json\n    (additional private real and synthetic pairs). Writes new_root/manifest.json.\n    extra_roots: further uploaded folders with their own manifest-new.json (v3: real pairs);\n    their groups keep their folder as "root" too."""\n    base_root, new_root = Path(base_root), Path(new_root)\n    v1, _ = read_manifest(base_root)\n    if v1.get("kind") != WIDE_KIND:\n        raise ValueError("v1の全体素材ではありません。")\n    groups = [{**g, "root": str(base_root)} for g in v1["groups"]]\n    names = {g["file"] for g in groups}\n    for root in [new_root]+[Path(r) for r in extra_roots]:\n        new = json.loads((root/"manifest-new.json").read_text())\n        for g in new["groups"]:\n            if any(g.get("source", "").startswith(prefix) for prefix in exclude):\n                continue      # v14: titles left out (interlaced truth, or excluded content)\n            if g["file"] in names or not (root/g["file"]).is_file():\n                raise ValueError(f"新しい素材のファイル名が重複・欠落しています: {g[\'file\']}")\n            names.add(g["file"])\n            groups.append(g if root == new_root else {**g, "root": str(root)})\n    counts = {s: sum(len(g["records"]) for g in groups if g["split"] == s) for s in ("train", "val", "test")}\n    manifest = {"version": paired.VERSION, "kind": WIDE_KIND, "complete": True, "groups": groups, "counts": counts,\n                "split_unit": "whole-video-file", "merged_from": [str(base_root)]+[str(Path(r)/"manifest-new.json") for r in [new_root, *extra_roots]],\n                "excluded_sources": list(exclude)}\n    path = new_root/"manifest.json"\n    if not path.exists() or json.loads(path.read_text()) != manifest:\n        base.atomic_json(path, manifest)\n    return manifest\n\n\ndef mask_rows(path, step):\n    """Yield (frame_index, row) for every `step`-th native mask JSON line with detections."""\n    with Path(path).open("rb") as handle:\n        if not handle.readline(4*1024*1024).endswith(b\',"frames":[\\n\'):\n            raise ValueError("miohの逐次書き出しJSON形式ではありません。")\n        for line in handle:\n            if line.startswith(b"]"):\n                return\n            if not line.strip():\n                continue\n            row = json.loads(line.strip().rstrip(b","))\n            if row["frame_index"] % step == 0 and row["detections"]:\n                yield row["frame_index"], row\n\n\ndef build_texture_bank(config):\n    """Dense 128 px uncensored crops on the masked regions of the TRAIN pairs\n    only (val/test stay unseen). Needs the uncensored video and its mask JSON,\n    not the restoration, so more uncensored titles can be added later the\n    same way. Written once to BANK_ROOT; an existing complete bank is reused."""\n    import av\n    bank = Path(config["bank_root"])\n    if (bank/"manifest.json").is_file():\n        manifest = json.loads((bank/"manifest.json").read_text())\n        if manifest.get("version") == BANK_VERSION and manifest.get("complete"):\n            return manifest\n        raise ValueError("BANK_ROOTに未完成の質感バンクがあります。別のフォルダを指定してください。")\n    if bank.exists() and any(bank.iterdir()):\n        raise ValueError("BANK_ROOTが空ではありません。")\n    bank.mkdir(parents=True, exist_ok=True)\n    step, side = int(config.get("bank_stride", 4)), 128\n    limit, cover = int(config.get("bank_limit", 0)), float(config.get("bank_coverage", .3))\n    rng = np.random.default_rng(int(config.get("seed", 20261006)))\n    items = paired.discover(config["dataset_root"], config.get("material_overrides"))[:-2]   # train pairs\n    shards, buffer, total = [], {"texture": [], "mask": []}, 0\n\n    def flush():\n        nonlocal buffer\n        if not buffer["texture"]:\n            return\n        name = f"bank-{len(shards):03d}.npz"\n        temporary = bank/(name+".partial")\n        with temporary.open("wb") as handle:\n            np.savez_compressed(handle, texture=np.stack(buffer["texture"]), mask=np.stack(buffer["mask"]))\n        os.replace(temporary, bank/name)\n        shards.append({"file": name, "count": len(buffer["texture"]), "sha256": paired.sha256(bank/name)})\n        buffer = {"texture": [], "mask": []}\n\n    for item in items:\n        info = paired.video_info(item["target"])\n        height, width = info["height"], info["width"]\n        # Mask row t (restoration logical index) shows uncensored frame t-1+offset.\n        wanted = {t-1+item["target_offset"]: row for t, row in mask_rows(item["masks"], step)}\n        print(f"質感バンク: {Path(item[\'target\']).name} の {len(wanted)} フレームを読み出します", flush=True)\n        with av.open(str(item["target"])) as container:\n            stream = container.streams.video[0]\n            stream.thread_type = "AUTO"\n            for index, frame in enumerate(container.decode(stream)):\n                row = wanted.get(index)\n                if row is None:\n                    continue\n                image = frame.to_ndarray(format="rgb24")\n                full = paired.raw_mask(row, height, width)\n                det = row["detections"][int(rng.integers(len(row["detections"])))]\n                x, y, w, h = det["bbox"]\n                cx = x+w/2+rng.uniform(-.25, .25)*max(0, w-side)\n                cy = y+h/2+rng.uniform(-.25, .25)*max(0, h-side)\n                left = int(np.clip(round(cx-side/2), 0, width-side))\n                top = int(np.clip(round(cy-side/2), 0, height-side))\n                crop = full[top:top+side, left:left+side]\n                if crop.mean() < cover:\n                    continue\n                buffer["texture"].append(image[top:top+side, left:left+side])\n                buffer["mask"].append((crop*255).astype(np.uint8))\n                total += 1\n                if len(buffer["texture"]) >= 2000:\n                    flush()\n                    print(f"質感バンク: {total}枚", flush=True)\n                if limit and total >= limit:\n                    break\n        if limit and total >= limit:\n            break\n    flush()\n    if not total:\n        raise ValueError("質感バンクに使える切り抜きがありません。")\n    manifest = {"version": BANK_VERSION, "complete": True, "side": side, "stride": step, "coverage": cover,\n                "pairs": len(items), "count": total, "shards": shards}\n    base.atomic_json(bank/"manifest.json", manifest)\n    print(f"質感バンク完成: {total}枚", flush=True)\n    return manifest\n\n\nclass TextureBank:\n    """All bank crops in memory (uint8); random batches for the discriminator."""\n    def __init__(self, root):\n        root = Path(root)\n        manifest = json.loads((root/"manifest.json").read_text())\n        if manifest.get("version") != BANK_VERSION or not manifest.get("complete"):\n            raise ValueError("完成した質感バンクではありません。")\n        textures, masks = [], []\n        for shard in manifest["shards"]:\n            if paired.sha256(root/shard["file"]) != shard["sha256"]:\n                raise ValueError("質感バンクのハッシュが不一致です。")\n            with np.load(root/shard["file"], allow_pickle=False) as archive:\n                textures.append(archive["texture"])\n                masks.append(archive["mask"])\n        self.texture, self.mask = np.concatenate(textures), np.concatenate(masks)\n        self.digest = paired.sha256(root/"manifest.json")\n\n    def __len__(self):\n        return len(self.texture)\n\n    def batch(self, count, device):\n        index = np.random.randint(0, len(self), count)\n        texture = torch.from_numpy(self.texture[index]).to(device).permute(0, 3, 1, 2).float()/255\n        mask = torch.from_numpy(self.mask[index]).to(device)[:, None].float()/255\n        flip = torch.rand(count, device=device) < .5\n        return torch.where(flip[:, None, None, None], texture.flip(-1), texture), \\\n            torch.where(flip[:, None, None, None], mask.flip(-1), mask)\n\n\ndef teacher_name(group):\n    """v19: the SwiftVR teacher shard of a material group (as swiftvr_teacher_runtime names it)."""\n    import hashlib\n    key = f"{group.get(\'root\', \'\')}|{group[\'file\']}"\n    return f"{hashlib.sha1(key.encode()).hexdigest()[:10]}-{Path(group[\'file\']).name}"\n\n\ndef restore_teachers(config, manifest):\n    """v19: copy the teacher shards next to the material shards (parallel, one read each)."""\n    from concurrent.futures import ThreadPoolExecutor\n    source, root = Path(config["teacher_root"]), Path(config["work_root"])/"teacher"\n    root.mkdir(parents=True, exist_ok=True)\n    names = [teacher_name(g) for g in manifest["groups"]]\n    missing = [n for n in names if not (source/n).is_file()]\n    if missing:\n        raise FileNotFoundError(f"先生データがありません: {len(missing)}件（例: {missing[0]}）")\n\n    def copy(name):\n        if not (root/name).is_file() or (root/name).stat().st_size != (source/name).stat().st_size:\n            shutil.copyfile(source/name, root/(name+".partial"))\n            os.replace(root/(name+".partial"), root/name)\n    with ThreadPoolExecutor(int(config.get("copy_workers", 8))) as pool:\n        for i, _ in enumerate(pool.map(copy, names), 1):\n            if i % 50 == 0 or i == len(names):\n                print(f"先生データ配置: {i}/{len(names)}", flush=True)\n    return root\n\n\nclass TeacherPatches(paired.Patches):\n    """v19: the material sample plus its SwiftVR teacher frames [3,3,512,512] (same flip).\n    v19b (teacher_as_target): in training the teacher also takes the truth\'s place, so the\n    discriminator\'s real samples, the texture statistics and the local amount all follow SwiftVR\'s\n    grain as a distribution; validation keeps the uncensored truth."""\n    def __init__(self, root, manifest, split, teacher_root, teacher_as_target=False):\n        super().__init__(root, manifest, split)\n        self.teacher_as_target = bool(teacher_as_target)\n        self.teacher_root = Path(teacher_root)\n        self.teacher_file = {g["file"]: teacher_name(g) for g in manifest["groups"]}\n        self.teacher_cached_name, self.teacher_cached = None, None\n\n    def __getitem__(self, index):\n        name, offset = self.items[index]\n        if name != self.cached_name:\n            with np.load(self.root/name, allow_pickle=False) as archive:\n                self.cached = {key: archive[key] for key in ("restoration", "mosaic", "target", "mask")}\n            self.cached_name = name\n        if name != self.teacher_cached_name:\n            with np.load(self.teacher_root/self.teacher_file[name], allow_pickle=False) as archive:\n                self.teacher_cached = archive["teacher"]\n            self.teacher_cached_name = name\n        arrays = [self.cached[k][offset] for k in ("restoration", "mosaic", "target", "mask")]+[self.teacher_cached[offset]]\n        if self.augment and random.random() < .5:\n            arrays = [a[:, :, ::-1].copy() for a in arrays]\n        result = []\n        for i, array in enumerate(arrays):\n            array = array[:, None] if i == 3 else array.transpose(0, 3, 1, 2)\n            result.append(torch.from_numpy(np.ascontiguousarray(array)).float()/255)\n        if self.teacher_as_target:\n            result[2] = result[4]\n        return tuple(result)\n\n\ndef _teacher_layers(prediction, swift, coarse, roi):\n    """v19: the <= 17 px luma layers the model and SwiftVR add over the distill, and where to\n    compare them: deep inside the mask (64 px ramp >= .75) and away from outlines."""\n    def luma(x):\n        return .299*x[:, :, 0:1]+.587*x[:, :, 1:2]+.114*x[:, :, 2:3]\n    now, goal = highpass(luma(prediction-coarse), 17), highpass(luma(swift-coarse), 17)\n    b, t, _, h, w = roi.shape\n    deep = (mask_fade(roi.reshape(-1, 1, h, w).float(), 64) >= .75).float().reshape(b, t, 1, h, w)\n    away = (1-outline_protection(coarse.reshape(-1, 3, h, w), 6, 8)).reshape(b, t, 1, h, w)\n    return now, goal, deep*away\n\n\ndef teacher_loss(prediction, swift, coarse, roi):\n    now, goal, weight = _teacher_layers(prediction, swift, coarse, roi)\n    return (paired.weighted_mean((now-goal).abs(), weight)\n            /(paired.weighted_mean(goal.abs(), weight)+1e-4))\n\n\ndef teacher_correlation(prediction, swift, coarse, roi):\n    now, goal, weight = _teacher_layers(prediction, swift, coarse, roi)\n    return (weight*now*goal).sum()/((weight*now.square()).sum().sqrt()*(weight*goal.square()).sum().sqrt()+1e-8)\n\n\ndef highlight_map(x):\n    """v16: bright specular glints [B,T,1,H,W] of x [B,T,3,H,W] (0..1): a local luminance peak\n    above its 33 px surround, or plain near-white. Skin grain is not counted."""\n    y = .299*x[:, :, 0:1]+.587*x[:, :, 1:2]+.114*x[:, :, 2:3]\n    smooth = blur(y, 3)\n    return torch.maximum(((smooth-blur(y, 33)-.04)/.06).clamp(0, 1), ((smooth-.80)/.10).clamp(0, 1))\n\n\ndef fine_layer_amount(added):\n    """v16: local amount (17 px) of the 5-17 px band of an added layer [B,T,3,H,W]."""\n    band = highpass(added, 17)-highpass(added, 5)\n    return (blur(band.square().mean(2, keepdim=True), 17)+1e-10).sqrt()\n\n\ndef drop_highlights(logits_support, highlight):\n    """v16: the discriminator\'s support without the highlight places."""\n    logits, support = logits_support\n    away = 1-F.interpolate(highlight.reshape(-1, 1, *highlight.shape[-2:]).float(), size=support.shape[-2:],\n                           mode="area")\n    return logits, support*away\n\n\ndef train(config):\n    # v6: the frozen ROI distill takes the place of the v1 stage.\n    distill_path = Path(config["distill_checkpoint"]).resolve()\n    prepared, initial, save, work = validate_paths({**config, "init_checkpoint": str(distill_path)})\n    manifest, manifest_hash = read_manifest(prepared)\n    device = torch.device(config.get("device", "cuda"))\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("GPUへ接続してください。T4/A100などに対応します。")\n    max_steps, minutes = int(config.get("max_steps", 2000)), float(config.get("max_minutes", 120))\n    interval, batch_size = int(config.get("eval_interval", 250)), int(config.get("batch_size", 4))\n    gan_weight = float(config.get("gan_weight", .3))\n    warmup, ramp = int(config.get("gan_warmup", 500)), int(config.get("gan_ramp", 1500))\n    weights = {"anchor": float(config.get("anchor_weight", 2.)), "style": float(config.get("style_weight", 1.)),\n               "amplitude": float(config.get("amplitude_weight", .5)),\n               "flicker": float(config.get("flicker_weight", 1.))}\n    context = TRAIN_ARCH.get("order") == "distill-context-texture"\n    if context and not CLEAN_UNET:\n        weights["structure"] = float(config.get("structure_weight", .5))\n        weights["fine"] = float(config.get("fine_weight", .5))\n    if VERSION in ("roi-mask-gan-v13d", "roi-mask-gan-v13h", "roi-mask-gan-v13i"):\n        weights["ghost"] = float(config.get("ghost_weight", 1.))\n    if NO_RELIEF:\n        # This is a training invariant, not an inference repair: the texture residual\n        # itself must not create an edge beside one already owned by the frozen distill.\n        weights["ghost"] = 1.\n    if TRAIN_ARCH.get("line_mode"):\n        weights["misalign"] = float(config.get("orient_weight", 1.))   # 1 - orient\n    if V19:\n        weights["teacher"] = float(config.get("teacher_weight", 0. if config.get("teacher_as_target") else 4.))\n    if VERSION in ("roi-mask-gan-v13g", "roi-mask-gan-v13h", "roi-mask-gan-v13i", "roi-mask-gan-v17", "roi-mask-gan-v18", "roi-mask-gan-v19"):\n        weights["fine_amp"] = float(config.get("fine_amp_weight", 1.))\n    bank_weight = float(config.get("bank_weight", .7 if V16 else .5))\n    if V16:\n        weights.update(keep=float(config.get("keep_weight", 2.)), highlight=float(config.get("highlight_weight", 2.)),\n                       temporal=float(config.get("temporal_weight", 2.)))\n    band = 9   # texture band of the native-size output (D, style reference, flicker), as v4\n    if TRAIN_ARCH.get("band", 9) > 9:\n        band = JUDGE_BAND   # v12b: the wide band everywhere (equal to the output band at the eval scale)\n    # v10c: the texture goal is `gain` x the truth\'s (amplitude, style reference, real\n    # samples for D, detail-ratio score, flicker allowance all scale together).\n    gain = float(config.get("texture_gain", TRAIN_ARCH.get("texture_gain", 1.)))\n    # The seed starts at the same goal as the training (P2 x gain).\n    arch = {**TRAIN_ARCH, "texture_gain": gain} if TRAIN_ARCH.get("order") == "distill-then-seed" else TRAIN_ARCH\n    scales = tuple(float(x) for x in config.get("grid_scales", GRID_SCALES))\n    eval_scale = float(config.get("eval_grid_scale", EVAL_GRID_SCALE))\n    if (not 0 < max_steps <= 100000 or not math.isfinite(minutes) or minutes <= 0 or\n            not 1 <= interval <= 10000 or not 1 <= batch_size <= 64 or not 0 <= gan_weight <= .5 or\n            not 0 <= warmup <= 20000 or not 1 <= ramp <= 20000 or\n            not all(math.isfinite(v) and 0 <= v <= 100 for v in weights.values()) or\n            not scales or not all(.2 <= x <= 1 for x in scales+(eval_scale,)) or not .5 <= gain <= 3):\n        raise ValueError("学習設定が範囲外です。")\n    bank = TextureBank(config["bank_root"]) if config.get("bank_root") else None\n    typed = bool(arch.get("types")) and not arch.get("plain_unet")\n    book = None\n    if typed:\n        if not config.get("codebook") or not Path(config["codebook"]).is_file():\n            raise ValueError("v11には質感の種類の辞書(codebook)が必要です。セル4で作成してください。")\n        book = torch.load(config["codebook"], map_location="cpu", weights_only=True)\n        if book.get("version") != CODEBOOK_VERSION or book["centres"].shape[0] != arch["types"]:\n            raise ValueError("質感の種類の辞書の版・種類数が一致しません。")\n    type_weight = float(config.get("type_weight", .2))\n    teacher_rate = float(config.get("type_teacher_rate", .5))\n    # No term rewards the target\'s exact texture positions: the frozen stage\n    # owns colour/shape (anchor), texture is judged by statistics and realism.\n    contract = {"manifest": manifest_hash, "distill": paired.sha256(initial), "detail": arch,\n                "grid_scales": list(scales), "eval_grid_scale": eval_scale,\n                "objective": "unpaired-texture", "weights": weights, "gan_weight": gan_weight,\n                "gan_warmup": warmup, "gan_ramp": ramp, "stage": "distill-frozen", "texture_side": band,\n                "bank": bank.digest if bank else None, "lr_detail": 1e-4, "lr_disc": 1e-4,\n                "batch_size": batch_size, "eval_samples": int(config.get("eval_samples", 64)), "texture_gain": gain,\n                "codebook": paired.sha256(config["codebook"]) if typed else None,\n                "type_weight": type_weight if typed else None, "type_teacher_rate": teacher_rate if typed else None,\n                "seed": int(config.get("seed", 20261006)),\n                **({"teacher_root": str(config["teacher_root"]),\n                    "teacher_as_target": bool(config.get("teacher_as_target", False))} if V19 else {}),\n                **({"init_texture": paired.sha256(config["init_texture"])} if VERSION in ("roi-mask-gan-v14f", "roi-mask-gan-v16") else {}),\n                **({"bank_weight": bank_weight} if V16 or CLEAN_UNET else {})}\n    random.seed(contract["seed"])\n    torch.manual_seed(contract["seed"])\n    np.random.seed(contract["seed"])\n    save.mkdir(parents=True, exist_ok=True)\n    latest = save/"latest.pt"\n    model = texture_distill_model(load_distill(initial), arch).to(device)\n    model.distill.requires_grad_(False)\n    discriminator = (ContextTextureDiscriminator(band, low_cut=TRAIN_ARCH.get("detail_low_cut", 0)) if CONTEXT_DISCRIMINATOR\n                     else TextureDiscriminator(band)).to(device)\n\n    def judge(x, m, context=None):    # v14: the place goes with the detail\n        return discriminator(x, m, context) if CONTEXT_DISCRIMINATOR else discriminator(x, m)\n    start_model = None\n    if VERSION in ("roi-mask-gan-v14f", "roi-mask-gan-v16"):\n        # v14f/v16: start from a v14e (or v15) snapshot; the parts that decide the relief gain stay.\n        source = torch.load(config["init_texture"], map_location="cpu", weights_only=True)\n        allowed = (("roi-mask-gan-v14e", "roi-mask-gan-v15-temporal-fine") if V16 else ("roi-mask-gan-v14e",))\n        if source.get("version") not in allowed:\n            raise ValueError(f"開始点は{allowed}のスナップショットにしてください。")\n        model.load_state_dict(source["model"], strict=True)\n        for n, p in model.named_parameters():\n            if n.startswith(FROZEN_PREFIXES_V14F):\n                p.requires_grad_(False)\n        if V16:   # the frozen start: the added layer may not lose detail against it\n            start_model = copy.deepcopy(model).eval().requires_grad_(False)\n    trainable = [p for n, p in model.named_parameters() if not n.startswith("distill.") and p.requires_grad]\n    optimizers = [torch.optim.AdamW(trainable, lr=contract["lr_detail"],\n                                    betas=(.5, .99), weight_decay=0),\n                  torch.optim.Adam(discriminator.parameters(), lr=contract["lr_disc"], betas=(.5, .99))]\n    step, stale, state = 0, 0, None\n    if latest.is_file():\n        state = torch.load(latest, map_location="cpu", weights_only=True)\n        if state.get("version") != VERSION or state["contract"] != contract or not (save/"best.pt").is_file():\n            raise ValueError("再開条件が違います。元の設定を使うか、新しいSAVE_ROOTへ保存してください。")\n        model.load_state_dict(state["model"], strict=True)\n        discriminator.load_state_dict(state["discriminator"], strict=True)\n        for optimizer, saved in zip(optimizers, state["optimizers"]):\n            optimizer.load_state_dict(saved)\n            for values in optimizer.state.values():\n                for key, value in values.items():\n                    if isinstance(value, torch.Tensor) and key != "step":\n                        values[key] = value.to(device)\n        step, stale = int(state["step"]), int(state["stale"])\n        torch.set_rng_state(state["torch_rng"])\n        random.setstate(state["python_rng"])\n    elif any(save.iterdir()):\n        raise ValueError("SAVE_ROOTが空ではありません。新しい保存先を指定してください。")\n    root = paired.restore_patches(config, manifest)\n    if V19:\n        teachers = restore_teachers(config, manifest)\n        # v19b: the paired L1 to SwiftVR (v19) drove the residual to zero (change .05 levels):\n        # SwiftVR\'s grain is not predictable pixel for pixel, so zero is the cheapest answer.\n        # SwiftVR is used as the realism reference instead (teacher_as_target, teacher_weight 0).\n        training = TeacherPatches(root, manifest, "train", teachers,\n                                  teacher_as_target=bool(config.get("teacher_as_target", False)))\n        validation = TeacherPatches(root, manifest, "val", teachers)\n    else:\n        training, validation = paired.Patches(root, manifest, "train"), paired.Patches(root, manifest, "val")\n    print("動き推定(RAFT)と質感特徴(VGG16)を読み込みます（初回は公開重みを取得）。", flush=True)\n    flow_model, features = load_flow(device), load_features(device)\n    if bank:\n        print(f"質感バンク: {len(bank)}枚", flush=True)\n    pool = None\n    uses_seed = bool(arch.get("seed_channels")) and arch.get("seed_inject", True)\n    if uses_seed:\n        print("種の素材(16種類の形" + ({"stamps10": "・10pxの粒をランダムに散らす", "stamps10-30": "・10〜30pxの粒をランダムに散らす"}.get(arch.get("seed_shapes"), "")) + ")を作ります。", flush=True)\n        pool = seed_pool(int(config.get("seed_pool", 24)), device, shapes=arch.get("seed_shapes", "fields"))\n    reference = state["reference"] if state else evaluate_v6(model, validation, device, flow_model, features,\n                                                             contract["eval_samples"], save/"comparison-initial.png",\n                                                             band, eval_scale, gain, reference=start_model)\n    best_score = state["best_score"] if state else score_v4(reference, gain)\n    if not state:\n        base.atomic_json(save/"reference.json", reference)\n        base.atomic_json(save/"evaluation-best.json", {**reference, "step": 0, "selected": False})\n        for prefix in ("", "detail-"):\n            shutil.copyfile(save/(prefix+"comparison-initial.png"), save/(prefix+"comparison-best.png"))\n        save_checkpoint(save/"best.pt", model, discriminator, optimizers, step, contract, reference, best_score, stale)\n        save_checkpoint(latest, model, discriminator, optimizers, step, contract, reference, best_score, stale)\n    bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()\n    loader = torch.utils.data.DataLoader(training, batch_sampler=paired.ShardBatches(training, batch_size), num_workers=0)\n    started, last_eval = time.monotonic(), step\n    print(f"学習開始: step={step}/{max_steps}, {TRAIN_ARCH[\'order\']}, 質感{TRAIN_ARCH[\'band\']}px帯域"\n          f"（モザイクは見ない, {\'決定論的U-Net・種なし・種類辞書なし\' if CLEAN_UNET else \'種=\'+str(TRAIN_ARCH.get(\'seed_channels\', 0))}）"\n          f"→マスク合成, 初期={\'distillのみ\' if CLEAN_UNET else \'学習モデル\'}, "\n          f"質感の目標=正解の{gain:g}倍{f\'（{FINE_GAIN_BAND}px以下）\' if V18 else f\'（{FINE_GAIN_BAND}px以下、起伏は{MID_GAIN:g}倍）\' if FINE_GAIN_BAND else \'\'}, "\n          f"グリッド縮小率{list(scales)}（検証{eval_scale}）, "\n          f"GAN={gan_weight}, {\'BF16\' if bf16 else \'FP32\'}, 初期: 質感距離={reference[\'style\']:.4f}, "\n          f"構造距離={reference[\'structure\']:.4f}, "\n          f"細部量/正解={reference[\'detail_ratio\']:.3f}, チラつき={reference[\'flicker\']:.5f}"\n          f"（正解{reference[\'flicker_truth\']:.5f}）", flush=True)\n\n    def validate():\n        nonlocal stale, best_score\n        report = evaluate_v6(model, validation, device, flow_model, features, contract["eval_samples"],\n                             save/"comparison-latest.png", band, eval_scale, gain, reference=start_model)\n        candidate, passed = score_v4(report, gain), guard_v4(report, reference, gain)\n        selected = passed and candidate < best_score*(1-1e-4)\n        stale = 0 if selected else stale+1\n        if selected:\n            best_score = candidate\n            save_checkpoint(save/"best.pt", model, discriminator, optimizers, step, contract, reference, best_score, stale)\n            for prefix in ("", "detail-"):\n                shutil.copyfile(save/(prefix+"comparison-latest.png"), save/(prefix+"comparison-best.png"))\n            base.atomic_json(save/"evaluation-best.json", {**report, "step": step, "selected": True})\n        result = {**report, "step": step, "stale": stale, "selected": selected, "guard": passed, "score": candidate}\n        base.atomic_json(save/"evaluation-latest.json", result)\n        with (save/"training.jsonl").open("a") as log:\n            log.write(json.dumps(result)+"\\n")\n        if config.get("snapshots", True):\n            keep = save/"snapshots"\n            keep.mkdir(exist_ok=True)\n            save_snapshot(keep/f"step-{step:05d}.pt", model, step, contract, result)\n            for prefix in ("", "detail-"):\n                shutil.copyfile(save/(prefix+"comparison-latest.png"), keep/f"{prefix}comparison-step-{step:05d}.png")\n        print(f"検証step={step}: 質感距離={report[\'style\']:.4f}, 構造距離={report[\'structure\']:.4f}, "\n              f"細部量/正解={report[\'detail_ratio\']:.3f}, "\n              + (f"細部の形の距離={report.get(\'fine\', float(\'nan\')):.4f}, distillからの変化={report[\'change\']*255:.2f}段階, "\n                 f"起伏の量/正解={report[\'mid_ratio\']:.2f}, 二重の輪郭={report[\'ghost\']:.3f}, "\n                 f"しわの向きの一致={report[\'orient\']:.3f}（開始時{reference.get(\'orient\', float(\'nan\')):.3f}）, "\n                 f"細部(17px以下)の量/目標={report.get(\'fine_ratio\', float(\'nan\')):.2f}, " if \'fine\' in report or CLEAN_UNET else "")\n              + (f"先生（SwiftVR）との一致={report.get(\'teacher_corr\', float(\'nan\')):.3f}, " if V19 else "")\n              + f"チラつき={report[\'flicker\']:.5f}（正解{report[\'flicker_truth\']:.5f}）, 色ずれ={report[\'low\']:.5f}"\n              f"（開始時{reference[\'low\']:.5f}）, 相関(参考)={report[\'detail_corr\']:.3f}, "\n              f"細かい帯域の相関={report[\'fine_corr\']:.3f}（開始時{reference.get(\'fine_corr\', float(\'nan\')):.3f}）, PSNR={report[\'psnr\']:.3f}, "\n              f"guard={passed}, stale={stale}, best更新={selected}", flush=True)\n        save_checkpoint(latest, model, discriminator, optimizers, step, contract, reference, best_score, stale)\n        model.train()\n        model.distill.eval()\n\n    try:\n        model.train()\n        model.distill.eval()\n        while step < max_steps and time.monotonic()-started < minutes*60:\n            for batch in loader:\n                if step >= max_steps or time.monotonic()-started >= minutes*60:\n                    break\n                restored, mosaic, target, mask = [x.to(device) for x in batch[:4]]\n                swift = batch[4].to(device) if V19 else None\n                roi = up_masks(mask[:, 2:5], target.shape[-1]) if context else mask[:, 2:5]\n                teacher = None\n                if typed:\n                    # Texture type of the real texture of each window centre (bank codebook).\n                    with torch.no_grad():\n                        nb, nt = target.shape[:2]\n                        descriptors = texture_descriptor(features, target.reshape(-1, 3, *target.shape[-2:]),\n                                                         roi.reshape(-1, 1, *roi.shape[-2:]))\n                        teacher = texture_type_targets(book, descriptors.float())\n                scale = random.choice(scales)\n                type_probs = teacher if (typed and random.random() < teacher_rate) else None\n                with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=bf16):\n                    prediction, coarse = sequence_v6(\n                        model, restored, mosaic, mask, scale, flow_model=flow_model, pool=pool,\n                        type_probs=type_probs)\n                prediction, coarse = prediction.float(), coarse.float()\n                if V16:\n                    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=bf16):\n                        start_prediction, _ = sequence_v6(start_model, restored, mosaic, mask, scale,\n                                                          flow_model=flow_model, pool=pool, type_probs=type_probs)\n                    start_prediction = start_prediction.float()\n                if HIGHLIGHT_AWARE:\n                    glints = torch.maximum(highlight_map(coarse), highlight_map(target)).detach()\n                flows, valid = motion(flow_model, coarse)\n                strength = gan_weight*min(1., max(0., (step-warmup)/ramp))\n                if strength > 0:\n                    discriminator.train().requires_grad_(True)\n                    optimizers[1].zero_grad(set_to_none=True)\n                    # REAL = uncensored texture (patch targets, plus bank crops):\n                    # the discriminator sees only the high band and the mask.\n                    # REAL texture at the goal amount: the truth\'s high band x gain.\n                    real = texture_goal(target, gain, FINE_GAIN_BAND or band, MID_GAIN, band)\n                    real_logits, real_support = judge(real, roi, coarse)\n                    if HIGHLIGHT_AWARE:\n                        real_logits, real_support = drop_highlights((real_logits, real_support), glints)\n                    real_term = paired.weighted_mean(F.relu(1-real_logits), real_support)\n                    if bank:\n                        texture, texture_mask = bank.batch(target.shape[0]*target.shape[1], device)\n                        bank_glints = highlight_map(texture[:, None]) if HIGHLIGHT_AWARE else None\n                        texture = texture_goal(texture, gain, FINE_GAIN_BAND or band, MID_GAIN, band)\n                        bank_logits, bank_support = judge(texture, texture_mask)\n                        if HIGHLIGHT_AWARE:\n                            bank_logits, bank_support = drop_highlights((bank_logits, bank_support), bank_glints)\n                        real_term = ((1-bank_weight)*real_term\n                                     + bank_weight*paired.weighted_mean(F.relu(1-bank_logits), bank_support))\n                    fake_logits, fake_support = judge(prediction.detach(), roi, coarse)\n                    if HIGHLIGHT_AWARE:\n                        fake_logits, fake_support = drop_highlights((fake_logits, fake_support), glints)\n                    d_loss = real_term+paired.weighted_mean(F.relu(1+fake_logits), fake_support)\n                    if not torch.isfinite(d_loss):\n                        raise RuntimeError("判別器損失が非有限です。")\n                    d_loss.backward()\n                    torch.nn.utils.clip_grad_norm_(discriminator.parameters(), 1., error_if_nonfinite=True)\n                    optimizers[1].step()\n                    discriminator.eval().requires_grad_(False)\n                optimizers[0].zero_grad(set_to_none=True)\n                m = metrics_v4(prediction, target, roi, coarse, flows, valid, features, band, gain, FINE_GAIN_BAND)\n                if context and not CLEAN_UNET:\n                    goal, goal_gain = ((texture_goal(target, gain, FINE_GAIN_BAND, MID_GAIN, JUDGE_BAND), 1.) if FINE_GAIN_BAND else (target, gain))\n                    m["fine"] = fine_structure(features, prediction, texture_target(prediction, goal, roi, band, goal_gain), roi)\n                if "misalign" in weights:\n                    m["misalign"] = 1-m["orient"]\n                if V19:\n                    m["teacher"] = teacher_loss(prediction, swift, coarse, roi)\n                if V16:\n                    now = fine_layer_amount(prediction-coarse)\n                    before = fine_layer_amount(start_prediction-coarse).detach()\n                    h, w = roi.shape[-2:]\n                    deep = (mask_fade(roi.reshape(-1, 1, h, w).float(), 64).reshape_as(roi) >= .75).float()\n                    skin = deep*(1-glints)\n                    m["keep"] = paired.weighted_mean(F.relu(before-now), skin)/(paired.weighted_mean(before, skin)+1e-4)\n                    m["highlight"] = (paired.weighted_mean(F.relu(now-before), glints*roi)\n                                      /(paired.weighted_mean(before, glints*roi)+1e-4))\n                    m["temporal"] = motion_change(prediction-coarse, flows, valid, roi, 17)\n                loss = sum(weights[k]*m[k] for k in weights)\n                if typed:\n                    # The type head learns to name the bank type of the real texture.\n                    logits = model.last_type_logits.float()\n                    support = F.interpolate(roi.reshape(-1, 1, *roi.shape[-2:]).float(), size=logits.shape[-2:], mode="area")\n                    ce = -(teacher[..., None, None]*torch.log_softmax(logits, 1)).sum(1, keepdim=True)\n                    m["type"] = paired.weighted_mean(ce, support)\n                    loss = loss+type_weight*m["type"]\n                if strength > 0:\n                    logits, support = judge(prediction, roi, coarse)\n                    if HIGHLIGHT_AWARE:\n                        logits, support = drop_highlights((logits, support), glints)\n                    loss = loss-strength*paired.weighted_mean(logits, support)\n                if not torch.isfinite(loss):\n                    raise RuntimeError("生成器損失が非有限です。更新せず停止します。")\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(trainable, 1., error_if_nonfinite=True)\n                optimizers[0].step()\n                step += 1\n                if step % 100 == 0:\n                    print(f"step {step}/{max_steps}, loss={float(loss.detach()):.5f}, GAN強度={strength:.4f}, "\n                          f"質感距離={float(m[\'style\'].detach()):.4f}, 構造距離={float(m[\'structure\'].detach()):.4f}, "\n                          f"細部量/正解={float(m[\'detail_ratio\'].detach()):.3f}, "\n                          f"チラつき={float(m[\'flicker\'].detach()):.5f}, v1からの色ずれ={float(m[\'anchor\'].detach()):.5f}, "\n                          + (f"種類の当て={float(m[\'type\'].detach()):.3f}, " if \'type\' in m else "")\n                          + (f"細部の形の距離={float(m.get(\'fine\', m[\'fine_amp\']).detach()):.4f}, 相関={float(m[\'detail_corr\'].detach()):.3f}, "\n                             f"二重の輪郭={float(m[\'ghost\'].detach()):.3f}, 向きの一致={float(m[\'orient\'].detach()):.3f}, "\n                             f"細部の量/目標={float(m[\'fine_ratio\'].detach()):.2f}, " if \'fine\' in m or CLEAN_UNET else "")\n                          + (f"先生との差={float(m[\'teacher\'].detach()):.3f}, " if V19 else "")\n                          + (f"減量={float(m[\'keep\'].detach()):.3f}, ハイライト増={float(m[\'highlight\'].detach()):.3f}, "\n                             f"追加層の揺れ={float(m[\'temporal\'].detach()):.5f}, " if V16 else "")\n                          + f"{(time.monotonic()-started)/60:.1f}分", flush=True)\n                if step % interval == 0:\n                    validate()\n                    last_eval = step\n                    if config.get("early_stopping", True) and stale >= int(config.get("patience", 8)):\n                        print("検証の改善が止まりました。bestを保持します。", flush=True)\n                        return {"step": step, "stale": stale, "best_score": best_score}\n        if step != last_eval:\n            validate()\n    finally:\n        save_checkpoint(latest, model, discriminator, optimizers, step, contract, reference, best_score, stale)\n    return {"step": step, "stale": stale, "best_score": best_score}\n\n\ndef test_best(config):\n    """Held-out test on explicit request only; never selects best."""\n    distill_path = Path(config["distill_checkpoint"]).resolve()\n    prepared, initial, save, _ = validate_paths({**config, "init_checkpoint": str(distill_path)})\n    manifest, manifest_hash = read_manifest(prepared)\n    state = torch.load(save/"best.pt", map_location="cpu", weights_only=True)\n    if (state.get("version") != VERSION or state["contract"]["manifest"] != manifest_hash\n            or state["contract"]["distill"] != paired.sha256(initial)):\n        raise ValueError("best.ptと評価素材・distillが一致しません。")\n    device = torch.device(config.get("device", "cuda"))\n    model = texture_distill_model(load_distill(initial), state["detail_architecture"]).to(device)\n    model.load_state_dict(state["model"], strict=True)\n    root = paired.restore_patches(config, manifest)\n    report = evaluate_v6(model, paired.Patches(root, manifest, "test"), device, load_flow(device),\n                         load_features(device), state["contract"]["eval_samples"], save/"comparison-test.png",\n                         state["contract"]["texture_side"], state["contract"]["eval_grid_scale"])\n    report.update(step=state["step"], split="test", app_integrated=False)\n    base.atomic_json(save/"evaluation-test.json", report)\n    return report\n', 'roi_material_retrieval_runtime.py': '"""Leakage-safe retrieval pilot for whole-ROI material references.\n\nThe deployable query sees only ROI-Distill output and its mask.  A training-only\nkey encoder sees the paired clear target.  The two encoders are aligned with a\ncontrastive objective, then a vector bank is built from *training videos only*.\n\nThis module deliberately stops at retrieval validation.  It does not paste a\nforeign image, generate detail, or integrate anything into mioh.  A retrieved\nrecord is useful only if a held-out query selects training references whose\nclear-image material statistics are closer than random references.\n"""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport time\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nimport roi_mask_detail_runtime as paired\nimport roi_mask_gan_runtime as gan\n\n\nVERSION = "roi-material-retrieval-v1"\nEMBEDDING_DIM = 128\nDESCRIPTOR_DIM = 14\n\n\ndef weighted_mean(value, weight):\n    weight = weight.expand_as(value)\n    return (value * weight).sum((-2, -1)) / weight.sum((-2, -1)).clamp_min(1e-6)\n\n\ndef luma(value):\n    return .299 * value[..., 0:1, :, :] + .587 * value[..., 1:2, :, :] + .114 * value[..., 2:3, :, :]\n\n\ndef _blur(value, side):\n    shape = value.shape\n    flat = value.reshape(-1, shape[-3], *shape[-2:])\n    result = gan.blur(flat.float(), side)\n    return result.reshape(shape)\n\n\ndef material_descriptor(target, mask):\n    """Fourteen interpretable whole-ROI statistics, averaged over three frames.\n\n    The vector describes colour, contrast and spatial-frequency amount.  It is\n    used only as an external retrieval check and regression target; it never\n    identifies the positive pair for the contrastive loss.\n    """\n    target, mask = target.float(), mask.float()\n    if target.ndim != 5 or mask.shape[:2] != target.shape[:2]:\n        raise ValueError("target/mask must be [B,T,C,H,W] with matching B,T")\n    mean = weighted_mean(target, mask)\n    variance = weighted_mean((target - mean[..., None, None]).square(), mask).sqrt()\n    y = luma(target)\n    low5, low17, low65 = _blur(y, 5), _blur(y, 17), _blur(y, 65)\n    bands = (y - low5, low5 - low17, low17 - low65)\n    band_rms = [weighted_mean(value.square(), mask).sqrt() * 10 for value in bands]\n    band_abs = [weighted_mean(value.abs(), mask) * 10 for value in bands]\n    gx = (y[..., :, 1:] - y[..., :, :-1]).abs()\n    gy = (y[..., 1:, :] - y[..., :-1, :]).abs()\n    wx = torch.minimum(mask[..., :, 1:], mask[..., :, :-1])\n    wy = torch.minimum(mask[..., 1:, :], mask[..., :-1, :])\n    gradient = torch.cat((weighted_mean(gx, wx) * 10,\n                          weighted_mean(gy, wy) * 10), -1)\n    values = torch.cat((mean, variance, *band_rms, *band_abs, gradient), -1)\n    return values.mean(1)\n\n\ndef mask_geometry(mask):\n    """Area, centroid and spread; all values are normalized to roughly 0..1."""\n    mask = mask.float()\n    batch, frames, _, height, width = mask.shape\n    y = torch.linspace(-1, 1, height, device=mask.device, dtype=mask.dtype).view(1, 1, 1, height, 1)\n    x = torch.linspace(-1, 1, width, device=mask.device, dtype=mask.dtype).view(1, 1, 1, 1, width)\n    mass = mask.sum((-2, -1)).clamp_min(1e-6)\n    cx = (mask * x).sum((-2, -1)) / mass\n    cy = (mask * y).sum((-2, -1)) / mass\n    sx = ((mask * (x - cx[..., None, None]).square()).sum((-2, -1)) / mass).sqrt()\n    sy = ((mask * (y - cy[..., None, None]).square()).sum((-2, -1)) / mass).sqrt()\n    area = mass / (height * width)\n    return torch.cat((area, cx, cy, sx, sy), -1).mean(1)\n\n\nclass ResidualBlock(nn.Module):\n    def __init__(self, channels):\n        super().__init__()\n        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)\n        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)\n\n    def forward(self, value):\n        return value + .1 * self.conv2(F.silu(self.conv1(value)))\n\n\nclass WholeROIEncoder(nn.Module):\n    """Encode full context, masked appearance and mask for a 3-frame track."""\n    def __init__(self, embedding_dim=EMBEDDING_DIM):\n        super().__init__()\n        self.query_stem = nn.Conv2d(7, 32, 5, stride=2, padding=2)\n        self.key_stem = nn.Conv2d(7, 32, 5, stride=2, padding=2)\n        self.shared = nn.Sequential(\n            nn.SiLU(), ResidualBlock(32),\n            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.SiLU(), ResidualBlock(64),\n            nn.Conv2d(64, 96, 3, stride=2, padding=1), nn.SiLU(), ResidualBlock(96),\n            nn.Conv2d(96, 128, 3, stride=2, padding=1), nn.SiLU(), ResidualBlock(128),\n        )\n        self.project = nn.Sequential(nn.Linear(128 * 3 + 5, 256), nn.SiLU(),\n                                     nn.Linear(256, embedding_dim))\n        self.descriptor = nn.Sequential(nn.Linear(embedding_dim, 64), nn.SiLU(),\n                                        nn.Linear(64, DESCRIPTOR_DIM))\n\n    @staticmethod\n    def _input(image, mask, side=128):\n        shape = image.shape\n        flat_image = image.reshape(-1, 3, *shape[-2:]).float()\n        flat_mask = mask.reshape(-1, 1, *mask.shape[-2:]).float()\n        if flat_mask.shape[-2:] != flat_image.shape[-2:]:\n            flat_mask = F.interpolate(flat_mask, size=flat_image.shape[-2:], mode="area")\n        value = torch.cat((flat_image, flat_image * flat_mask, flat_mask), 1)\n        return F.interpolate(value, size=(side, side), mode="area"), flat_mask.reshape(\n            shape[0], shape[1], 1, *flat_image.shape[-2:])\n\n    def encode(self, image, mask, domain):\n        if image.shape[1] != 3:\n            raise ValueError("検索ベクトルは3フレーム単位です。")\n        value, full_mask = self._input(image, mask)\n        stem = self.query_stem if domain == "query" else self.key_stem if domain == "key" else None\n        if stem is None:\n            raise ValueError("domain must be query or key")\n        feature = self.shared(stem(value)).mean((-2, -1))\n        feature = feature.reshape(image.shape[0], image.shape[1], -1)\n        summary = torch.cat((feature.mean(1), feature.std(1, unbiased=False), feature[:, 1],\n                             mask_geometry(full_mask)), -1)\n        return F.normalize(self.project(summary), dim=-1)\n\n    def query(self, coarse, mask):\n        return self.encode(coarse, mask, "query")\n\n    def key(self, target, mask):\n        return self.encode(target, mask, "key")\n\n\nclass EmbeddingQueue:\n    def __init__(self, capacity, dimension, device):\n        self.capacity = int(capacity)\n        self.q = torch.empty((0, dimension), device=device)\n        self.k = torch.empty((0, dimension), device=device)\n\n    def push(self, query, key):\n        self.q = torch.cat((self.q, query.detach()), 0)[-self.capacity:]\n        self.k = torch.cat((self.k, key.detach()), 0)[-self.capacity:]\n\n    def state_dict(self):\n        return {"q": self.q.cpu(), "k": self.k.cpu(), "capacity": self.capacity}\n\n    def load_state_dict(self, state, device):\n        if int(state["capacity"]) != self.capacity:\n            raise ValueError("検索queueの容量が異なります。")\n        self.q, self.k = state["q"].to(device), state["k"].to(device)\n\n\ndef contrastive_loss(query, key, queue, temperature=.07):\n    """CLIP-style paired alignment with cross-shard memory negatives."""\n    if query.shape != key.shape or query.ndim != 2:\n        raise ValueError("query/key embeddings must have the same [B,D] shape")\n    batch = query.shape[0]\n    all_keys = torch.cat((key, queue.k), 0)\n    all_queries = torch.cat((query, queue.q), 0)\n    labels = torch.arange(batch, device=query.device)\n    q_to_k = F.cross_entropy(query @ all_keys.T / temperature, labels)\n    k_to_q = F.cross_entropy(key @ all_queries.T / temperature, labels)\n    return .5 * (q_to_k + k_to_q)\n\n\ndef coarse_sequence(distill, restored, masks):\n    batch, _, _, height, width = restored.shape\n    windows = torch.stack([restored[:, index:index + 5] for index in range(3)], 1)\n    enhanced = distill(windows.reshape(batch * 3, 15, height, width))\n    enhanced = enhanced.reshape(batch, 3, 3, height * 2, width * 2)\n    under = F.interpolate(restored[:, 2:5].reshape(batch * 3, 3, height, width),\n                          scale_factor=2., mode="bilinear", align_corners=False).reshape_as(enhanced)\n    roi = gan.up_masks(masks[:, 2:5], enhanced.shape[-1])\n    return under + roi * (enhanced - under), roi\n\n\ndef _batch(dataset, indices):\n    values = [dataset[int(index)] for index in indices]\n    return tuple(torch.stack([sample[column] for sample in values]) for column in range(4))\n\n\ndef encode_dataset(model, distill, dataset, device, limit=0, batch_size=8):\n    count = len(dataset) if not limit else min(len(dataset), int(limit))\n    indices = np.linspace(0, len(dataset) - 1, count).round().astype(int)\n    embeddings, keys, descriptors, predictions = [], [], [], []\n    model.eval()\n    original_augment = dataset.augment\n    dataset.augment = False\n    try:\n        with torch.inference_mode():\n            for first in range(0, len(indices), batch_size):\n                selected = indices[first:first + batch_size]\n                restored, _, target, mask = [value.to(device) for value in _batch(dataset, selected)]\n                coarse, roi = coarse_sequence(distill, restored, mask)\n                query, key = model.query(coarse, roi), model.key(target, roi)\n                embeddings.append(query.cpu())\n                keys.append(key.cpu())\n                descriptors.append(material_descriptor(target, roi).cpu())\n                predictions.append(model.descriptor(query).cpu())\n    finally:\n        dataset.augment = original_augment\n    return {"indices": torch.from_numpy(indices), "query": torch.cat(embeddings),\n            "key": torch.cat(keys), "descriptor": torch.cat(descriptors),\n            "predicted_descriptor": torch.cat(predictions)}\n\n\ndef _reference(dataset, index):\n    restored, _, target, mask = dataset[int(index)]\n    # Preview is intentionally not the retrieval input contract; it is a visual audit.\n    return restored[3], target[1], mask[3]\n\n\ndef _save_preview(path, validation, bank, dataset_val, dataset_train, top_indices, oracle_indices):\n    rows, labels = [], ["restored query", "retrieved truth", "oracle truth", "actual truth", "mask"]\n    for row in range(min(6, len(validation["indices"]))):\n        restored, target, mask = _reference(dataset_val, validation["indices"][row])\n        _, retrieved, _ = _reference(dataset_train, bank["indices"][top_indices[row]])\n        _, oracle, _ = _reference(dataset_train, bank["indices"][oracle_indices[row]])\n        rows.append((restored, retrieved, oracle, target, mask.expand(3, -1, -1)))\n    if not rows:\n        return\n    side, header = 256, 32\n    canvas = Image.new("RGB", (side * len(labels), header + side * len(rows)), "white")\n    draw = ImageDraw.Draw(canvas)\n    for column, label in enumerate(labels):\n        draw.text((column * side + 5, 8), label, fill="black")\n    for row, values in enumerate(rows):\n        for column, value in enumerate(values):\n            value = F.interpolate(value[None].float(), (side, side), mode="bilinear",\n                                  align_corners=False)[0]\n            array = np.rint(value.clamp(0, 1).permute(1, 2, 0).numpy() * 255).astype(np.uint8)\n            canvas.paste(Image.fromarray(array), (column * side, header + row * side))\n    canvas.save(path)\n\n\ndef evaluate_retrieval(model, distill, training, validation, device, output,\n                       bank_samples=2048, eval_samples=96, batch_size=8):\n    """Held-out whole-video retrieval.  No validation target enters query search."""\n    bank = encode_dataset(model, distill, training, device, bank_samples, batch_size)\n    val = encode_dataset(model, distill, validation, device, eval_samples, batch_size)\n    similarity = val["query"] @ bank["key"].T\n    truth_similarity = val["key"] @ bank["key"].T\n    top = similarity.argmax(1)\n    oracle = truth_similarity.argmax(1)\n    generator = torch.Generator().manual_seed(20261014)\n    random_index = torch.randint(len(bank["key"]), (len(val["key"]),), generator=generator)\n    row = torch.arange(len(val["key"]))\n    retrieved_descriptor = bank["descriptor"][top]\n    random_descriptor = bank["descriptor"][random_index]\n    descriptor_mse = (retrieved_descriptor - val["descriptor"]).square().mean()\n    random_mse = (random_descriptor - val["descriptor"]).square().mean()\n    predicted_mse = (val["predicted_descriptor"] - val["descriptor"]).square().mean()\n    retrieved_truth = truth_similarity[row, top].mean()\n    random_truth = truth_similarity[row, random_index].mean()\n    oracle_truth = truth_similarity[row, oracle].mean()\n    top10 = similarity.topk(min(10, similarity.shape[1]), 1).indices\n    oracle_in_top10 = (top10 == oracle[:, None]).any(1).float().mean()\n    report = {\n        "samples": len(val["key"]), "bank_samples": len(bank["key"]),\n        "paired_domain_cosine": float((val["query"] * val["key"]).sum(1).mean()),\n        "retrieved_truth_cosine": float(retrieved_truth),\n        "random_truth_cosine": float(random_truth),\n        "oracle_truth_cosine": float(oracle_truth),\n        "oracle_in_query_top10": float(oracle_in_top10),\n        "descriptor_mse": float(descriptor_mse), "random_descriptor_mse": float(random_mse),\n        "predicted_descriptor_mse": float(predicted_mse),\n        "descriptor_gain_over_random": float(1 - descriptor_mse / random_mse.clamp_min(1e-12)),\n    }\n    report["retrieval_passed"] = bool(\n        report["descriptor_gain_over_random"] >= .10\n        and report["retrieved_truth_cosine"] > report["random_truth_cosine"] + .02)\n    _save_preview(output, val, bank, validation, training, top, oracle)\n    return report, bank\n\n\ndef describe(report):\n    return (f"質感統計改善={100 * report[\'descriptor_gain_over_random\']:.1f}%, "\n            f"正解側類似度={report[\'retrieved_truth_cosine\']:.3f} "\n            f"(random {report[\'random_truth_cosine\']:.3f}, oracle {report[\'oracle_truth_cosine\']:.3f}), "\n            f"oracle@10={100 * report[\'oracle_in_query_top10\']:.1f}%, "\n            f"判定={\'合格\' if report[\'retrieval_passed\'] else \'不合格\'}")\n\n\ndef _atomic_torch(path, value):\n    temporary = Path(path).with_suffix(Path(path).suffix + ".partial")\n    torch.save(value, temporary)\n    os.replace(temporary, path)\n\n\ndef _checkpoint(path, model, optimizer, queue, step, contract, best, stale, report):\n    _atomic_torch(path, {"version": VERSION, "model": model.state_dict(),\n                         "optimizer": optimizer.state_dict(), "queue": queue.state_dict(),\n                         "step": step, "contract": contract, "best_score": best,\n                         "stale": stale, "report": report, "app_integrated": False,\n                         "stage": "retrieval-validation-only"})\n\n\ndef train(config):\n    device = torch.device(config.get("device", "cuda"))\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("GPUランタイムへ接続してください。")\n    save = Path(config["save_root"])\n    save.mkdir(parents=True, exist_ok=True)\n    manifest = config["manifest"]\n    if manifest.get("split_unit") != "whole-video-file":\n        raise ValueError("検索評価は動画単位で分離された素材だけを使用します。")\n    root = paired.restore_patches(config, manifest)\n    training = paired.Patches(root, manifest, "train")\n    validation = paired.Patches(root, manifest, "val")\n    distill = gan.load_distill(config["distill_checkpoint"]).to(device).eval().requires_grad_(False)\n    model = WholeROIEncoder(int(config.get("embedding_dim", EMBEDDING_DIM))).to(device)\n    max_steps = int(config.get("max_steps", 2500))\n    interval = int(config.get("eval_interval", 250))\n    batch_size = int(config.get("batch_size", 8))\n    lr = float(config.get("learning_rate", 2e-4))\n    queue_size = int(config.get("queue_size", 1024))\n    patience = int(config.get("patience", 8))\n    contract = {"version": VERSION, "manifest": paired.digest(manifest),\n                "distill": paired.sha256(config["distill_checkpoint"]),\n                "embedding_dim": model.project[-1].out_features,\n                "max_steps": max_steps, "batch_size": batch_size,\n                "learning_rate": lr, "queue_size": queue_size,\n                "train_split_only_bank": True, "whole_video_holdout": True,\n                "seed": int(config.get("seed", 20261014))}\n    random.seed(contract["seed"])\n    np.random.seed(contract["seed"])\n    torch.manual_seed(contract["seed"])\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(.9, .99), weight_decay=1e-4)\n    queue = EmbeddingQueue(queue_size, contract["embedding_dim"], device)\n    step, stale, best = 0, 0, float("inf")\n    latest = save / "latest.pt"\n    if latest.is_file():\n        state = torch.load(latest, map_location="cpu", weights_only=True)\n        if state.get("contract") != contract:\n            raise ValueError("再開条件が異なります。新しいSAVE_ROOTを指定してください。")\n        model.load_state_dict(state["model"])\n        optimizer.load_state_dict(state["optimizer"])\n        queue.load_state_dict(state["queue"], device)\n        step, stale, best = int(state["step"]), int(state["stale"]), float(state["best_score"])\n    elif any(save.iterdir()):\n        raise ValueError("新規検索実験のSAVE_ROOTが空ではありません。")\n\n    loader = torch.utils.data.DataLoader(\n        training, batch_sampler=paired.ShardBatches(training, batch_size), num_workers=0)\n    started = time.monotonic()\n    while step < max_steps:\n        model.train()\n        for restored, _, target, mask in loader:\n            if step >= max_steps:\n                break\n            restored, target, mask = [value.to(device) for value in (restored, target, mask)]\n            with torch.no_grad():\n                coarse, roi = coarse_sequence(distill, restored, mask)\n                descriptor = material_descriptor(target, roi)\n            query, key = model.query(coarse, roi), model.key(target, roi)\n            alignment = contrastive_loss(query, key, queue)\n            prediction_q, prediction_k = model.descriptor(query), model.descriptor(key)\n            regression_q = F.smooth_l1_loss(prediction_q, descriptor, beta=.02)\n            regression_k = F.smooth_l1_loss(prediction_k, descriptor, beta=.02)\n            loss = alignment + regression_q + .25 * regression_k\n            if not torch.isfinite(loss):\n                raise RuntimeError("検索モデルの損失が非有限です。")\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)\n            optimizer.step()\n            queue.push(query, key)\n            step += 1\n            if step % int(config.get("log_interval", 50)) == 0:\n                print(f"step {step}/{max_steps}: loss={float(loss):.4f}, "\n                      f"検索={float(alignment):.4f}, 質感予測={float(regression_q):.5f}, "\n                      f"queue={len(queue.k)}, {(time.monotonic() - started) / 60:.1f}分", flush=True)\n            if step % interval == 0:\n                report, bank = evaluate_retrieval(\n                    model, distill, training, validation, device,\n                    save / "comparison-latest.png",\n                    int(config.get("bank_samples", 2048)),\n                    int(config.get("eval_samples", 96)),\n                    int(config.get("eval_batch_size", 8)))\n                score = ((1 - report["retrieved_truth_cosine"])\n                         + report["descriptor_mse"] / max(report["random_descriptor_mse"], 1e-12))\n                selected = score < best\n                if selected:\n                    best, stale = score, 0\n                    _checkpoint(save / "best.pt", model, optimizer, queue, step,\n                                contract, best, stale, report)\n                    shutil.copyfile(save / "comparison-latest.png", save / "comparison-best.png")\n                    records = [{"dataset_index": int(index),\n                                "shard": training.items[int(index)][0],\n                                "offset": int(training.items[int(index)][1])}\n                               for index in bank["indices"]]\n                    _atomic_torch(save / "bank.pt", {\n                        "version": VERSION, "step": step, "manifest": contract["manifest"],\n                        "key": bank["key"].half(), "descriptor": bank["descriptor"].half(),\n                        "records": records, "contains_raw_images": False,\n                        "split": "train"})\n                else:\n                    stale += 1\n                _checkpoint(latest, model, optimizer, queue, step, contract, best, stale, report)\n                line = dict(report, step=step, score=score, selected=selected, stale=stale)\n                with (save / "training.jsonl").open("a") as handle:\n                    handle.write(json.dumps(line, ensure_ascii=False) + "\\n")\n                print(f"検証 step={step}: {describe(report)}, best更新={selected}, stale={stale}", flush=True)\n                if stale >= patience:\n                    return {"step": step, "best_score": best, "stale": stale,\n                            "retrieval_passed": report["retrieval_passed"]}\n    return {"step": step, "best_score": best, "stale": stale,\n            "retrieval_passed": report["retrieval_passed"]}\n', 'roi_material_retrieval_v2_runtime.py': '"""Stable second-stage training for the whole-ROI material retrieval pilot.\n\nStage v1 proved that a query can predict useful material statistics, but its\nnormalized embedding repeatedly collapsed to almost one direction.  This stage\nwarm-starts the v1 feature extractor while changing the objective:\n\n* only low-frequency appearance enters the retrieval embedding;\n* descriptor-neighbour samples are positives instead of forcing an unknowable\n  exact high-frequency match;\n* variance/covariance regularisation keeps the embedding informative;\n* visual top-K results are reranked with the deployable descriptor prediction.\n\nThe stage still validates retrieval only.  It does not copy pixels or integrate\nthe bank into mioh.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nimport roi_mask_detail_runtime as paired\nimport roi_mask_gan_runtime as gan\nimport roi_material_retrieval_runtime as v1\n\n\nVERSION = "roi-material-retrieval-v2"\n\n\nclass StableWholeROIEncoder(v1.WholeROIEncoder):\n    """Keep the v1 parameter layout but remove inaccessible fine detail.\n\n    The train-only key image has genuine fine texture while the deployable query\n    does not.  Letting the key stem see that detail creates a shortcut which the\n    query can never reproduce.  Both domains therefore see the same 17-pixel\n    low-pass representation, masked appearance and mask geometry.\n    """\n\n    @staticmethod\n    def _input(image, mask, side=128):\n        shape = image.shape\n        image = v1._blur(image.float(), 17)\n        flat_image = image.reshape(-1, 3, *shape[-2:])\n        flat_mask = mask.reshape(-1, 1, *mask.shape[-2:]).float()\n        if flat_mask.shape[-2:] != flat_image.shape[-2:]:\n            flat_mask = F.interpolate(flat_mask, size=flat_image.shape[-2:], mode="area")\n        value = torch.cat((flat_image, flat_image * flat_mask, flat_mask), 1)\n        return F.interpolate(value, size=(side, side), mode="area"), flat_mask.reshape(\n            shape[0], shape[1], 1, *flat_image.shape[-2:])\n\n\nclass SemanticQueue:\n    def __init__(self, capacity, dimension, descriptor_dimension, device):\n        self.capacity = int(capacity)\n        self.q = torch.empty((0, dimension), device=device)\n        self.k = torch.empty((0, dimension), device=device)\n        self.d = torch.empty((0, descriptor_dimension), device=device)\n\n    def push(self, query, key, descriptor):\n        self.q = torch.cat((self.q, query.detach()), 0)[-self.capacity:]\n        self.k = torch.cat((self.k, key.detach()), 0)[-self.capacity:]\n        self.d = torch.cat((self.d, descriptor.detach()), 0)[-self.capacity:]\n\n    def state_dict(self):\n        return {"q": self.q.cpu(), "k": self.k.cpu(), "d": self.d.cpu(),\n                "capacity": self.capacity}\n\n    def load_state_dict(self, state, device):\n        if int(state["capacity"]) != self.capacity:\n            raise ValueError("検索queueの容量が異なります。")\n        self.q = state["q"].to(device)\n        self.k = state["k"].to(device)\n        self.d = state["d"].to(device)\n\n\ndef standardized_descriptor_distance(left, right, reference):\n    """Mean squared descriptor distance after per-field robust scaling."""\n    scale = reference.float().std(0, unbiased=False).clamp_min(.01)\n    return ((left.float()[:, None] - right.float()[None]) / scale).square().mean(-1)\n\n\ndef semantic_contrastive_loss(query, key, descriptor, queue, temperature=.10,\n                              descriptor_temperature=.50):\n    """Soft contrastive loss whose neighbours share observable material state."""\n    if query.shape != key.shape or len(query) != len(descriptor):\n        raise ValueError("query/key/descriptor batch does not match")\n    batch = len(query)\n    all_key = torch.cat((key, queue.k), 0)\n    all_query = torch.cat((query, queue.q), 0)\n    all_descriptor = torch.cat((descriptor, queue.d), 0)\n    distance = standardized_descriptor_distance(descriptor, all_descriptor, all_descriptor)\n    target = torch.exp(-distance / descriptor_temperature)\n    target[:, :batch] += torch.eye(batch, device=query.device) * 2.\n    target = target / target.sum(1, keepdim=True).clamp_min(1e-6)\n    q_to_k = -(target * F.log_softmax(query @ all_key.T / temperature, 1)).sum(1).mean()\n    k_to_q = -(target * F.log_softmax(key @ all_query.T / temperature, 1)).sum(1).mean()\n    return .5 * (q_to_k + k_to_q)\n\n\ndef embedding_spread_loss(query, key, target_std=.045):\n    """VICReg-style anti-collapse penalty for unit-normalized embeddings."""\n    value = torch.cat((query, key), 0)\n    centered = value - value.mean(0, keepdim=True)\n    std = (centered.square().mean(0) + 1e-4).sqrt()\n    variance = F.relu(float(target_std) - std).mean()\n    covariance = centered.T @ centered / max(len(centered) - 1, 1)\n    covariance = covariance - torch.diag(torch.diagonal(covariance))\n    return variance + .05 * covariance.square().mean(), std.mean()\n\n\ndef rerank_candidates(query, bank_key, predicted_descriptor, bank_descriptor,\n                      top_k=64, descriptor_weight=.35):\n    """Visual candidate gate followed by deployable material-stat reranking."""\n    similarity = query.float() @ bank_key.float().T\n    count = min(int(top_k), similarity.shape[1])\n    candidates = similarity.topk(count, 1).indices\n    candidate_similarity = similarity.gather(1, candidates)\n    scale = bank_descriptor.float().std(0, unbiased=False).clamp_min(.01)\n    selected_descriptor = bank_descriptor.float()[candidates]\n    distance = ((selected_descriptor - predicted_descriptor.float()[:, None]) / scale).square().mean(-1)\n    # Keep visual geometry as a gate, but let the predicted material state decide\n    # between plausible candidates.  Centering makes the weight stable when the\n    # cosine range changes during training.\n    visual = candidate_similarity - candidate_similarity.mean(1, keepdim=True)\n    score = visual - float(descriptor_weight) * distance\n    order = score.argsort(1, descending=True)\n    return candidates.gather(1, order), similarity\n\n\ndef encode_dataset(model, distill, dataset, device, limit=0, batch_size=8):\n    return v1.encode_dataset(model, distill, dataset, device, limit, batch_size)\n\n\ndef _save_preview(path, validation, bank, dataset_val, dataset_train,\n                  reranked_indices, visual_indices, descriptor_indices):\n    rows = []\n    labels = ["restored query", "reranked truth", "visual truth",\n              "descriptor oracle", "actual truth", "mask"]\n    for row in range(min(6, len(validation["indices"]))):\n        restored, target, mask = v1._reference(dataset_val, validation["indices"][row])\n        _, reranked, _ = v1._reference(\n            dataset_train, bank["indices"][reranked_indices[row]])\n        _, visual, _ = v1._reference(\n            dataset_train, bank["indices"][visual_indices[row]])\n        _, oracle, _ = v1._reference(\n            dataset_train, bank["indices"][descriptor_indices[row]])\n        rows.append((restored, reranked, visual, oracle, target,\n                     mask.expand(3, -1, -1)))\n    if not rows:\n        return\n    from PIL import Image, ImageDraw\n    side, header = 256, 32\n    canvas = Image.new("RGB", (side * len(labels), header + side * len(rows)), "white")\n    draw = ImageDraw.Draw(canvas)\n    for column, label in enumerate(labels):\n        draw.text((column * side + 5, 8), label, fill="black")\n    for row, values in enumerate(rows):\n        for column, value in enumerate(values):\n            value = F.interpolate(value[None].float(), (side, side), mode="bilinear",\n                                  align_corners=False)[0]\n            array = np.rint(value.clamp(0, 1).permute(1, 2, 0).numpy() * 255).astype(np.uint8)\n            canvas.paste(Image.fromarray(array), (column * side, header + row * side))\n    canvas.save(path)\n\n\ndef evaluate_retrieval(model, distill, training, validation, device, output,\n                       bank_samples=2048, eval_samples=96, batch_size=8,\n                       rerank_top_k=64, descriptor_weight=.35):\n    bank = encode_dataset(model, distill, training, device, bank_samples, batch_size)\n    val = encode_dataset(model, distill, validation, device, eval_samples, batch_size)\n    reranked, similarity = rerank_candidates(\n        val["query"], bank["key"], val["predicted_descriptor"], bank["descriptor"],\n        rerank_top_k, descriptor_weight)\n    visual = similarity.argmax(1)\n    top = reranked[:, 0]\n    true_distance = standardized_descriptor_distance(\n        val["descriptor"], bank["descriptor"], bank["descriptor"])\n    descriptor_oracle = true_distance.argmin(1)\n    descriptor_neighbours = true_distance.topk(min(10, true_distance.shape[1]), 1,\n                                                largest=False).indices\n    recall10 = (reranked[:, :min(10, reranked.shape[1])][:, :, None]\n                == descriptor_neighbours[:, None, :]).any((1, 2)).float().mean()\n    generator = torch.Generator().manual_seed(20261014)\n    random_index = torch.randint(len(bank["key"]), (len(val["key"]),), generator=generator)\n    row = torch.arange(len(val["key"]))\n    descriptor_mse = (bank["descriptor"][top] - val["descriptor"]).square().mean()\n    visual_descriptor_mse = (bank["descriptor"][visual] - val["descriptor"]).square().mean()\n    random_mse = (bank["descriptor"][random_index] - val["descriptor"]).square().mean()\n    predicted_mse = (val["predicted_descriptor"] - val["descriptor"]).square().mean()\n    truth_similarity = val["key"] @ bank["key"].T\n    pairwise = val["query"] @ val["query"].T\n    off_diagonal = pairwise[~torch.eye(len(pairwise), dtype=torch.bool)]\n    dimension_std = val["query"].std(0, unbiased=False)\n    report = {\n        "samples": len(val["key"]), "bank_samples": len(bank["key"]),\n        "paired_domain_cosine": float((val["query"] * val["key"]).sum(1).mean()),\n        "reranked_truth_cosine": float(truth_similarity[row, top].mean()),\n        "visual_truth_cosine": float(truth_similarity[row, visual].mean()),\n        "random_truth_cosine": float(truth_similarity[row, random_index].mean()),\n        "descriptor_mse": float(descriptor_mse),\n        "visual_descriptor_mse": float(visual_descriptor_mse),\n        "random_descriptor_mse": float(random_mse),\n        "predicted_descriptor_mse": float(predicted_mse),\n        "descriptor_gain_over_random": float(1 - descriptor_mse / random_mse.clamp_min(1e-12)),\n        "descriptor_gain_over_visual": float(1 - descriptor_mse / visual_descriptor_mse.clamp_min(1e-12)),\n        "descriptor_neighbour_recall10": float(recall10),\n        "embedding_pairwise_cosine": float(off_diagonal.mean()),\n        "embedding_dimension_std": float(dimension_std.mean()),\n        "embedding_min_dimension_std": float(dimension_std.min()),\n    }\n    report["retrieval_passed"] = bool(\n        report["descriptor_gain_over_random"] >= .35\n        and report["descriptor_neighbour_recall10"] >= .15\n        and report["embedding_pairwise_cosine"] < .995\n        and report["embedding_dimension_std"] >= .02)\n    _save_preview(output, val, bank, validation, training, top, visual, descriptor_oracle)\n    return report, bank\n\n\ndef describe(report):\n    return (f"質感統計改善={100 * report[\'descriptor_gain_over_random\']:.1f}% "\n            f"(visual比 {100 * report[\'descriptor_gain_over_visual\']:.1f}%), "\n            f"質感近傍recall@10={100 * report[\'descriptor_neighbour_recall10\']:.1f}%, "\n            f"埋め込みcos={report[\'embedding_pairwise_cosine\']:.3f}, "\n            f"次元std={report[\'embedding_dimension_std\']:.3f}, "\n            f"判定={\'合格\' if report[\'retrieval_passed\'] else \'不合格\'}")\n\n\ndef _atomic_torch(path, value):\n    temporary = Path(path).with_suffix(Path(path).suffix + ".partial")\n    torch.save(value, temporary)\n    os.replace(temporary, path)\n\n\ndef _checkpoint(path, model, optimizer, queue, step, contract, best, stale, report):\n    _atomic_torch(path, {"version": VERSION, "model": model.state_dict(),\n                         "optimizer": optimizer.state_dict(), "queue": queue.state_dict(),\n                         "step": step, "contract": contract, "best_score": best,\n                         "stale": stale, "report": report, "app_integrated": False,\n                         "stage": "stable-reranked-retrieval-validation-only"})\n\n\ndef train(config):\n    device = torch.device(config.get("device", "cuda"))\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("GPUランタイムへ接続してください。")\n    save = Path(config["save_root"])\n    save.mkdir(parents=True, exist_ok=True)\n    manifest = config["manifest"]\n    if manifest.get("split_unit") != "whole-video-file":\n        raise ValueError("検索評価は動画単位で分離された素材だけを使用します。")\n    root = paired.restore_patches(config, manifest)\n    training = paired.Patches(root, manifest, "train")\n    validation = paired.Patches(root, manifest, "val")\n    distill = gan.load_distill(config["distill_checkpoint"]).to(device).eval().requires_grad_(False)\n    model = StableWholeROIEncoder(int(config.get("embedding_dim", v1.EMBEDDING_DIM))).to(device)\n    init_checkpoint = Path(config["init_checkpoint"])\n    max_steps = int(config.get("max_steps", 3000))\n    interval = int(config.get("eval_interval", 250))\n    batch_size = int(config.get("batch_size", 8))\n    lr = float(config.get("learning_rate", 0.00008))\n    queue_size = int(config.get("queue_size", 2048))\n    patience = int(config.get("patience", 8))\n    contract = {"version": VERSION, "manifest": paired.digest(manifest),\n                "distill": paired.sha256(config["distill_checkpoint"]),\n                "init": paired.sha256(init_checkpoint),\n                "embedding_dim": model.project[-1].out_features,\n                "max_steps": max_steps, "batch_size": batch_size,\n                "learning_rate": lr, "queue_size": queue_size,\n                "rerank_top_k": int(config.get("rerank_top_k", 64)),\n                "descriptor_weight": float(config.get("descriptor_weight", .35)),\n                "train_split_only_bank": True, "whole_video_holdout": True,\n                "seed": int(config.get("seed", 20261014))}\n    random.seed(contract["seed"])\n    np.random.seed(contract["seed"])\n    torch.manual_seed(contract["seed"])\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(.9, .99), weight_decay=2e-4)\n    queue = SemanticQueue(queue_size, contract["embedding_dim"], v1.DESCRIPTOR_DIM, device)\n    step, stale, best = 0, 0, float("inf")\n    latest = save / "latest.pt"\n    if latest.is_file():\n        state = torch.load(latest, map_location="cpu", weights_only=True)\n        if state.get("contract") != contract:\n            raise ValueError("再開条件が異なります。新しいSAVE_ROOTを指定してください。")\n        model.load_state_dict(state["model"])\n        optimizer.load_state_dict(state["optimizer"])\n        queue.load_state_dict(state["queue"], device)\n        step, stale, best = int(state["step"]), int(state["stale"]), float(state["best_score"])\n        print(f"v2を再開: step={step}, stale={stale}", flush=True)\n    elif any(save.iterdir()):\n        raise ValueError("新規v2実験のSAVE_ROOTが空ではありません。")\n    else:\n        state = torch.load(init_checkpoint, map_location="cpu", weights_only=True)\n        model.load_state_dict(state["model"], strict=True)\n        print(f"v1 best step={state.get(\'step\')}から特徴抽出器を初期化。optimizerとqueueは新規です。",\n              flush=True)\n\n    loader = torch.utils.data.DataLoader(\n        training, batch_sampler=paired.ShardBatches(training, batch_size), num_workers=0)\n    started = time.monotonic()\n    report = {"retrieval_passed": False}\n    while step < max_steps:\n        model.train()\n        for restored, _, target, mask in loader:\n            if step >= max_steps:\n                break\n            restored, target, mask = [value.to(device) for value in (restored, target, mask)]\n            with torch.no_grad():\n                coarse, roi = v1.coarse_sequence(distill, restored, mask)\n                descriptor = v1.material_descriptor(target, roi)\n            query, key = model.query(coarse, roi), model.key(target, roi)\n            semantic = semantic_contrastive_loss(query, key, descriptor, queue)\n            prediction_q, prediction_k = model.descriptor(query), model.descriptor(key)\n            regression_q = F.smooth_l1_loss(prediction_q, descriptor, beta=.02)\n            regression_k = F.smooth_l1_loss(prediction_k, descriptor, beta=.02)\n            spread, mean_std = embedding_spread_loss(query, key)\n            pair = (1 - (query * key).sum(1)).mean()\n            loss = semantic + regression_q + .25 * regression_k + 5. * spread + .25 * pair\n            if not torch.isfinite(loss):\n                raise RuntimeError("v2検索モデルの損失が非有限です。")\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)\n            optimizer.step()\n            queue.push(query, key, descriptor)\n            step += 1\n            if step % int(config.get("log_interval", 50)) == 0:\n                print(f"step {step}/{max_steps}: loss={float(loss):.4f}, "\n                      f"意味検索={float(semantic):.4f}, 質感予測={float(regression_q):.5f}, "\n                      f"spread={float(spread):.5f}, std={float(mean_std):.4f}, "\n                      f"{(time.monotonic() - started) / 60:.1f}分", flush=True)\n            if step % interval == 0:\n                report, bank = evaluate_retrieval(\n                    model, distill, training, validation, device,\n                    save / "comparison-latest.png",\n                    int(config.get("bank_samples", 2048)),\n                    int(config.get("eval_samples", 96)),\n                    int(config.get("eval_batch_size", 8)),\n                    contract["rerank_top_k"], contract["descriptor_weight"])\n                score = (report["descriptor_mse"] / max(report["random_descriptor_mse"], 1e-12)\n                         + (1 - report["descriptor_neighbour_recall10"])\n                         + max(0., report["embedding_pairwise_cosine"] - .95) * 4.)\n                selected = score < best\n                if selected:\n                    best, stale = score, 0\n                    _checkpoint(save / "best.pt", model, optimizer, queue, step,\n                                contract, best, stale, report)\n                    shutil.copyfile(save / "comparison-latest.png", save / "comparison-best.png")\n                    records = [{"dataset_index": int(index),\n                                "shard": training.items[int(index)][0],\n                                "offset": int(training.items[int(index)][1])}\n                               for index in bank["indices"]]\n                    _atomic_torch(save / "bank.pt", {\n                        "version": VERSION, "step": step, "manifest": contract["manifest"],\n                        "key": bank["key"].half(), "descriptor": bank["descriptor"].half(),\n                        "records": records, "contains_raw_images": False, "split": "train"})\n                else:\n                    stale += 1\n                _checkpoint(latest, model, optimizer, queue, step, contract, best, stale, report)\n                line = dict(report, step=step, score=score, selected=selected, stale=stale)\n                with (save / "training.jsonl").open("a") as handle:\n                    handle.write(json.dumps(line, ensure_ascii=False) + "\\n")\n                print(f"検証 step={step}: {describe(report)}, best更新={selected}, stale={stale}",\n                      flush=True)\n                if stale >= patience:\n                    return {"step": step, "best_score": best, "stale": stale,\n                            "retrieval_passed": report["retrieval_passed"]}\n    return {"step": step, "best_score": best, "stale": stale,\n            "retrieval_passed": report["retrieval_passed"]}\n', 'roi_texture_distribution_runtime.py': '"""Teacher-image-free distributional texture generator for mioh ROI video.\n\nThe model learns several plausible sub-5-pixel luminance texture modes from\nground-truth training images.  It does not retrieve, copy, or align a teacher\nimage.  Geometry, colour, and low-frequency shading remain those of the frozen\nROI-distill result.  At deployment a mode is selected from observable coarse\ncontent, then held constant for the lifetime of an ROI track.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nimport roi_mask_detail_runtime as paired\nimport roi_mask_gan_runtime as gan\n\n\nVERSION = "roi-texture-distribution-v1"\nARCH = {"width": 40, "blocks": 6, "style_dim": 48, "frames": 3,\n        "modes": 4, "band": 5}\n\n\ndef blur(value, side):\n    side = max(1, int(side)); side += 1 - side % 2\n    shape = value.shape\n    flat = value.reshape(-1, shape[-3], *shape[-2:]).float()\n    return F.avg_pool2d(flat, side, 1, side // 2).reshape(shape)\n\n\ndef luma(value):\n    return .299 * value[..., 0:1, :, :] + .587 * value[..., 1:2, :, :] + .114 * value[..., 2:3, :, :]\n\n\ndef highpass(value, side=5):\n    return value.float() - blur(value.float(), side)\n\n\ndef resize_mask(mask, size):\n    shape = mask.shape\n    flat = mask.reshape(-1, 1, *shape[-2:]).float()\n    return F.interpolate(flat, size=size, mode="bilinear", align_corners=False).reshape(*shape[:-2], *size)\n\n\ndef support_mask(mask, size, erosion=9, feather=13):\n    mask = resize_mask(mask, size).clamp(0, 1)\n    shape = mask.shape\n    flat = mask.reshape(-1, 1, *size)\n    erosion += 1 - erosion % 2; feather += 1 - feather % 2\n    inner = -F.max_pool2d(-flat, erosion, 1, erosion // 2)\n    inner = F.avg_pool2d(inner.clamp(0, 1), feather, 1, feather // 2)\n    return inner.reshape(shape).clamp(0, 1)\n\n\ndef weighted_mean(value, mask):\n    weight = mask.expand_as(value)\n    return (value * weight).sum((-2, -1)) / weight.sum((-2, -1)).clamp_min(1e-6)\n\n\ndef texture_statistics(image, mask):\n    """Position-free fine-texture amount and orientation statistics."""\n    y = luma(image.float())\n    mask = resize_mask(mask.float(), y.shape[-2:])\n    values = []\n    for side in (3, 5):\n        detail = highpass(y, side)\n        # sqrt\'(0) is infinite.  Flat/masked training crops can therefore turn\n        # an otherwise finite loss into NaN gradients on the first update.\n        # A tiny variance floor keeps the statistic effectively unchanged\n        # (1e-4 before the x10 display scale) while making its derivative\n        # finite for perfectly flat regions.\n        rms = (weighted_mean(detail.square(), mask) + 1e-8).sqrt()\n        values.extend((weighted_mean(detail.abs(), mask) * 10, rms * 10))\n    detail = highpass(y, 5)\n    gx = detail[..., :, 1:] - detail[..., :, :-1]\n    gy = detail[..., 1:, :] - detail[..., :-1, :]\n    values.extend((weighted_mean(gx.abs(), torch.minimum(mask[..., :, 1:], mask[..., :, :-1])) * 10,\n                   weighted_mean(gy.abs(), torch.minimum(mask[..., 1:, :], mask[..., :-1, :])) * 10))\n    return torch.cat(values, -1).mean(1)\n\n\ndef edge_concentration(image, coarse, mask):\n    """Fraction of fine energy lying on coarse structural edges."""\n    fine = highpass(luma(image), 5).abs()\n    structure = highpass(blur(luma(coarse), 5), 17).abs()\n    threshold = weighted_mean(structure, mask)[..., None, None] * 1.5\n    edge = (structure > threshold).float()\n    value = weighted_mean(fine * edge, mask) / weighted_mean(fine, mask).clamp_min(1e-6)\n    return value.mean((1, 2))\n\n\nclass FiLMBlock(nn.Module):\n    def __init__(self, width, style_dim):\n        super().__init__()\n        self.conv1 = nn.Conv2d(width, width, 3, padding=1)\n        self.conv2 = nn.Conv2d(width, width, 3, padding=1)\n        self.affine = nn.Linear(style_dim, width * 2)\n        nn.init.zeros_(self.affine.bias)\n\n    def forward(self, value, style):\n        scale, bias = self.affine(style).chunk(2, 1)\n        hidden = self.conv1(value)\n        hidden = hidden * (1 + .15 * scale[..., None, None]) + .15 * bias[..., None, None]\n        return value + .1 * self.conv2(F.silu(hidden))\n\n\nclass DistributionTextureGenerator(nn.Module):\n    def __init__(self, width=40, blocks=6, style_dim=48, frames=3, modes=4, band=5):\n        super().__init__()\n        self.architecture = dict(width=width, blocks=blocks, style_dim=style_dim,\n                                 frames=frames, modes=modes, band=band)\n        self.mode_embedding = nn.Embedding(modes, style_dim)\n        self.stem = nn.Conv2d(frames * 4, width, 5, padding=2)\n        self.blocks = nn.ModuleList([FiLMBlock(width, style_dim) for _ in range(blocks)])\n        self.head = nn.Conv2d(width, frames, 3, padding=1)\n        self.selector = nn.Sequential(\n            nn.Conv2d(frames * 4, 24, 5, 2, 2), nn.SiLU(),\n            nn.Conv2d(24, 40, 3, 2, 1), nn.SiLU(),\n            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(40, modes))\n        nn.init.zeros_(self.head.weight); nn.init.zeros_(self.head.bias)\n\n    def features(self, coarse, mask):\n        if coarse.ndim != 5 or coarse.shape[1:3] != (self.architecture["frames"], 3):\n            raise ValueError("coarse must be [B,3,3,H,W]")\n        mask = resize_mask(mask, coarse.shape[-2:])\n        query = torch.cat((coarse.reshape(coarse.shape[0], -1, *coarse.shape[-2:]),\n                           mask.reshape(mask.shape[0], -1, *mask.shape[-2:])), 1)\n        return query, mask\n\n    def forward(self, coarse, mask, mode):\n        query, mask = self.features(coarse, mask)\n        mode = torch.as_tensor(mode, device=coarse.device, dtype=torch.long).reshape(-1)\n        if len(mode) == 1 and len(coarse) != 1:\n            mode = mode.expand(len(coarse))\n        if len(mode) != len(coarse):\n            raise ValueError("mode count must match batch")\n        feature = F.silu(self.stem(query)); style = self.mode_embedding(mode)\n        for block in self.blocks: feature = block(feature, style)\n        detail = highpass(self.head(feature)[:, :, None], self.architecture["band"])\n        support = support_mask(mask, coarse.shape[-2:])\n        return coarse.float() + detail.expand(-1, -1, 3, -1, -1) * support, detail, support\n\n    def mode_logits(self, coarse, mask):\n        query, _ = self.features(coarse, mask)\n        return self.selector(F.interpolate(query, (128, 128), mode="area"))\n\n    def candidates(self, coarse, mask):\n        outputs = [self(coarse, mask, torch.full((len(coarse),), mode, device=coarse.device))[0]\n                   for mode in range(self.architecture["modes"])]\n        return torch.stack(outputs, 1), self.mode_logits(coarse, mask)\n\n\nclass TextureDiscriminator(nn.Module):\n    def __init__(self, width=32):\n        super().__init__()\n        layers, channels = [], 3\n        for out in (width, width * 2, width * 3, width * 4):\n            layers += [nn.Conv2d(channels, out, 4, 2, 1), nn.LeakyReLU(.2)]\n            channels = out\n        layers.append(nn.Conv2d(channels, 1, 3, padding=1))\n        self.body = nn.Sequential(*layers)\n\n    def forward(self, image, coarse, mask):\n        fine = highpass(luma(image[:, None])[:, 0], 5) * 8\n        low = blur(luma(coarse[:, None])[:, 0], 17)\n        return self.body(torch.cat((fine, low, mask), 1))\n\n\ndef per_sample_candidate_cost(candidates, coarse, target, mask):\n    """Oracle assignment used only while training/evaluating, never at deployment."""\n    costs = []\n    target_stats = texture_statistics(target, mask)\n    target_edge = edge_concentration(target, coarse, mask)\n    hp_target = highpass(luma(target), 5)\n    for mode in range(candidates.shape[1]):\n        value = candidates[:, mode]\n        stats = (texture_statistics(value, mask) - target_stats).abs().mean(1)\n        edge = (edge_concentration(value, coarse, mask) - target_edge).abs()\n        hp = highpass(luma(value), 5)\n        temporal = weighted_mean(((hp[:, 1:] - hp[:, :-1]) -\n                                  (hp_target[:, 1:] - hp_target[:, :-1])).abs(),\n                                 torch.minimum(mask[:, 1:], mask[:, :-1])).mean((1, 2))\n        costs.append(stats + .35 * edge + .2 * temporal)\n    return torch.stack(costs, 1)\n\n\ndef select_candidates(candidates, mode):\n    index = mode[:, None, None, None, None, None].expand(-1, 1, *candidates.shape[2:])\n    return candidates.gather(1, index)[:, 0]\n\n\ndef generator_loss(chosen, candidates, logits, coarse, target, mask, discriminator, oracle):\n    hp, hp_target = highpass(luma(chosen), 5), highpass(luma(target), 5)\n    stats = F.smooth_l1_loss(texture_statistics(chosen, mask), texture_statistics(target, mask), beta=.02)\n    edge = (edge_concentration(chosen, coarse, mask) - edge_concentration(target, coarse, mask)).abs().mean()\n    temporal = weighted_mean(((hp[:, 1:] - hp[:, :-1]) -\n                              (hp_target[:, 1:] - hp_target[:, :-1])).abs(),\n                             torch.minimum(mask[:, 1:], mask[:, :-1])).mean()\n    low = weighted_mean(blur(chosen - coarse, 17).abs(), mask).mean()\n    detail = weighted_mean((hp - hp_target).abs(), mask).mean()\n    selector = F.cross_entropy(logits, oracle)\n    # The same scene must admit genuinely different plausible modes.\n    candidate_hp = highpass(luma(candidates), 5)\n    pairwise = []\n    for first in range(candidates.shape[1]):\n        for second in range(first + 1, candidates.shape[1]):\n            pairwise.append(weighted_mean((candidate_hp[:, first] - candidate_hp[:, second]).abs(), mask).mean())\n    diversity = F.relu(.006 - torch.stack(pairwise).mean())\n    adversarial = -discriminator(chosen[:, 1], coarse[:, 1], mask[:, 1]).mean()\n    total = stats + .25 * edge + .25 * temporal + .6 * low + .12 * detail + .12 * selector + .5 * diversity + .06 * adversarial\n    return total, {"stats": stats, "edge": edge, "temporal": temporal, "low": low,\n                   "detail": detail, "selector": selector, "diversity": diversity,\n                   "adversarial": adversarial}\n\n\ndef discriminator_loss(discriminator, chosen, coarse, target, mask):\n    real = discriminator(target[:, 1], coarse[:, 1], mask[:, 1])\n    fake = discriminator(chosen[:, 1].detach(), coarse[:, 1], mask[:, 1])\n    return F.relu(1 - real).mean() + F.relu(1 + fake).mean()\n\n\ndef atomic_torch(path, value):\n    path = Path(path); temporary = path.with_suffix(path.suffix + ".partial")\n    torch.save(value, temporary); os.replace(temporary, path)\n\n\ndef crop_batch(coarse, target, mask, side, generator):\n    height, width = coarse.shape[-2:]\n    if side > min(height, width): return coarse, target, mask\n    y = int(torch.randint(height - side + 1, (), generator=generator))\n    x = int(torch.randint(width - side + 1, (), generator=generator))\n    cut = (..., slice(y, y + side), slice(x, x + side))\n    return coarse[cut], target[cut], mask[cut]\n\n\ndef _distill_sequence(distill, restored, masks):\n    batch, _, _, height, width = restored.shape\n    windows = torch.stack([restored[:, index:index + 5] for index in range(3)], 1)\n    enhanced = distill(windows.reshape(batch * 3, 15, height, width)).reshape(batch, 3, 3, height * 2, width * 2)\n    under = F.interpolate(restored[:, 2:5].reshape(batch * 3, 3, height, width), scale_factor=2.,\n                          mode="bilinear", align_corners=False).reshape_as(enhanced)\n    roi = gan.up_masks(masks[:, 2:5], enhanced.shape[-1])\n    return under + roi * (enhanced - under), support_mask(roi, enhanced.shape[-2:])\n\n\ndef evaluate(model, distill, dataset, device, output, samples=48):\n    model.eval(); chosen_indices = np.linspace(0, len(dataset) - 1, min(samples, len(dataset))).round().astype(int)\n    totals = {key: 0. for key in ("stats", "edge", "temporal", "low", "oracle_gap", "diversity")}\n    rows = []\n    with torch.inference_mode():\n        for number, index in enumerate(chosen_indices):\n            restored, _, target, masks = [value[None].to(device) for value in dataset[int(index)]]\n            coarse, support = _distill_sequence(distill, restored, masks)\n            candidates, logits = model.candidates(coarse, support)\n            selected_mode = logits.argmax(1); selected = select_candidates(candidates, selected_mode)\n            oracle_cost = per_sample_candidate_cost(candidates, coarse, target, support)\n            hp, hp_target = highpass(luma(selected), 5), highpass(luma(target), 5)\n            totals["stats"] += float((texture_statistics(selected, support) - texture_statistics(target, support)).abs().mean())\n            totals["edge"] += float((edge_concentration(selected, coarse, support) - edge_concentration(target, coarse, support)).abs().mean())\n            totals["temporal"] += float(weighted_mean(((hp[:, 1:] - hp[:, :-1]) - (hp_target[:, 1:] - hp_target[:, :-1])).abs(), torch.minimum(support[:, 1:], support[:, :-1])).mean())\n            totals["low"] += float(weighted_mean(blur(selected - coarse, 17).abs(), support).mean())\n            totals["oracle_gap"] += float(oracle_cost.gather(1, selected_mode[:, None]).mean() - oracle_cost.min(1).values.mean())\n            mode_hp = highpass(luma(candidates), 5)\n            totals["diversity"] += float((mode_hp[:, 1:] - mode_hp[:, :-1]).abs().mean())\n            if number < 6:\n                row = [coarse[0, 1].cpu()] + [candidates[0, mode, 1].cpu() for mode in range(candidates.shape[1])]\n                row += [selected[0, 1].cpu(), target[0, 1].cpu(), support[0, 1].expand(3, -1, -1).cpu()]\n                rows.append(row)\n    report = {key: value / len(chosen_indices) for key, value in totals.items()}\n    report["score"] = report["stats"] + .3 * report["edge"] + .25 * report["temporal"] + report["low"] + .2 * report["oracle_gap"]\n    save_preview(output, rows, model.architecture["modes"])\n    return report\n\n\ndef save_preview(path, rows, modes):\n    labels = ["distill"] + [f"mode {i}" for i in range(modes)] + ["selected", "actual truth", "mask"]\n    side, header = 256, 32\n    canvas = Image.new("RGB", (side * len(labels), header + side * len(rows)), "white"); draw = ImageDraw.Draw(canvas)\n    for column, label in enumerate(labels): draw.text((column * side + 4, 8), label, fill="black")\n    for row, images in enumerate(rows):\n        for column, image in enumerate(images):\n            image = F.interpolate(image[None].float(), (side, side), mode="bilinear", align_corners=False)[0]\n            array = np.rint(image.clamp(0, 1).permute(1, 2, 0).numpy() * 255).astype(np.uint8)\n            canvas.paste(Image.fromarray(array), (column * side, header + row * side))\n    canvas.save(path)\n\n\ndef _file_digest(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef train(config):\n    device = torch.device(config.get("device", "cuda"))\n    if device.type == "cuda" and not torch.cuda.is_available(): raise RuntimeError("GPUランタイムへ接続してください。")\n    save = Path(config["save_root"]); save.mkdir(parents=True, exist_ok=True)\n    root = paired.restore_patches(config, config["manifest"])\n    distill = gan.load_distill(config["distill_checkpoint"]).to(device).eval().requires_grad_(False)\n    training = paired.Patches(root, config["manifest"], "train")\n    validation = paired.Patches(root, config["manifest"], "val"); validation.augment = False\n    model = DistributionTextureGenerator(**ARCH).to(device); discriminator = TextureDiscriminator().to(device)\n    optimizer_g = torch.optim.AdamW(model.parameters(), lr=float(config.get("learning_rate", 1e-4)), betas=(.5, .99))\n    optimizer_d = torch.optim.AdamW(discriminator.parameters(), lr=float(config.get("discriminator_lr", 5e-5)), betas=(.5, .99))\n    contract = {"version": VERSION, "manifest": paired.digest(config["manifest"]),\n                "distill": _file_digest(config["distill_checkpoint"]), "architecture": ARCH,\n                "batch_size": int(config.get("batch_size", 4)), "crop_side": int(config.get("crop_side", 160)),\n                "deployment": "average selector logits over initial ROI observations, then freeze mode per track"}\n    latest = save / "latest.pt"; step, best, stale = 0, float("inf"), 0\n    if latest.is_file():\n        state = torch.load(latest, map_location="cpu", weights_only=False)\n        if state.get("contract") != contract: raise ValueError("再開条件が異なります。新しいSAVE_ROOTを指定してください。")\n        model.load_state_dict(state["model"]); discriminator.load_state_dict(state["discriminator"])\n        optimizer_g.load_state_dict(state["optimizer_g"]); optimizer_d.load_state_dict(state["optimizer_d"])\n        step, best, stale = int(state["step"]), float(state["best_score"]), int(state["stale"])\n        print(f"質感分布学習を再開: step={step}, stale={stale}", flush=True)\n    elif any(save.iterdir()): raise ValueError("新規実験のSAVE_ROOTが空ではありません。")\n    loader = torch.utils.data.DataLoader(training, batch_size=contract["batch_size"], shuffle=True, num_workers=0, drop_last=True)\n    max_steps, interval = int(config.get("max_steps", 6000)), int(config.get("eval_interval", 500))\n    crop_generator = torch.Generator().manual_seed(int(config.get("seed", 20261014))); started = time.monotonic()\n    while step < max_steps:\n        model.train(); discriminator.train()\n        for restored, _, target, masks in loader:\n            if step >= max_steps: break\n            restored, target, masks = restored.to(device), target.to(device), masks.to(device)\n            with torch.no_grad(): coarse, support = _distill_sequence(distill, restored, masks)\n            coarse, target, support = crop_batch(coarse, target, support, contract["crop_side"], crop_generator)\n            candidates, logits = model.candidates(coarse, support)\n            with torch.no_grad():\n                costs = per_sample_candidate_cost(candidates, coarse, target, support)\n                # At the zero-residual start all modes tie.  Random tie-breaking\n                # gives every mode training examples instead of collapsing to 0.\n                oracle = (costs + torch.rand_like(costs) * 1e-5).argmin(1)\n            chosen = select_candidates(candidates, oracle)\n            loss_d = discriminator_loss(discriminator, chosen, coarse, target, support)\n            optimizer_d.zero_grad(set_to_none=True); loss_d.backward(); torch.nn.utils.clip_grad_norm_(discriminator.parameters(), 1., error_if_nonfinite=True); optimizer_d.step()\n            for parameter in discriminator.parameters(): parameter.requires_grad_(False)\n            candidates, logits = model.candidates(coarse, support); chosen = select_candidates(candidates, oracle)\n            loss_g, terms = generator_loss(chosen, candidates, logits, coarse, target, support, discriminator, oracle)\n            optimizer_g.zero_grad(set_to_none=True); loss_g.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True); optimizer_g.step()\n            for parameter in discriminator.parameters(): parameter.requires_grad_(True)\n            step += 1\n            if step % 50 == 0:\n                print(f"step {step}/{max_steps}: G={float(loss_g):.4f}, D={float(loss_d):.4f}, stats={float(terms[\'stats\']):.5f}, edge={float(terms[\'edge\']):.5f}, temporal={float(terms[\'temporal\']):.5f}, diversity={float(terms[\'diversity\']):.5f}, {(time.monotonic()-started)/60:.1f}分", flush=True)\n            if step % interval == 0:\n                report = evaluate(model, distill, validation, device, save / "distribution-latest.png", int(config.get("eval_samples", 48)))\n                selected = report["score"] < best\n                if selected:\n                    best, stale = report["score"], 0\n                    atomic_torch(save / "best.pt", {"version": VERSION, "model": model.state_dict(), "step": step,\n                                                     "contract": contract, "report": report, "app_integrated": False})\n                    import shutil; shutil.copyfile(save / "distribution-latest.png", save / "distribution-best.png")\n                else: stale += 1\n                state = {"version": VERSION, "model": model.state_dict(), "discriminator": discriminator.state_dict(),\n                         "optimizer_g": optimizer_g.state_dict(), "optimizer_d": optimizer_d.state_dict(), "step": step,\n                         "best_score": best, "stale": stale, "contract": contract, "report": report}\n                atomic_torch(latest, state)\n                with (save / "training.jsonl").open("a") as handle:\n                    handle.write(json.dumps({**report, "step": step, "selected": selected, "stale": stale}) + "\\n")\n                print(f"検証 step={step}: score={report[\'score\']:.5f}, stats={report[\'stats\']:.5f}, edge={report[\'edge\']:.5f}, temporal={report[\'temporal\']:.5f}, diversity={report[\'diversity\']:.5f}, best更新={selected}, stale={stale}", flush=True)\n                if stale >= int(config.get("patience", 8)): return {"step": step, "best_score": best, "stale": stale}\n    return {"step": step, "best_score": best, "stale": stale}\n', 'roi_retrieval_track_texture_runtime.py': '"""Retrieval-conditioned, track-stable fine texture generation.\n\nThe frozen retrieval-v2 encoder predicts an observable material descriptor from\nROI-Distill output.  No retrieved pixels, bank images, target embedding, or\ntruth-conditioned state enters the generator.  A continuous latent is chosen\nonce for an ROI track and shared by every frame in the sequence.\n\nThe generator may change only the sub-five-pixel luminance band inside the ROI.\nIts residual is locally high-pass and exactly zero-mean on each supported frame,\nso geometry, colour, and low-frequency shading remain those of ROI Distill.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport time\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nimport roi_mask_detail_runtime as paired\nimport roi_mask_gan_runtime as gan\nimport roi_material_retrieval_v2_runtime as retrieval\nimport roi_texture_distribution_runtime as distribution\n\n\nVERSION = "roi-retrieval-track-texture-v2-spatial-latent"\nARCH = {"width": 40, "blocks": 6, "style_dim": 64, "latent_dim": 16,\n        "descriptor_dim": 14, "frames": 3, "band": 5,\n        "noise_channels": 4, "noise_side": 16}\n\n\ndef latent_from_track_ids(track_ids, dimension=16, device=None):\n    """Return reproducible Gaussian-like latents without global RNG state."""\n    rows = []\n    for track_id in track_ids:\n        values = []\n        counter = 0\n        while len(values) < int(dimension):\n            digest = hashlib.sha256(f"{track_id}:{counter}".encode()).digest()\n            values.extend((byte / 127.5 - 1.) for byte in digest)\n            counter += 1\n        row = torch.tensor(values[:int(dimension)], dtype=torch.float32)\n        row = (row - row.mean()) / row.std(unbiased=False).clamp_min(1e-6)\n        rows.append(row)\n    return torch.stack(rows).to(device=device)\n\n\ndef zero_mean_detail(detail, support):\n    """Remove each frame\'s supported mean without leaking outside the mask."""\n    # The returned residual is multiplied by support.  Its supported mean is\n    # therefore weighted by support a second time; use support² here so soft\n    # feather pixels obey the same exact zero-mean contract as hard interiors.\n    weight = support.expand_as(detail).square()\n    mean = (detail * weight).sum((-2, -1), keepdim=True) / weight.sum(\n        (-2, -1), keepdim=True).clamp_min(1e-6)\n    return (detail - mean) * support\n\n\nclass FiLMBlock(nn.Module):\n    def __init__(self, width, style_dim):\n        super().__init__()\n        self.conv1 = nn.Conv2d(width, width, 3, padding=1)\n        self.conv2 = nn.Conv2d(width, width, 3, padding=1)\n        self.affine = nn.Linear(style_dim, width * 2)\n        nn.init.zeros_(self.affine.bias)\n\n    def forward(self, value, style):\n        scale, bias = self.affine(style).chunk(2, 1)\n        hidden = self.conv1(value)\n        hidden = hidden * (1 + .15 * scale[..., None, None]) + .15 * bias[..., None, None]\n        return value + .1 * self.conv2(F.silu(hidden))\n\n\nclass TrackTextureGenerator(nn.Module):\n    def __init__(self, width=40, blocks=6, style_dim=64, latent_dim=16,\n                 descriptor_dim=14, frames=3, band=5, noise_channels=4,\n                 noise_side=16):\n        super().__init__()\n        self.architecture = dict(width=width, blocks=blocks, style_dim=style_dim,\n                                 latent_dim=latent_dim, descriptor_dim=descriptor_dim,\n                                 frames=frames, band=band,\n                                 noise_channels=noise_channels,\n                                 noise_side=noise_side)\n        self.style = nn.Sequential(\n            nn.Linear(descriptor_dim + latent_dim, style_dim), nn.SiLU(),\n            nn.Linear(style_dim, style_dim))\n        self.noise_map = nn.Linear(latent_dim, noise_channels * noise_side * noise_side)\n        self.stem = nn.Conv2d(frames * 4 + noise_channels, width, 5, padding=2)\n        self.blocks = nn.ModuleList([FiLMBlock(width, style_dim) for _ in range(blocks)])\n        self.head = nn.Conv2d(width, frames, 3, padding=1)\n        # Give the spatial latent path a gradient from the first update while\n        # keeping the initial residual visually negligible.\n        nn.init.normal_(self.head.weight, std=1e-4)\n        nn.init.zeros_(self.head.bias)\n\n    def spatial_latent(self, track_latent, size):\n        """Turn one track latent into one normalized field shared by all frames."""\n        batch = len(track_latent)\n        channels = self.architecture["noise_channels"]\n        side = self.architecture["noise_side"]\n        field = self.noise_map(track_latent.float()).reshape(batch, channels, side, side)\n        field = F.interpolate(field, size=size, mode="bilinear", align_corners=False)\n        field = distribution.highpass(field, self.architecture["band"])\n        deviation = field.flatten(2).std(2, unbiased=False)[..., None, None]\n        return field / deviation.clamp_min(1e-4)\n\n    def forward(self, coarse, mask, descriptor, track_latent):\n        frames = self.architecture["frames"]\n        if coarse.ndim != 5 or coarse.shape[1:3] != (frames, 3):\n            raise ValueError(f"coarse must be [B,{frames},3,H,W]")\n        if descriptor.shape != (len(coarse), self.architecture["descriptor_dim"]):\n            raise ValueError("descriptor shape does not match the generator contract")\n        if track_latent.shape != (len(coarse), self.architecture["latent_dim"]):\n            raise ValueError("track_latent shape does not match the generator contract")\n        support = distribution.support_mask(mask, coarse.shape[-2:])\n        spatial_latent = self.spatial_latent(track_latent, coarse.shape[-2:])\n        query = torch.cat((coarse.reshape(len(coarse), -1, *coarse.shape[-2:]),\n                           support.reshape(len(coarse), -1, *coarse.shape[-2:]),\n                           spatial_latent), 1)\n        style = self.style(torch.cat((descriptor.float(), track_latent.float()), 1))\n        feature = F.silu(self.stem(query))\n        for block in self.blocks:\n            feature = block(feature, style)\n        raw = self.head(feature)[:, :, None]\n        detail = distribution.highpass(raw, self.architecture["band"])\n        detail = zero_mean_detail(detail, support)\n        prediction = coarse.float() + detail.expand(-1, -1, 3, -1, -1)\n        return prediction, detail, support\n\n    def candidates(self, coarse, mask, descriptor, track_ids=(0, 1, 2, 3)):\n        values = []\n        for track_id in track_ids:\n            latent = latent_from_track_ids(\n                [track_id] * len(coarse), self.architecture["latent_dim"], coarse.device)\n            values.append(self(coarse, mask, descriptor, latent)[0])\n        return torch.stack(values, 1)\n\n\ndef load_retrieval(path, device):\n    state = torch.load(path, map_location="cpu", weights_only=True)\n    if state.get("version") != retrieval.VERSION:\n        raise ValueError("Retrieval v2 best.ptが必要です。")\n    if not state.get("report", {}).get("retrieval_passed"):\n        raise ValueError("合格済みRetrieval v2 checkpointが必要です。")\n    model = retrieval.StableWholeROIEncoder(state["contract"]["embedding_dim"])\n    model.load_state_dict(state["model"], strict=True)\n    return model.to(device).eval().requires_grad_(False), state\n\n\ndef predicted_descriptor(model, coarse, mask):\n    return model.descriptor(model.query(coarse, mask))\n\n\ndef _texture_terms(prediction, coarse, target, support):\n    hp = distribution.highpass(distribution.luma(prediction), 5)\n    hp_target = distribution.highpass(distribution.luma(target), 5)\n    stats = F.smooth_l1_loss(\n        distribution.texture_statistics(prediction, support),\n        distribution.texture_statistics(target, support), beta=.02)\n    edge = (distribution.edge_concentration(prediction, coarse, support)\n            - distribution.edge_concentration(target, coarse, support)).abs().mean()\n    temporal = distribution.weighted_mean(\n        ((hp[:, 1:] - hp[:, :-1]) - (hp_target[:, 1:] - hp_target[:, :-1])).abs(),\n        torch.minimum(support[:, 1:], support[:, :-1])).mean()\n    low = distribution.weighted_mean(\n        distribution.blur(prediction - coarse, 17).abs(), support).mean()\n    return stats, edge, temporal, low\n\n\ndef generator_loss(first, second, coarse, target, support, discriminator,\n                   diversity_floor=.012):\n    first_terms = _texture_terms(first, coarse, target, support)\n    second_terms = _texture_terms(second, coarse, target, support)\n    stats, edge, temporal, low = [(.5 * (a + b)) for a, b in zip(first_terms, second_terms)]\n    first_hp = distribution.highpass(distribution.luma(first), 5)\n    second_hp = distribution.highpass(distribution.luma(second), 5)\n    distance = distribution.weighted_mean((first_hp - second_hp).abs(), support).mean()\n    diversity = F.relu(float(diversity_floor) - distance)\n    adversarial = -.5 * (\n        discriminator(first[:, 1], coarse[:, 1], support[:, 1]).mean()\n        + discriminator(second[:, 1], coarse[:, 1], support[:, 1]).mean())\n    total = stats + .3 * edge + .25 * temporal + .8 * low + 12.0 * diversity + .06 * adversarial\n    return total, {"stats": stats, "edge": edge, "temporal": temporal,\n                   "low": low, "diversity": distance, "diversity_penalty": diversity,\n                   "adversarial": adversarial}\n\n\ndef discriminator_loss(discriminator, prediction, coarse, target, support):\n    real = discriminator(target[:, 1], coarse[:, 1], support[:, 1])\n    fake = discriminator(prediction[:, 1].detach(), coarse[:, 1], support[:, 1])\n    return F.relu(1 - real).mean() + F.relu(1 + fake).mean()\n\n\ndef evaluate(model, retrieval_model, distill, dataset, device, output, samples=48,\n             candidate_ids=(0, 1, 2, 3)):\n    model.eval()\n    indices = np.linspace(0, len(dataset) - 1, min(samples, len(dataset))).round().astype(int)\n    totals = {key: 0. for key in ("stats", "edge", "temporal", "low", "diversity")}\n    rows = []\n    with torch.inference_mode():\n        for number, index in enumerate(indices):\n            restored, _, target, masks = [value[None].to(device) for value in dataset[int(index)]]\n            coarse, roi = retrieval.v1.coarse_sequence(distill, restored, masks)\n            descriptor = predicted_descriptor(retrieval_model, coarse, roi)\n            candidates = model.candidates(coarse, roi, descriptor, candidate_ids)\n            support = distribution.support_mask(roi, coarse.shape[-2:])\n            sample_terms = [_texture_terms(candidates[:, c], coarse, target, support)\n                            for c in range(candidates.shape[1])]\n            for position, key in enumerate(("stats", "edge", "temporal", "low")):\n                totals[key] += float(torch.stack([terms[position] for terms in sample_terms]).mean())\n            hp = distribution.highpass(distribution.luma(candidates), 5)\n            pairs = []\n            for first in range(candidates.shape[1]):\n                for second in range(first + 1, candidates.shape[1]):\n                    pairs.append(distribution.weighted_mean(\n                        (hp[:, first] - hp[:, second]).abs(), support).mean())\n            totals["diversity"] += float(torch.stack(pairs).mean())\n            if number < 6:\n                row = [coarse[0, 1].cpu()]\n                row += [candidates[0, candidate, 1].cpu()\n                        for candidate in range(candidates.shape[1])]\n                row += [target[0, 1].cpu()]\n                detail = distribution.highpass(\n                    distribution.luma(candidates[0, :, 1]), 5)\n                row += [(.5 + 8 * detail[candidate]).expand(3, -1, -1).cpu()\n                        for candidate in range(candidates.shape[1])]\n                truth_detail = distribution.highpass(\n                    distribution.luma(target[0, 1]), 5)\n                row += [(.5 + 8 * truth_detail).expand(3, -1, -1).cpu(),\n                        support[0, 1].expand(3, -1, -1).cpu()]\n                rows.append(row)\n    report = {key: value / len(indices) for key, value in totals.items()}\n    report["diversity_passed"] = report["diversity"] >= .012\n    report["score"] = (report["stats"] + .3 * report["edge"] + .25 * report["temporal"]\n                       + report["low"] + 12.0 * max(0., .012 - report["diversity"]))\n    save_preview(output, rows, len(candidate_ids))\n    return report\n\n\ndef save_preview(path, rows, candidates):\n    labels = (["distill"] + [f"track seed {index}" for index in range(candidates)]\n              + ["truth"] + [f"seed {index} detail x8" for index in range(candidates)]\n              + ["truth detail x8", "mask"])\n    side, header = 256, 32\n    canvas = Image.new("RGB", (side * len(labels), header + side * len(rows)), "white")\n    draw = ImageDraw.Draw(canvas)\n    for column, label in enumerate(labels):\n        draw.text((column * side + 4, 8), label, fill="black")\n    for row, images in enumerate(rows):\n        for column, value in enumerate(images):\n            value = F.interpolate(value[None].float(), (side, side), mode="bilinear",\n                                  align_corners=False)[0]\n            array = np.rint(value.clamp(0, 1).permute(1, 2, 0).numpy() * 255).astype(np.uint8)\n            canvas.paste(Image.fromarray(array), (column * side, header + row * side))\n    canvas.save(path)\n\n\ndef _digest(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _atomic_torch(path, value):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".partial")\n    torch.save(value, temporary)\n    os.replace(temporary, path)\n\n\ndef train(config):\n    device = torch.device(config.get("device", "cuda"))\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("GPUランタイムへ接続してください。")\n    save = Path(config["save_root"]); save.mkdir(parents=True, exist_ok=True)\n    root = paired.restore_patches(config, config["manifest"])\n    training = paired.Patches(root, config["manifest"], "train")\n    validation = paired.Patches(root, config["manifest"], "val"); validation.augment = False\n    retrieval_model, retrieval_state = load_retrieval(config["retrieval_checkpoint"], device)\n    distill = gan.load_distill(config["distill_checkpoint"]).to(device).eval().requires_grad_(False)\n    model = TrackTextureGenerator(**ARCH).to(device)\n    discriminator = distribution.TextureDiscriminator().to(device)\n    optimizer_g = torch.optim.AdamW(model.parameters(), lr=float(config.get("learning_rate", 1e-4)), betas=(.5, .99))\n    optimizer_d = torch.optim.AdamW(discriminator.parameters(), lr=float(config.get("discriminator_lr", 5e-5)), betas=(.5, .99))\n    contract = {"version": VERSION, "architecture": ARCH,\n                "manifest": paired.digest(config["manifest"]),\n                "distill": _digest(config["distill_checkpoint"]),\n                "retrieval": _digest(config["retrieval_checkpoint"]),\n                "retrieval_step": retrieval_state["step"],\n                "batch_size": int(config.get("batch_size", 4)),\n                "crop_side": int(config.get("crop_side", 160)),\n                "deployment": "one deterministic latent per ROI track; no reference pixels"}\n    latest = save / "latest.pt"\n    step, best, stale = 0, float("inf"), 0\n    if latest.is_file():\n        state = torch.load(latest, map_location="cpu", weights_only=False)\n        if state.get("contract") != contract:\n            raise ValueError("再開条件が異なります。新しいSAVE_ROOTを指定してください。")\n        model.load_state_dict(state["model"]); discriminator.load_state_dict(state["discriminator"])\n        optimizer_g.load_state_dict(state["optimizer_g"]); optimizer_d.load_state_dict(state["optimizer_d"])\n        step, best, stale = int(state["step"]), float(state["best_score"]), int(state["stale"])\n    elif any(save.iterdir()):\n        raise ValueError("新規実験のSAVE_ROOTが空ではありません。")\n    loader = torch.utils.data.DataLoader(training, batch_size=contract["batch_size"],\n                                         shuffle=True, num_workers=0, drop_last=True)\n    max_steps = int(config.get("max_steps", 6000)); interval = int(config.get("eval_interval", 500))\n    crop_rng = torch.Generator().manual_seed(int(config.get("seed", 20261014)))\n    latent_rng = torch.Generator().manual_seed(int(config.get("seed", 20261014)) + 1)\n    started = time.monotonic()\n    while step < max_steps:\n        model.train(); discriminator.train()\n        for restored, _, target, masks in loader:\n            if step >= max_steps: break\n            restored, target, masks = restored.to(device), target.to(device), masks.to(device)\n            with torch.no_grad():\n                coarse, roi = retrieval.v1.coarse_sequence(distill, restored, masks)\n                descriptor = predicted_descriptor(retrieval_model, coarse, roi)\n            coarse, target, roi = distribution.crop_batch(\n                coarse, target, roi, contract["crop_side"], crop_rng)\n            support = distribution.support_mask(roi, coarse.shape[-2:])\n            first_latent = torch.randn(len(coarse), ARCH["latent_dim"], generator=latent_rng).to(device)\n            second_latent = torch.randn(len(coarse), ARCH["latent_dim"], generator=latent_rng).to(device)\n            first = model(coarse, roi, descriptor, first_latent)[0]\n            loss_d = discriminator_loss(discriminator, first, coarse, target, support)\n            optimizer_d.zero_grad(set_to_none=True); loss_d.backward()\n            torch.nn.utils.clip_grad_norm_(discriminator.parameters(), 1., error_if_nonfinite=True); optimizer_d.step()\n            for parameter in discriminator.parameters(): parameter.requires_grad_(False)\n            first = model(coarse, roi, descriptor, first_latent)[0]\n            second = model(coarse, roi, descriptor, second_latent)[0]\n            loss_g, terms = generator_loss(first, second, coarse, target, support, discriminator)\n            optimizer_g.zero_grad(set_to_none=True); loss_g.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True); optimizer_g.step()\n            for parameter in discriminator.parameters(): parameter.requires_grad_(True)\n            step += 1\n            if step % 50 == 0:\n                print(f"step {step}/{max_steps}: G={float(loss_g):.4f}, D={float(loss_d):.4f}, "\n                      f"stats={float(terms[\'stats\']):.5f}, edge={float(terms[\'edge\']):.5f}, "\n                      f"temporal={float(terms[\'temporal\']):.5f}, diversity={float(terms[\'diversity\']):.5f}, "\n                      f"{(time.monotonic()-started)/60:.1f}分", flush=True)\n            if step % interval == 0:\n                report = evaluate(model, retrieval_model, distill, validation, device,\n                                  save / "track-texture-latest.png",\n                                  int(config.get("eval_samples", 48)))\n                selected = report["score"] < best\n                if selected:\n                    best, stale = report["score"], 0\n                    _atomic_torch(save / "best.pt", {"version": VERSION, "model": model.state_dict(),\n                        "step": step, "contract": contract, "report": report, "app_integrated": False})\n                    shutil.copyfile(save / "track-texture-latest.png", save / "track-texture-best.png")\n                else:\n                    stale += 1\n                state = {"version": VERSION, "model": model.state_dict(),\n                         "discriminator": discriminator.state_dict(),\n                         "optimizer_g": optimizer_g.state_dict(), "optimizer_d": optimizer_d.state_dict(),\n                         "step": step, "best_score": best, "stale": stale,\n                         "contract": contract, "report": report}\n                _atomic_torch(latest, state)\n                with (save / "training.jsonl").open("a") as handle:\n                    handle.write(json.dumps({**report, "step": step, "selected": selected,\n                                             "stale": stale}) + "\\n")\n                print(f"検証 step={step}: score={report[\'score\']:.5f}, stats={report[\'stats\']:.5f}, "\n                      f"edge={report[\'edge\']:.5f}, temporal={report[\'temporal\']:.5f}, "\n                      f"diversity={report[\'diversity\']:.5f}, best更新={selected}", flush=True)\n                if stale >= int(config.get("patience", 8)):\n                    return {"step": step, "best_score": best, "stale": stale}\n    return {"step": step, "best_score": best, "stale": stale}\n'}
for name, source in SOURCES.items():
    (work / name).write_text(source)
    sys.modules.pop(Path(name).stem, None)
sys.path.insert(0, str(work))
import roi_mask_gan_runtime as gan
import roi_retrieval_track_texture_runtime as track_texture
manifest = gan.merge_wide_manifests(
    WIDE_V1_ROOT, WIDE_ROOT,
    tuple(x for x in EXCLUDE_SOURCES.split(",") if x),
    tuple(x.strip() for x in WIDE_EXTRA_ROOTS.split(",") if x.strip()))
print("素材:", manifest["counts"], "ファイル数:", len(manifest["groups"]))

In [ ]:
#@title 3. トラック固定質感を学習
result = track_texture.train({
    "manifest": manifest, "prepared_root": WIDE_ROOT,
    "distill_checkpoint": DISTILL_CHECKPOINT,
    "retrieval_checkpoint": RETRIEVAL_CHECKPOINT,
    "save_root": SAVE_ROOT, "work_root": WORK_ROOT, "device": "cuda",
    "max_steps": int(MAX_STEPS), "eval_interval": int(EVAL_INTERVAL),
    "batch_size": int(BATCH_SIZE), "crop_side": 160,
    "learning_rate": 0.0001, "discriminator_lr": 0.00005,
    "eval_samples": 48, "patience": 8,
})
print(result)

In [ ]:
#@title 4. 最良結果
from PIL import Image
from IPython.display import display
state = torch.load(Path(SAVE_ROOT) / "best.pt", map_location="cpu", weights_only=True)
print("best step:", state["step"])
print(state["report"])
print("左から: distill / track seed 0〜3 / truth / seed細部x8 / truth細部x8 / mask")
display(Image.open(Path(SAVE_ROOT) / "track-texture-best.png"))